# Process-level equilibrium-information fidelity analysis

**Executable analysis artifact — not an exploratory analysis.** Research question, endpoints, representations, climates and admissible parameter families are frozen in Section 0 *before* any result is computed. Do not edit them after viewing results (spec §43).

## 0. Frozen scope and research question

> 동일 sorbent, 동일 weather ensemble, 동일 equilibrium-representation family에서 equilibrium information reduction에 따른 process-level climate-ordering fidelity가 environmental controller와 externally imposed controller 사이에서 달라지는가?

* Primary endpoint: $\tau_{m,\pi}^{(r)} = \tau[\mathrm{rank}_c Y^{(r)}_{m,c,\pi}, \mathrm{rank}_c Y^{(\mathrm{full})}_{m,c,\pi}]$ (Kendall τ-b, within controller).
* Controller comparison: $\Delta\tau^{(r)}_m = \tau^{(r)}_{m,\mathrm{env}} - \tau^{(r)}_{m,\mathrm{active}}$.
* Secondary: Spearman ρ, pairwise reversal fraction, every reversed pair, climate-level magnitude error, controller-parameter sensitivity.
* Existing-H/process bridge (where keys align exactly): $\eta_{m,\pi}^{(r)}=\tau[\mathrm{rank}_c H_c^{(r)},\mathrm{rank}_c Y_{m,c,\pi}^{(r)}]$. This is rank correspondence, not a magnitude comparison.
* **Not** an objective: absolute productivity; env-vs-active absolute performance. $Y_\mathrm{env}$ and $Y_\mathrm{active}$ are never compared directly.


**Supported-domain process rule (v1.6).** For ImCOF process endpoints, unsupported weather/cycle states are not extrapolated and do not invalidate an entire climate. `single` and `tensor` are first evaluated on the same cycles; only their common finite-support cycles are eligible. The primary process endpoint is the mean cycle response over that common supported subset, while supported-cycle count/fraction are reported separately. Zero-supported climates are non-evaluable, not assigned zero process response.

**Primary paired cases.** A: IRA-900-C environmental moisture response vs externally imposed moisture regeneration. B: ImCOF-TAEA PADAC-architecture environmental thermal regeneration vs imposed 80 °C regeneration. LWT-bPPI5 / Mg-Al-PEI excluded from primary analysis.

**Interpretation constraint (§46).** $H$ = equilibrium-side environmental opportunity descriptor; $Y$ = controller/process-dependent dynamic outcome. $H$ is not productivity. Environmentally induced IRA loading release is **not** recovered CO₂. PADAC is field-tested with Zeolite 13X; ImCOF adsorption/TPD are measured independently; **PADAC+ImCOF is a literature-constrained model composition, not an experimentally validated ImCOF reactor.**

**How the notebook works.** Section 0 writes two modules into `/content`: `fidelity_core.py` (provenance, gates, fidelity metrics, PADAC method-of-lines model) and `fidelity_adapters.py` (the only place where Repository code is bound). Each section reloads configuration and state from files in the run directory, so it can be rerun on its own after Section 0 has written the modules (after a Colab restart, rerun Section 0 with `RESUME_RUN_DIR` set). A failed gate records `NOT_EXECUTED_<REASON>` and independent branches continue.

**Drive binding revision v1.6.4. Reordered by scientific dependency and adds an explicit existing-H to process-Y climate-ordering bridge where keys align exactly.  Google Workspace .gsheet/.gdoc/.gslides placeholders are provenance locators and are not byte-hashed; ordinary files retain SHA256 hashing.  NASA POWER G/v augmentation, PADAC-13X architecture-level thermal forcing, and a regression-gated information-preserving thermal solver optimization are available as explicit provenance-tracked branches.** Paths and schemas are bound to the canonical P0 v4 lineage, IRA continuous-weather v7, process-bridge final outputs, canonical UTC weather, and root thermal QA files verified on 2026-11-13. The P0 reference representation is `tensor`; `single` is the reduced representation. Existing P0 H is frozen as the equilibrium-side baseline before any process branch is evaluated. IRA environmental logic must reuse `IRA_continuous_weather_process_bridge_v7.ipynb`; RH-threshold state controllers are not admissible.


In [ ]:
%%writefile /content/fidelity_core.py
# fidelity_core.py — helpers for the process-level equilibrium-information fidelity notebook.
# Written by Section 0. Contains NO analysis state: all state lives in RUN_DIR/_state and
# is reloaded explicitly by each section (no hidden notebook state).
#
# Interpretation constraint (Section 46, repeated here on purpose):
#   H = equilibrium-side environmental opportunity descriptor;  Y = controller/process outcome.
#   IRA environmental loading release is NOT recovered CO2.
#   PADAC (field-tested with Zeolite 13X) + ImCOF (adsorption/TPD measured independently)
#   is a literature-constrained MODEL COMPOSITION, not an experimentally validated ImCOF reactor.
import os, re, ast, json, hashlib, pickle, inspect, itertools, datetime as _dt
import numpy as np
import pandas as pd

PROVENANCE_CLASSES = (
    "DIRECT_REPORTED_MEASURED", "DIRECT_REPORTED_DESIGN", "DIGITIZED_FROM_FIGURE",
    "LITERATURE_PARAMETER", "FITTED_BY_SOURCE_AUTHORS", "DERIVED_FROM_REPORTED_EQUATION",
    "CALCULATED_THIS_NOTEBOOK", "ASSUMED_SENSITIVITY_PARAMETER", "REANALYSIS_DATA", "NASA_POWER_API", "NOT_AVAILABLE")
OK_PREFIXES = ("EXECUTED", "PASSED", "STRUCTURAL_REPRODUCTION_ONLY",
               "FIELD_TRACE_NOT_EXACTLY_REPRODUCIBLE_FROM_AVAILABLE_INPUT")
SESSION_PATH = "/content/fidelity_session.json"
CONFIG_PATH = "/content/fidelity_config.json"


class HardStop(Exception):
    """Raised when a gate fails. Callers record NOT_EXECUTED_<REASON> and continue other branches."""


# ----------------------------------------------------------------------------- session / state
def now():
    return _dt.datetime.now().isoformat(timespec="seconds")


def load_config():
    if not os.path.exists(CONFIG_PATH):
        raise HardStop("Config missing: run Section 0.")
    with open(CONFIG_PATH) as f:
        return json.load(f)


def load_session():
    if not os.path.exists(SESSION_PATH):
        raise HardStop("Session missing: run Sections 0–2.")
    with open(SESSION_PATH) as f:
        return json.load(f)


def update_session(**kw):
    s = {}
    if os.path.exists(SESSION_PATH):
        with open(SESSION_PATH) as f:
            s = json.load(f)
    s.update(kw)
    with open(SESSION_PATH, "w") as f:
        json.dump(s, f, indent=2, default=str)
    if s.get("RUN_DIR"):   # mirror into the run so a Colab restart can recover it
        os.makedirs(s["RUN_DIR"], exist_ok=True)
        with open(os.path.join(s["RUN_DIR"], "session.json"), "w") as f:
            json.dump(s, f, indent=2, default=str)
    return s


def run_dir():
    return load_session()["RUN_DIR"]


def _state_dir():
    d = os.path.join(run_dir(), "_state")
    os.makedirs(d, exist_ok=True)
    return d


_MISSING = object()


def put(name, obj):
    with open(os.path.join(_state_dir(), name + ".pkl"), "wb") as f:
        pickle.dump(obj, f)


def get(name, default=_MISSING):
    p = os.path.join(_state_dir(), name + ".pkl")
    if not os.path.exists(p):
        if default is _MISSING:
            raise HardStop(f"State '{name}' not found — rerun the section that produces it.")
        return default
    with open(p, "rb") as f:
        return pickle.load(f)


def _jsonl_append(fname, rec):
    with open(os.path.join(_state_dir(), fname), "a") as f:
        f.write(json.dumps(rec, default=str) + "\n")


def _jsonl_read(fname):
    p = os.path.join(_state_dir(), fname)
    if not os.path.exists(p):
        return []
    with open(p) as f:
        return [json.loads(l) for l in f if l.strip()]


def warn(code, message, section):
    _jsonl_append("warnings.jsonl", dict(time=now(), section=section, code=code, message=message))
    print(f"[WARN {code}] {message}")


def set_status(branch, status, reason="", section=""):
    _jsonl_append("eligibility.jsonl",
                  dict(time=now(), branch=branch, status=status, reason=reason, section=section))
    print(f"[STATUS] {branch}: {status} {('— ' + reason) if reason else ''}")


def not_executed(branch, reason_code, detail, section):
    set_status(branch, f"NOT_EXECUTED_{reason_code}", detail, section)


def statuses():
    latest = {}
    for r in _jsonl_read("eligibility.jsonl"):
        latest[r["branch"]] = r
    return latest


def status_of(branch):
    return statuses().get(branch, {}).get("status", "NOT_RUN")


def is_ok(*branches):
    return all(status_of(b).startswith(OK_PREFIXES) for b in branches)


def record_assumption(name, value, provenance, rationale, section, branch=""):
    if provenance not in PROVENANCE_CLASSES:
        raise ValueError(f"Unknown provenance class {provenance}")
    _jsonl_append("assumptions.jsonl", dict(time=now(), section=section, branch=branch, name=name,
                                            value=value, provenance=provenance, rationale=rationale))


def save_csv(df, name):
    p = os.path.join(run_dir(), name)
    df.to_csv(p, index=False)
    print("saved", p, f"({len(df)} rows)")
    return p


def guarded(branch, section, fn):
    """Run fn(); on HardStop record NOT_EXECUTED and return None instead of crashing other branches."""
    try:
        return fn()
    except HardStop as e:
        msg = str(e)
        code = msg.split(":")[0] if msg.split(":")[0].isupper() else "HARD_STOP"
        not_executed(branch, code, msg, section)
        return None


# ----------------------------------------------------------------------------- files / provenance
def drive_id(path):
    for attr in ("user.drive.id", "user.drive.item_id", "user.drive.itemprotostr"):
        try:
            v = os.getxattr(path, attr).decode(errors="ignore")
            m = re.search(r"[A-Za-z0-9_-]{20,}", v)
            return m.group(0) if m else v
        except Exception:
            pass
    return None


_DRIVE_PLACEHOLDER_EXTS = {".gsheet", ".gdoc", ".gslides", ".gdraw", ".gform", ".gmap", ".gsite"}


def _drive_placeholder(path):
    """True for Colab/Drive FUSE locator files that are not normal byte streams."""
    return os.path.splitext(str(path))[1].lower() in _DRIVE_PLACEHOLDER_EXTS


def sha256(path, chunk=1 << 20, max_bytes=5 * 1024 ** 3):
    """Hash ordinary byte-backed files; never crash on Google Workspace placeholders.

    Google Drive mounted .gsheet/.gdoc/.gslides entries are locators, not the
    underlying Workspace document bytes. Their content hash is therefore not a
    meaningful file SHA256. Other FUSE read failures are recorded explicitly
    rather than aborting the scientific pipeline.
    """
    if _drive_placeholder(path):
        return "GOOGLE_DRIVE_PLACEHOLDER_UNHASHED"
    try:
        if os.path.getsize(path) > max_bytes:
            return "SKIPPED_FILE_TOO_LARGE"
        h = hashlib.sha256()
        with open(path, "rb") as f:
            for b in iter(lambda: f.read(chunk), b""):
                h.update(b)
        return h.hexdigest()
    except OSError as e:
        # Drive FUSE can expose metadata for cloud-native files while rejecting
        # POSIX byte reads (e.g. Errno 95 Operation not supported). Preserve the
        # locator/provenance record and make the missing byte hash explicit.
        return f"UNHASHED_OSERROR_{getattr(e, 'errno', 'UNKNOWN')}"


def register_input(path, provenance, role, section, source_id=None, note=""):
    if provenance not in PROVENANCE_CLASSES + ("CODE_NON_NUMERIC", "AUDIT_ONLY"):
        raise ValueError(provenance)
    try:
        st = os.stat(path)
        size_bytes = st.st_size
        mtime = _dt.datetime.fromtimestamp(st.st_mtime).isoformat(timespec="seconds")
    except OSError as e:
        size_bytes = None
        mtime = None
        note = (note + "; " if note else "") + f"stat unavailable: OSError errno={getattr(e, 'errno', None)}"
    if os.path.isdir(path):
        digest = "DIRECTORY"
    else:
        digest = sha256(path)
    if _drive_placeholder(path):
        note = (note + "; " if note else "") + "Google Workspace Drive placeholder; locator registered, underlying cloud document bytes not hashed"
    rec = dict(path=os.path.abspath(path), filename=os.path.basename(path),
               drive_id=source_id or drive_id(path), size_bytes=size_bytes,
               mtime=mtime, sha256=digest,
               provenance=provenance, role=role, section=section, note=note)
    _jsonl_append("inputs.jsonl", rec)
    return rec


def walk_find(roots, name_regex, dirs=False, skip_regex=r"(^|/)\.|/_state(/|$)"):
    rx = re.compile(name_regex, re.I)
    sk = re.compile(skip_regex)
    out = []
    for root in roots:
        if not os.path.isdir(root):
            continue
        for dp, dns, fns in os.walk(root):
            if sk.search(dp):
                dns[:] = []
                continue
            for n in (dns if dirs else fns):
                if rx.search(n):
                    out.append(os.path.join(dp, n))
    return sorted(set(out))


# ----------------------------------------------------------------------------- reuse repository code
def load_code_defs(path, extra_globals=None):
    """Import definitions (imports, functions, classes, literal constants) from a repository .py/.ipynb
    WITHOUT executing its top-level pipeline. Cells that fail to parse are skipped and reported."""
    if path.endswith(".ipynb"):
        with open(path) as f:
            nb = json.load(f)
        chunks = ["".join(c["source"]) if isinstance(c["source"], list) else c["source"]
                  for c in nb["cells"] if c["cell_type"] == "code"]
    else:
        with open(path) as f:
            chunks = [f.read()]
    ns = {"__name__": "repo_" + re.sub(r"\W", "_", os.path.basename(path))}
    ns.update(extra_globals or {})
    skipped = []
    for i, src in enumerate(chunks):
        src = "\n".join(l for l in src.splitlines() if not l.lstrip().startswith(("%", "!")))
        try:
            tree = ast.parse(src)
        except SyntaxError as e:
            skipped.append((i, f"SyntaxError: {e}"))
            continue
        keep = []
        for node in tree.body:
            if isinstance(node, (ast.Import, ast.ImportFrom, ast.FunctionDef,
                                 ast.AsyncFunctionDef, ast.ClassDef)):
                keep.append(node)
            elif isinstance(node, (ast.Assign, ast.AnnAssign)) and node.value is not None:
                try:
                    ast.literal_eval(node.value)
                    keep.append(node)
                except Exception:
                    pass
        try:
            exec(compile(ast.Module(body=keep, type_ignores=[]), f"{path}#cell{i}", "exec"), ns)
        except Exception as e:
            skipped.append((i, f"{type(e).__name__}: {e}"))
    ns["__skipped_cells__"] = skipped
    return ns


def list_defs(ns):
    rows = []
    for k, v in ns.items():
        if k.startswith("_") or not callable(v) or getattr(v, "__module__", "") not in (ns.get("__name__"), None):
            continue
        try:
            sig = str(inspect.signature(v))
        except Exception:
            sig = "?"
        doc = (inspect.getdoc(v) or "").split("\n")[0][:120]
        rows.append(dict(name=k, signature=sig, doc=doc))
    return pd.DataFrame(rows)


def detect_col(df, patterns, what, required=True):
    hits = [c for c in df.columns if any(re.search(p, str(c), re.I) for p in patterns)]
    if len(hits) == 1:
        return hits[0]
    if required:
        raise HardStop(f"AMBIGUOUS_COLUMN: {what}: candidates={hits} in {list(df.columns)} — set it in CONFIG.")
    return None


# ----------------------------------------------------------------------------- fidelity metrics
def fidelity(Y_rep, Y_full, label=""):
    """Kendall tau-b / Spearman rho between climate rankings, plus every pairwise reversal."""
    from scipy.stats import kendalltau, spearmanr
    a = pd.Series(Y_rep, dtype=float)
    b = pd.Series(Y_full, dtype=float)
    if set(a.index) != set(b.index):
        raise HardStop(f"CLIMATE_MISMATCH: {label}: rep vs full climates differ.")
    a = a.loc[b.index]
    if a.isna().any() or b.isna().any():
        raise HardStop(f"NAN_SCORE: {label}: NaN in Y; climates={list(a.index[a.isna()]) + list(b.index[b.isna()])}")
    import warnings
    degenerate = bool(a.nunique() < 2 or b.nunique() < 2)   # constant Y: tau/rho undefined (NaN), reported as such
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        tau, ptau = kendalltau(a.values, b.values)
        rho, prho = spearmanr(a.values, b.values)
    pairs = []
    cl = list(b.index)
    for i, j in itertools.combinations(range(len(cl)), 2):
        sf = np.sign(b.iloc[i] - b.iloc[j])
        sr = np.sign(a.iloc[i] - a.iloc[j])
        if sf != sr:
            pairs.append(dict(climate_i=cl[i], climate_j=cl[j], sign_full=int(sf), sign_rep=int(sr),
                              kind="STRICT_REVERSAL" if sf * sr < 0 else "TIE_CHANGE",
                              Y_full_i=b.iloc[i], Y_full_j=b.iloc[j], Y_rep_i=a.iloc[i], Y_rep_j=a.iloc[j]))
    n_pairs = len(cl) * (len(cl) - 1) // 2
    strict = sum(p["kind"] == "STRICT_REVERSAL" for p in pairs)
    rel = (a - b) / b.abs().where(b.abs() > 0)
    finite = rel[np.isfinite(rel)]
    summ = dict(n_climates=len(cl), degenerate_constant_Y=degenerate, kendall_tau_b=tau, kendall_p=ptau, spearman_rho=rho, spearman_p=prho,
                n_pairs=n_pairs, n_strict_reversals=strict, n_tie_changes=len(pairs) - strict,
                reversal_fraction=strict / n_pairs if n_pairs else np.nan,
                mean_abs_rel_error=float(np.abs(finite).mean()) if len(finite) else np.nan,
                max_abs_rel_error=float(np.abs(finite).max()) if len(finite) else np.nan)
    climate_err = pd.DataFrame(dict(climate=cl, Y_full=b.values, Y_rep=a.values,
                                    abs_error=(a - b).values, rel_error=rel.values,
                                    rank_full=b.rank(ascending=False).values,
                                    rank_rep=a.rank(ascending=False).values))
    return summ, pd.DataFrame(pairs), climate_err


# ----------------------------------------------------------------------------- kernels
def monotone_fraction(x, y):
    """Sort by x, enforce non-decreasing y in [0,1]; return x, y, number of corrected points."""
    o = np.argsort(x)
    x, y = np.asarray(x, float)[o], np.asarray(y, float)[o]
    ym = np.clip(np.maximum.accumulate(y), 0, 1)
    return x, ym, int(np.sum(np.abs(ym - np.clip(y, 0, 1)) > 1e-12))


def pseudotime_step(frac_now, dt, k_t, k_F):
    """Advance fractional approach along a measured step-response kernel F(t) by dt using
    equivalent elapsed time (no rate equation is introduced; kernel is used as measured)."""
    t_eq = np.interp(np.clip(frac_now, k_F[0], k_F[-1]), k_F, k_t)
    return float(np.interp(t_eq + dt, k_t, k_F))


# ----------------------------------------------------------------------------- PADAC thermal model
SIGMA = 5.670374419e-8
R_GAS = 8.314462618


def air_props(T_K, P=101325.0):
    """Dry-air properties (ideal gas; Sutherland viscosity; power-law conductivity). LITERATURE_PARAMETER."""
    rho = P * 0.0289647 / (R_GAS * T_K)
    cp = 1006.0 + 0.0 * T_K
    mu = 1.716e-5 * (T_K / 273.15) ** 1.5 * (273.15 + 110.4) / (T_K + 110.4)
    k = 0.0241 * (T_K / 273.15) ** 0.81
    return rho, cp, mu, k


def _ev(expr, env):
    return eval(expr, {"np": np, "__builtins__": {}}, env)


class Forcing:
    """Hourly forcing -> continuous functions of t [s]. Linear interpolation between hourly samples."""
    def __init__(self, t_s, Tamb_K, G, v, flow_on):
        self.t = np.asarray(t_s, float)
        self.Ta, self.G, self.v = (np.asarray(a, float) for a in (Tamb_K, G, v))
        self.flow = np.asarray(flow_on, bool)
        if np.any(np.isnan(self.Ta)) or np.any(np.isnan(self.G)) or np.any(np.isnan(self.v)):
            raise HardStop("FORCING_NAN: forcing contains NaN; no gap filling is performed.")

    def at(self, t):
        return (np.interp(t, self.t, self.Ta), np.interp(t, self.t, self.G), np.interp(t, self.t, self.v))

    def flow_at(self, t):
        """Right-continuous hourly flow schedule: flow[i] applies on [t_i, t_{i+1}).
        This preserves the original hourly schedule exactly; only solver orchestration changes."""
        i = int(np.searchsorted(self.t, float(t), side="right") - 1)
        i = min(max(i, 0), len(self.flow) - 1)
        return bool(self.flow[i])

    def segments(self):
        """Contiguous constant-flow segments (integrated separately: flow switches are discontinuities)."""
        idx = np.flatnonzero(np.diff(self.flow.astype(int))) + 1
        bounds = np.r_[0, idx, len(self.t) - 1]
        return [(self.t[a], self.t[b], bool(self.flow[a])) for a, b in zip(bounds[:-1], bounds[1:]) if b > a]


class PadacModel:
    """Method-of-lines PADAC model (Filahi et al., ECM 338 (2025) 119925, Eqs. 6–15).
    Unknowns per axial node i: T_v (glass), T_w (wall), T_g (gas), T_s (solid) [K], and q [mol/kg]
    when sorption is coupled. Finite-volume axial discretisation; stiff BDF integration.
    Terms whose printed form must be checked against the PDF are taken from P['form'] (string expressions)."""

    def __init__(self, P, nz=20, sorption=None):
        self.P, self.nz = P, nz
        r, s, f = P["reactor"], P["sorbent"], P["form"]
        self.r, self.s, self.f = r, s, f
        self.dz = r["L"] / nz
        self.A = np.pi * r["D_int"] ** 2 / 4
        geo = dict(D_int=r["D_int"], D_ext=r["D_ext"], e_v=r["e_v"], e_w=r["e_w"], L=r["L"], np=np)
        self.a_v = _ev(f["a_v"], geo)
        self.w_solar = _ev(f["wall_solar_factor"], geo)
        self.w_rad = _ev(f["wall_rad_factor"], geo)
        self.w_gas = _ev(f["wall_gas_factor"], geo)
        self.g_wall = _ev(f["gas_wall_factor"], geo)
        self.sorption = sorption   # None | callable(Ts_K, q, flow_on) -> dq/dt [mol/kg/s]
        # Formulation A for ImCOF: desorption heat expressed through the SAME empirical F_R pathway
        # (apparent heat capacity, applied only while the bed heats) -> no second desorption pathway.
        self.apparent_cap = None   # None | callable(Ts_K, t_s) -> extra J m^-3 K^-1
        self.nstate = 5 if sorption else 4

    # --- correlations (forms selectable; primary = Watmuff) ---
    def h_wind(self, v):
        c = self.f["wind_correlation"]
        if c == "watmuff":
            return 2.8 + 3.0 * v
        if c == "kumar":
            return 10.03 + 4.687 * v
        if c == "mcadams_low":
            return 5.7 + 3.8 * v
        if c == "table3_constant":
            return 23.8 + 0.0 * v
        raise HardStop(f"UNKNOWN_WIND_CORRELATION: {c}")

    def coeffs(self, Tg, u):
        """h_w, lambda_gz, h_gs: Table-3 fitted constants or Eqs. 11/13/14 evaluated locally."""
        r, s = self.r, self.s
        rho, cp, mu, k = air_props(Tg)
        Pr = cp * mu / k
        if self.f["coefficient_mode"] == "table3_fitted":
            return (r["h_w_fitted"] + 0 * Tg, r["lambda_gz_fitted"] + 0 * Tg, s["h_gs"] + 0 * Tg)
        Re_D = rho * abs(u) * r["D_int"] / mu
        Re_p = rho * abs(u) * s["d_p"] / mu
        h_w = (12.5 + 0.048 * Re_D) * k / r["D_int"]                       # Eq. 11
        lam = (7.0 + 0.5 * Pr * Re_p) * k                                   # Eq. 13 (Re basis: PDF check)
        h_gs = (2.0 + 1.1 * Pr ** 0.33 * Re_p ** 0.6) * k / s["d_p"]        # Eq. 14 (2R_p = d_p)
        return h_w, lam, h_gs

    def rhs(self, t, y, F, flow_on):
        n, r, s, f = self.nz, self.r, self.s, self.f
        Tv, Tw, Tg, Ts = y[:n], y[n:2 * n], y[2 * n:3 * n], y[3 * n:4 * n]
        Ta, G, v = F.at(t)
        G = G * r.get("G_to_reactor_factor", 1.0)
        Tsky = r["sky_coeff"] * Ta ** 1.5                                   # Eq. 7, absolute temperature [K]
        hw = self.h_wind(v)
        h_vsky = r["eps_v"] * SIGMA * (Tv ** 2 + Tsky ** 2) * (Tv + Tsky)   # Eq. 8
        h_wv = SIGMA * (Tv ** 2 + Tw ** 2) * (Tv + Tw) / (1 / r["eps_v"] + 1 / r["eps_w"] - 1)  # Eq. 9
        glass_wall = (Tw - Tv) if f["eq6_glass_wall_sign"] == "physical" else (Tv - Tw)
        dTv = (self.a_v * r["alpha_v"] * G + self.a_v * h_wv * glass_wall
               - hw * self.a_v * (Tv - Ta) - h_vsky * self.a_v * (Tv - Tsky)) / (r["rho_v"] * r["C_v"])  # Eq. 6

        u = (r["Q_m3_h"] / 3600.0 / self.A) if flow_on else 0.0           # superficial velocity
        h_w, lam, h_gs = self.coeffs(Tg, u)
        dTw = (self.w_solar * r["alpha_w"] * r["tau_g"] * G + self.w_rad * h_wv * (Tv - Tw)
               + self.w_gas * h_w * (Tg - Tw)) / (r["rho_w"] * r["C_w"])  # Eq. 10 (factors: PDF check)

        rho_g, cp, _, _ = air_props(Tg)
        eps, dz = s["eps_bed"], self.dz
        T_in = Ta                                                          # inlet air = ambient
        rho_in, cp_in, _, _ = air_props(np.asarray(T_in, float))
        adv_faces = np.r_[rho_in * cp_in * u, rho_g * cp * u]             # upwind: face 0 = inlet, face i+1 = node i
        F_adv = adv_faces * np.r_[T_in, Tg]                                # faces 0..n
        lam_f = 0.5 * (lam[:-1] + lam[1:])
        F_disp = np.r_[0.0, -lam_f * np.diff(Tg) / dz, 0.0]               # Danckwerts-equivalent in FV
        F_tot = F_adv + F_disp
        q_gs = 6.0 * h_gs / s["d_p"] * (Tg - Ts)                            # Eq. 14/15 volumetric exchange
        dTg = (-(F_tot[1:] - F_tot[:-1]) / dz - f["gas_solid_sign_factor"] * q_gs
               - self.g_wall * h_w * (Tg - Tw)) / (eps * rho_g * cp)       # Eq. 12 (form: PDF check)

        k_s = s["k_s"]
        F_cond = np.r_[0.0, -k_s * np.diff(Ts) / dz, 0.0]                  # adiabatic solid ends
        heat = 0.0
        dq = None
        if self.sorption is not None:
            q = y[4 * n:5 * n]
            dq = self.sorption(Ts, q, flow_on)
            heat = (1 - eps) * s["rho_s"] * s["minus_dH_J_mol"] * dq       # Eq. 15 source term
        cap = (1 - eps) * s["rho_s"] * s["C_s"]
        net = -(F_cond[1:] - F_cond[:-1]) / dz + q_gs + heat
        dTs = net / cap                                                    # Eq. 15
        if self.apparent_cap is not None:
            heating = dTs > 0
            dTs = np.where(heating, net / (cap + self.apparent_cap(Ts, t)), dTs)
        out = [dTv, dTw, dTg, dTs] + ([dq] if dq is not None else [])
        return np.concatenate(out)

    def sparsity(self):
        from scipy.sparse import lil_matrix
        n, m = self.nz, self.nstate
        S = lil_matrix((m * n, m * n), dtype=int)
        V, W, Gg, Sx, Q = (k * n for k in range(5))
        for i in range(n):
            for a, b in [(V, V), (V, W), (W, V), (W, W), (W, Gg), (Gg, Gg), (Gg, W), (Gg, Sx), (Sx, Sx), (Sx, Gg)]:
                S[a + i, b + i] = 1
            for d in (-1, 1):
                if 0 <= i + d < n:
                    S[Gg + i, Gg + i + d] = 1
                    S[Sx + i, Sx + i + d] = 1
            if m == 5:
                for a, b in [(Q, Q), (Q, Sx), (Sx, Q)]:
                    S[a + i, b + i] = 1
        return S

    def _initial_state(self, F, q0=None):
        n = self.nz
        y = np.full(4 * n, F.Ta[0])
        if self.sorption is not None:
            y = np.r_[y, np.full(n, 0.0 if q0 is None else q0)]
        return y

    def _reduce_solution(self, t, Y):
        """Reduce node states to the same hourly outputs used by the legacy implementation."""
        n = self.nz
        res = pd.DataFrame(dict(t_s=np.asarray(t, float)))
        for k, nm in enumerate(["T_v", "T_w", "T_g", "T_s"]):
            blk = Y[k * n:(k + 1) * n, :].T - 273.15
            res[nm + "_mean_C"] = blk.mean(1)
            res[nm + "_max_C"] = blk.max(1)
        res["T_s_mid_C"] = Y[3 * n + n // 2, :] - 273.15
        if self.sorption is not None:
            res["q_mean_mol_kg"] = Y[4 * n:5 * n, :].mean(0)
        return res

    def run(self, F, q0=None, rtol=1e-5, atol=1e-4, extra_heat=None):
        """Legacy segmented BDF reference implementation. Kept unchanged as the regression oracle."""
        from scipy.integrate import solve_ivp
        y = self._initial_state(F, q0)
        S = self.sparsity()
        ts, ys = [F.t[0]], [y.copy()]
        for t0, t1, flow in F.segments():
            te = F.t[(F.t > t0) & (F.t <= t1)]
            sol = solve_ivp(lambda t, yy: self.rhs(t, yy, F, flow), (t0, t1), y, method="BDF",
                            t_eval=te, jac_sparsity=S, rtol=rtol, atol=atol)
            if not sol.success:
                raise HardStop(f"PADAC_SOLVER_FAILURE: {sol.message}")
            y = sol.y[:, -1]
            ts.extend(sol.t)
            ys.extend(sol.y.T)
        Y = np.array(ys).T
        return self._reduce_solution(np.asarray(ts, float), Y)

    def run_optimized(self, F, q0=None, rtol=1e-5, atol=1e-4, chunk_hours=744, max_step_s=None):
        """Information-preserving orchestration of the SAME MOL equations and SAME hourly forcing.

        Differences from run():
          * flow is evaluated as the original right-continuous hourly schedule inside the RHS;
          * many day/night segments are integrated in one BDF call;
          * the integration is chunked only for memory/checkpoint scale, carrying the exact final state forward;
          * every original hourly timestamp is retained in the returned output.

        This is a numerical-orchestration optimization, not temporal aggregation. It is enabled only after
        regression against run() passes the frozen equivalence gate.
        """
        from scipy.integrate import solve_ivp
        if len(F.t) < 2:
            raise HardStop("PADAC_SOLVER_FAILURE: forcing has fewer than two samples")
        y = self._initial_state(F, q0)
        S = self.sparsity()
        dt = float(np.median(np.diff(F.t)))
        if not np.allclose(np.diff(F.t), dt, rtol=0, atol=max(1e-6, abs(dt)*1e-9)):
            raise HardStop("PADAC_OPT_REQUIRES_REGULAR_TIME_GRID: canonical forcing is not regular")
        n_per = max(2, int(round(float(chunk_hours) * 3600.0 / dt)) + 1)
        frames = []
        a = 0
        while a < len(F.t) - 1:
            b = min(len(F.t) - 1, a + n_per - 1)
            te = F.t[a:b+1]
            kw = {} if max_step_s in (None, 0) else {"max_step": float(max_step_s)}
            sol = solve_ivp(lambda t, yy: self.rhs(t, yy, F, F.flow_at(t)),
                            (float(te[0]), float(te[-1])), y, method="BDF", t_eval=te,
                            jac_sparsity=S, rtol=rtol, atol=atol, **kw)
            if not sol.success:
                raise HardStop(f"PADAC_SOLVER_FAILURE: {sol.message}")
            fr = self._reduce_solution(sol.t, sol.y)
            if frames:
                fr = fr.iloc[1:].copy()  # shared chunk boundary, no information removed
            frames.append(fr)
            y = sol.y[:, -1].copy()
            a = b
        out = pd.concat(frames, ignore_index=True)
        if len(out) != len(F.t) or not np.allclose(out.t_s.to_numpy(float), F.t, rtol=0, atol=1e-6):
            raise HardStop("PADAC_OPT_OUTPUT_GRID_MISMATCH: optimized solver did not preserve every hourly timestamp")
        return out

    def compare_engines(self, F, q0=None, rtol=1e-5, atol=1e-4, chunk_hours=744, max_step_s=None):
        """Regression oracle: legacy segmented vs optimized, on identical forcing and output timestamps."""
        import time
        t0=time.perf_counter(); a=self.run(F,q0=q0,rtol=rtol,atol=atol); legacy_s=time.perf_counter()-t0
        t0=time.perf_counter(); b=self.run_optimized(F,q0=q0,rtol=rtol,atol=atol,chunk_hours=chunk_hours,max_step_s=max_step_s); opt_s=time.perf_counter()-t0
        if len(a)!=len(b) or not np.allclose(a.t_s,b.t_s,rtol=0,atol=1e-6):
            raise HardStop("PADAC_OPT_REGRESSION_GRID_MISMATCH: engines returned different hourly grids")
        cols=[c for c in a.columns if c!='t_s']
        rows=[]
        for c in cols:
            d=a[c].to_numpy(float)-b[c].to_numpy(float)
            rows.append(dict(variable=c,n=len(d),max_abs=float(np.nanmax(np.abs(d))),rmse=float(np.sqrt(np.nanmean(d*d))),mean_abs=float(np.nanmean(np.abs(d))),legacy_seconds=legacy_s,optimized_seconds=opt_s,speedup=(legacy_s/opt_s if opt_s>0 else np.inf)))
        return pd.DataFrame(rows), a, b


# ----------------------------------------------------------------------------- schedules / IRA environmental
def local_time(w, cfg):
    flag = cfg.get("WEATHER_TIMESTAMPS_ARE_LOCAL")
    ts = pd.to_datetime(w["timestamp"], utc=(flag is not True))
    if flag is True:
        return ts.dt.tz_localize(None) if getattr(ts.dt, "tz", None) is not None else ts
    zones = cfg.get("WEATHER_CITY_TIMEZONES") or {}
    if flag is False and zones:
        out = pd.Series(pd.NaT, index=w.index, dtype="datetime64[ns]")
        for c, idx in w.groupby("climate", sort=False).groups.items():
            z = zones.get(str(c))
            if not z:
                raise HardStop(f"TIMEZONE_UNRESOLVED: no IANA timezone for {c}")
            try:
                out.loc[idx] = ts.loc[idx].dt.tz_convert(z).dt.tz_localize(None).values
            except Exception as e:
                raise HardStop(f"TIMEZONE_UNRESOLVED: {c}/{z}: {e}")
        return out
    if flag is False and cfg.get("WEATHER_UTC_OFFSET_H"):
        off = w["climate"].map(cfg["WEATHER_UTC_OFFSET_H"])
        if off.isna().any():
            raise HardStop(f"TIMEZONE_UNRESOLVED: no UTC offset for {sorted(w.climate[off.isna()].unique())}")
        return ts.dt.tz_localize(None) + pd.to_timedelta(off, unit="h")
    raise HardStop("TIMEZONE_UNRESOLVED: canonical weather is UTC; provide WEATHER_CITY_TIMEZONES or explicit offsets.")


def cycle_labels(local_ts, start_h, end_h):
    """Adsorption window [start_h, end_h) (may cross midnight). cycle = date on which adsorption starts;
    the regeneration window is the following [end_h, start_h) period, which belongs to the same cycle."""
    h = local_ts.dt.hour
    ads = ((h >= start_h) | (h < end_h)) if start_h > end_h else ((h >= start_h) & (h < end_h))
    cyc = (local_ts - pd.to_timedelta(start_h, unit="h")).dt.floor("D")
    return ads.values, cyc.values


def ira_loading_trajectory(qstar, dt_h, kern_up, kern_down, mode="kernel_pseudotime", q0=None):
    """Finite-rate loading q(t) toward q*(t) using the MEASURED step-response kernels (t[h], F in [0,1]).
    kernel_pseudotime: memory of the current monotone approach is kept through equivalent elapsed time.
    kernel_memoryless: each hour is a fresh step response (sensitivity only)."""
    n = len(qstar)
    q = np.empty(n)
    q[0] = qstar[0] if q0 is None else q0
    d_prev, q_from = 0, q[0]
    for k in range(n - 1):
        gap = qstar[k] - q[k]
        d = int(np.sign(gap))
        kt, kF = kern_up if d > 0 else kern_down
        if d == 0:
            q[k + 1] = q[k]
            d_prev = 0
            continue
        if mode == "kernel_memoryless":
            q[k + 1] = q[k] + gap * float(np.interp(dt_h, kt, kF))
            continue
        if d != d_prev:
            q_from = q[k]
        span = qstar[k] - q_from
        frac = 0.0 if abs(span) < 1e-15 else np.clip((q[k] - q_from) / span, 0.0, kF[-1] - 1e-12)
        if frac < 0:  # target moved past the start point: restart approach
            q_from, frac = q[k], 0.0
            span = qstar[k] - q_from
        q[k + 1] = q_from + span * pseudotime_step(frac, dt_h, kt, kF)
        d_prev = d
    return q


def _deprecated_hysteresis_states_DO_NOT_USE(rh, rh_low, rh_high, dwell_h):
    """S(t) in {A=0, R=1}: A->R when RH>=rh_high, R->A when RH<=rh_low, each after >= dwell_h in state."""
    s = np.zeros(len(rh), dtype=np.int8)
    st, age = 0, 0
    for k, x in enumerate(rh):
        age += 1
        if st == 0 and x >= rh_high and age >= dwell_h:
            st, age = 1, 0
        elif st == 1 and x <= rh_low and age >= dwell_h:
            st, age = 0, 0
        s[k] = st
    return s


def ira_env_endpoints(q, s):
    dq = np.diff(q)
    naive_all = float(np.sum(np.clip(-dq, 0, None)))
    inR = s[1:] == 1
    naive_inR = float(np.sum(np.clip(-dq[inR], 0, None)))
    seg = 0.0
    nseg = 0
    edges = np.flatnonzero(np.diff(np.r_[0, s, 0]))
    for a, b in zip(edges[::2], edges[1::2]):          # R segment = hours a..b-1; release from entry to exit
        q_entry = q[a - 1] if a > 0 else q[a]
        seg += max(q_entry - q[b - 1], 0.0)
        nseg += 1
    return dict(Y_env_loading_release_segmented=seg, Y_naive_all=naive_all, Y_naive_inR=naive_inR,
                n_R_segments=nseg, double_count_ratio=(naive_inR / seg) if seg > 0 else np.nan)


def padac_params(cfg, sorbent, variant=None):
    """Reactor-side parameters always from CONFIG['PADAC'] (PADAC architecture); sorbent-side passed explicitly."""
    P = cfg["PADAC"]
    v = lambda k: P[k][0]
    form = dict(cfg["PADAC_FORM"])
    r = dict(L=v("L"), D_int=v("D_int"), D_ext=v("D_ext"), e_v=v("e_v"), e_w=v("e_w"), alpha_v=v("alpha_v"),
             eps_v=v("eps_v"), tau_g=v("tau_g"), alpha_w=v("alpha_w"), eps_w=v("eps_w"), rho_v=v("rho_v"), C_v=v("C_v"),
             rho_w=v("rho_w"), C_w=v("C_w"), Q_m3_h=v("Q_m3_h"), sky_coeff=v("sky_coeff"),
             h_w_fitted=v("h_w_fitted"), lambda_gz_fitted=v("lambda_gz_fitted"))
    if variant:
        form["wind_correlation"] = variant["wind_correlation"]
        form["coefficient_mode"] = variant["coefficient_mode"]
        if variant.get("sky_coeff"):
            r["sky_coeff"] = variant["sky_coeff"]
    return dict(reactor=r, sorbent=sorbent, form=form)


def _thermal_job(args):
    """One climate x variant x theta PADAC run (module-level so multiprocessing can pickle it)."""
    P, nz, t, Ta, G, v, flow, out_path, cap, engine, chunk_hours, max_step_s = args
    if os.path.exists(out_path):
        return out_path
    m = PadacModel(P, nz=nz)
    if cap is not None:   # Formulation A: (q_rich mol/kg per time, F_R kernel T grid, dF/dT, bulk density, -dH)
        tq, qr, kT, dF, rho_b, dH = cap
        m.apparent_cap = lambda Ts, tt: rho_b * dH * np.interp(tt, tq, qr) * np.interp(Ts - 273.15, kT, dF, left=0, right=0)
    F = Forcing(t, Ta, G, v, flow)
    if engine == "optimized":
        r = m.run_optimized(F, chunk_hours=chunk_hours, max_step_s=max_step_s)
    elif engine == "legacy_segmented":
        r = m.run(F)
    else:
        raise HardStop(f"UNKNOWN_PADAC_SOLVER_ENGINE: {engine}")
    r.to_parquet(out_path, index=False)
    return out_path


# ----------------------------------------------------------------------------- ImCOF cycles / TPD kernels
def tpd_select(meta, T_ads, RH_ads, cfg):
    """Nearest MEASURED adsorption condition (no interpolation between TPD curves). Returns (curve_id, distance)."""
    d = np.sqrt(((meta["T_ads_C"].to_numpy(float) - T_ads) / cfg["TPD_T_SCALE_C"]) ** 2 +
                ((meta["RH_ads_pct"].to_numpy(float) - RH_ads) / cfg["TPD_RH_SCALE_PCT"]) ** 2)
    i = int(np.argmin(d))
    return meta["curve_id"].iloc[i], float(d[i])


def F_R(kern, cid, T_C):
    """Empirical recovery fraction on the measured TPD ramp; 0 below the first point; held at the ramp-end value
    above the measured range (flagged by callers, never extrapolated upward)."""
    k = kern[cid]
    return np.interp(T_C, k["T"], k["F"], left=0.0, right=k["F"][-1])


def imcof_cycles(local_ts, T, RH, start_h, end_h, mode):
    """Complete daily cycles: adsorption window then regeneration window. Returns DataFrame with
    cycle, T_ads, RH_ads, ads_idx, reg_idx. Incomplete first/last cycles are excluded (identically for all r, pi)."""
    ads, cyc = cycle_labels(pd.Series(pd.to_datetime(local_ts)), start_h, end_h)
    df = pd.DataFrame(dict(cyc=cyc, ads=ads, T=T, RH=RH, i=np.arange(len(T))))
    rows = []
    n_ads = (end_h - start_h) % 24
    for c, g in df.groupby("cyc", sort=True):
        a, r = g[g.ads], g[~g.ads]
        if len(a) != n_ads or len(r) != 24 - n_ads:
            continue
        if mode == "end_of_window":
            Ta, RHa = a["T"].iloc[-1], a["RH"].iloc[-1]
        else:
            Ta, RHa = a["T"].mean(), a["RH"].mean()
        rows.append(dict(cycle=c, T_ads=Ta, RH_ads=RHa, ads_idx=a.i.to_numpy(), reg_idx=r.i.to_numpy()))
    return pd.DataFrame(rows)


In [ ]:
%%writefile /content/fidelity_adapters.py
# fidelity_adapters.py — repository-specific bindings for the verified 2026-11-13 lineage.
# v1.2 repairs the adapters against the ACTUAL P0 v4, IRA v7, weather and process-final schemas.
# No substitute physical model is introduced here.
import os, re, inspect
import numpy as np
import pandas as pd
import fidelity_core as fc
from fidelity_core import HardStop


def load_p0():
    res = fc.get("resolved")
    p = res.get("P0_CODE")
    if not p:
        raise HardStop("P0_CODE_NOT_FOUND: canonical CODE_P0_tensor_reproduction_v4_regression_repair.ipynb not resolved.")
    return fc.load_code_defs(p)


def load_bridge():
    res = fc.get("resolved")
    paths = res.get("BRIDGE_CODE", [])
    if not paths:
        raise HardStop("BRIDGE_CODE_NOT_FOUND: no process-bridge code resolved.")
    ns = {}
    for p in paths:
        ns.update({k: v for k, v in fc.load_code_defs(p).items() if not k.startswith("__")})
    return ns


def load_ira_continuous():
    res = fc.get("resolved")
    p = res.get("IRA_CONTINUOUS_CODE")
    if not p:
        raise HardStop("IRA_CONTINUOUS_V7_NOT_FOUND: canonical IRA_continuous_weather_process_bridge_v7.ipynb not resolved.")
    return fc.load_code_defs(p)


def _bind(ns, names, what):
    found = [n for n in names if callable(ns.get(n))]
    if len(found) == 1:
        return ns[found[0]], found[0]
    raise HardStop(f"{what}_ADAPTER_UNBOUND: candidates {names} -> found {found}.")


def _call(fn, **avail):
    sig = inspect.signature(fn)
    kw = {k: v for k, v in avail.items() if k in sig.parameters}
    missing = [k for k, p in sig.parameters.items()
               if p.default is p.empty and p.kind in (p.POSITIONAL_OR_KEYWORD, p.KEYWORD_ONLY) and k not in kw]
    if missing:
        raise HardStop(f"ADAPTER_SIGNATURE: {fn.__name__}{sig} needs {missing}.")
    return fn(**kw)


# ----------------------------------------------------------------------------- weather
def load_climate_weather(cfg):
    """Canonical P0 weather -> long dataframe.
    Verified source schema: climate_class, city, lat, lon, timestamp_utc, temperature_C, RH_pct.
    The analysis key is city (36 unique climates/sites); climate_class is retained as metadata.
    """
    res = fc.get("resolved")
    p = res.get("WEATHER")
    if not p:
        raise HardStop("WEATHER_NOT_RESOLVED: canonical weather path missing.")
    df = pd.read_parquet(p) if p.endswith(".parquet") else (pd.read_excel(p) if p.endswith("xlsx") else pd.read_csv(p))
    cm = cfg["WEATHER_COLUMNS"]
    pats = dict(
        climate=[r"^city$", r"^climate$", r"^site$", r"^location$"],
        timestamp=[r"^timestamp_utc$", r"^timestamp$", r"time", r"date"],
        T_C=[r"^temperature_C$", r"^t(_?amb|emp|2m|_c|air)", r"temperature"],
        RH_pct=[r"^RH_pct$", r"^rh", r"relative.?hum"],
        G_Wm2=[r"ghi", r"^g(_|$)", r"irrad", r"solar", r"ssrd", r"swdown"],
        v_ms=[r"wind", r"^ws", r"^v(_|$)", r"u10"],
    )
    col = {k: (cm.get(k) or fc.detect_col(df, pp, k, required=(k in ("climate","timestamp","T_C","RH_pct"))))
           for k, pp in pats.items()}
    keep = {v:k for k,v in col.items() if v}
    # Preserve canonical metadata needed for exact civil-time conversion/auditing.
    for raw, std in [("climate_class","climate_class"),("lat","lat"),("lon","lon")]:
        if raw in df.columns and raw not in keep:
            keep[raw]=std
    w = df[list(keep)].rename(columns=keep).copy()
    w["timestamp"] = pd.to_datetime(w["timestamp"], utc=True)
    for c in ("T_C","RH_pct","G_Wm2","v_ms","lat","lon"):
        if c in w.columns:
            w[c]=pd.to_numeric(w[c],errors="coerce")
    if w.duplicated(["climate","timestamp"]).any():
        raise HardStop("CLIMATE_TIME_MERGE_MISMATCH: duplicate (climate,timestamp) keys in weather.")
    order = cfg.get("CLIMATE_ORDER") or list(pd.unique(w["climate"]))
    return w, order, p, col


# ----------------------------------------------------------------------------- exact P0 two-representation contract
_MATERIAL_NAME_MAP = {
    'Al2O3-TEPA(20)': 'Al2O3_TEPA(20)',
    'MIL-101(Cr)-TEPA(30)': 'MIL-101(Cr)_TEPA(30)',
    'Mg0.55Al-CO3-PEI(30)': 'Mg0.55Al-CO3_PEI(30)',
}


def _load_p0_tables():
    res=fc.get("resolved")
    eqp=res.get("EQ_MASTER")
    bdir=res.get("P0_OUTPUTS")
    bp=os.path.join(bdir,"P0_material_bounds.csv") if bdir else None
    if not eqp or not os.path.exists(eqp):
        raise HardStop("P0_EQ_MASTER_MISSING: canonical equilibrium master not resolved.")
    if not bp or not os.path.exists(bp):
        raise HardStop("P0_BOUNDS_MISSING: P0_material_bounds.csv missing from canonical frozen outputs.")
    req={'material','T_C','RH_pct','q_mmol_g'}
    xls=pd.ExcelFile(eqp); hit=None
    for sh in xls.sheet_names:
        raw=pd.read_excel(eqp,sheet_name=sh,header=None,nrows=20)
        for i in range(len(raw)):
            vals={str(v).strip() for v in raw.iloc[i].tolist() if pd.notna(v)}
            if req <= vals:
                hit=(sh,i); break
        if hit: break
    if not hit:
        raise HardStop("P0_EQ_SCHEMA: no sheet/header with material,T_C,RH_pct,q_mmol_g.")
    eq=pd.read_excel(eqp,sheet_name=hit[0],header=hit[1]).dropna(axis=1,how='all').dropna(axis=0,how='all').copy()
    eq.columns=[str(c).strip() for c in eq.columns]
    for c in ['T_C','RH_pct','q_mmol_g']:
        eq[c]=pd.to_numeric(eq[c],errors='raise')
    eq['material']=eq['material'].replace(_MATERIAL_NAME_MAP)
    bounds=pd.read_csv(bp)
    need=['material','Tlow','Tref','Thigh','Rmin','Rmax']
    if not set(need)<=set(bounds.columns):
        raise HardStop(f"P0_BOUNDS_SCHEMA: need {need}, got {list(bounds.columns)}")
    for c in need[1:]: bounds[c]=pd.to_numeric(bounds[c],errors='raise')
    return eq,bounds,hit


def _slice_knots(d,T):
    z=d[np.isclose(d['T_C'].to_numpy(float),float(T),atol=1e-9,rtol=0)].sort_values('RH_pct')
    if z.empty: raise HardStop(f"P0_SLICE_MISSING: no measured T slice at {T}")
    if z.duplicated('RH_pct',keep=False).any(): raise HardStop(f"P0_DUPLICATE_RH: T={T}")
    return z.RH_pct.to_numpy(float),z.q_mmol_g.to_numpy(float)


def _interp_rh(rh,x,y):
    rh=np.asarray(rh,float); out=np.full(rh.shape,np.nan,float)
    ok=np.isfinite(rh)&(rh>=x.min())&(rh<=x.max())
    out[ok]=np.interp(rh[ok],x,y)
    return out


def build_representations(material,cfg):
    """Verified canonical P0 v4 contract: single (Tref RH curve) and tensor (piecewise T/RH).
    The tensor representation is the highest-information P0 reference in this lineage.
    """
    eq,bounds,hit=_load_p0_tables()
    b=bounds[bounds.material.astype(str)==str(material)]
    if len(b)!=1: raise HardStop(f"P0_MATERIAL_BOUNDS: expected one bounds row for {material}, got {len(b)}")
    b=b.iloc[0]; d=eq[eq.material.astype(str)==str(material)].copy()
    if d.empty: raise HardStop(f"P0_MATERIAL_MISSING: {material}")
    Ts=np.array(sorted(float(x) for x in d.T_C.unique() if b.Tlow<=float(x)<=b.Thigh),float)
    slices={T:_slice_knots(d,T) for T in Ts}
    xr,yr=_slice_knots(d,b.Tref)
    def q_single(T,RH):
        T=np.asarray(T,float); RH=np.asarray(RH,float)
        out=_interp_rh(RH,xr,yr)
        out[(T<b.Tlow)|(T>b.Thigh)|(RH<b.Rmin)|(RH>b.Rmax)]=np.nan
        return out
    def q_tensor_branch(branch,T,RH):
        T=np.asarray(T,float); RH=np.asarray(RH,float); out=np.full(T.shape,np.nan,float)
        lo,hi=(b.Tlow,b.Tref) if branch=='lower' else ((b.Tref,b.Thigh) if branch=='upper' else (None,None))
        if lo is None: raise ValueError(branch)
        tb=Ts[(Ts>=lo)&(Ts<=hi)]
        if len(tb)<2 or not np.isclose(tb[0],lo) or not np.isclose(tb[-1],hi):
            raise HardStop(f"P0_TENSOR_SLICES: {material}/{branch}: {tb}")
        for j in range(len(tb)-1):
            t0,t1=float(tb[j]),float(tb[j+1]); x0,y0=slices[t0]; x1,y1=slices[t1]
            q0=_interp_rh(RH,x0,y0); q1=_interp_rh(RH,x1,y1)
            itm=(T>=t0)&((T<t1) if j<len(tb)-2 else (T<=t1))
            ok=np.isfinite(q0)&np.isfinite(q1)&itm&(RH>=b.Rmin)&(RH<=b.Rmax)
            if ok.any():
                w=(T[ok]-t0)/(t1-t0); out[ok]=(1-w)*q0[ok]+w*q1[ok]
        return out
    def q_tensor(T,RH):
        T=np.asarray(T,float); RH=np.asarray(RH,float); out=np.full(T.shape,np.nan,float)
        ml=np.isfinite(T)&(T<=b.Tref); mu=np.isfinite(T)&(T>=b.Tref)
        if ml.any(): out[ml]=q_tensor_branch('lower',T[ml],RH[ml])
        if mu.any(): out[mu]=q_tensor_branch('upper',T[mu],RH[mu])
        return out
    domain=dict(T_C=(float(b.Tlow),float(b.Thigh)),RH_pct=(float(b.Rmin),float(b.Rmax)))
    meta=dict(domain=domain,rh_levels=sorted(map(float,d.RH_pct.unique())),t_levels=sorted(map(float,d.T_C.unique())),
              eq_sheet=hit[0],eq_header_row=int(hit[1]))
    return {
        'tensor':dict(q=q_tensor,q_branch=q_tensor_branch,level=0,**meta),
        'single':dict(q=q_single,q_branch=lambda branch,T,RH:q_single(T,RH),level=1,**meta),
    }


def p0_regression_targets(material,reps,cfg):
    """Regression against saved canonical P0 climate scores for the unaggregated 1 h mode.
    This tests both representations on the exact branch/common-transition definition used by P0 v4.
    """
    res=fc.get("resolved"); od=res.get("P0_OUTPUTS")
    candidates=[os.path.join(od,n) for n in ('P0_tensor_climate_scores.csv','P0_tensor_climate_scores_reproduced.csv')] if od else []
    fp=next((p for p in candidates if os.path.exists(p)),None)
    if not fp: raise HardStop("P0_REGRESSION_TARGET_MISSING: canonical P0 climate-score file not found.")
    ref=pd.read_csv(fp)
    req={'material','branch','temporal_mode','city','H_single','H_tensor'}
    if not req<=set(ref.columns): raise HardStop(f"P0_REGRESSION_SCHEMA: missing {sorted(req-set(ref.columns))}")
    ref=ref[(ref.material.astype(str)==str(material))&(ref.temporal_mode.astype(str)=='subsample_1h')].copy()
    w,_,_,_=load_climate_weather(cfg)
    rows=[]
    for r in ref.itertuples(index=False):
        g=w[w.climate.astype(str)==str(r.city)].sort_values('timestamp')
        T=g.T_C.to_numpy(float); RH=g.RH_pct.to_numpy(float)
        qs=np.asarray(reps['single']['q_branch'](r.branch,T,RH),float)
        qt=np.asarray(reps['tensor']['q_branch'](r.branch,T,RH),float)
        ok=np.isfinite(qs)&np.isfinite(qt)&np.isfinite(RH); tr=ok[:-1]&ok[1:]
        if not tr.any(): hs=ht=np.nan
        else:
            hs=float(0.5*np.abs(np.diff(qs)[tr]).sum()); ht=float(0.5*np.abs(np.diff(qt)[tr]).sum())
        key=f"{r.branch}|subsample_1h|{r.city}"
        rows += [dict(rep='single',key=key,reproduced=hs,reference=float(r.H_single)),
                 dict(rep='tensor',key=key,reproduced=ht,reference=float(r.H_tensor))]
    return pd.DataFrame(rows)


# ----------------------------------------------------------------------------- canonical IRA v7
def ira_continuous_propagator():
    ns=load_ira_continuous()
    fn=ns.get('propagate_city')
    if not callable(fn): raise HardStop("IRA_CONTINUOUS_V7_ADAPTER_UNBOUND: propagate_city not found.")
    return fn,'propagate_city'


# ----------------------------------------------------------------------------- existing active-process final output
def load_active_optimized_scores(cfg):
    """Read the existing three-material process bridge's climate-level optimized-productivity output.
    No reimplementation: P_star_single/P_star_tensor are canonical outputs of the existing bridge.
    """
    res=fc.get('resolved'); p=res.get('FINAL:optimized_productivity_climate_scores.csv')
    if not p or not os.path.exists(p): raise HardStop("ACTIVE_OUTPUT_MISSING: optimized_productivity_climate_scores.csv")
    d=pd.read_csv(p)
    need={'material','branch','temporal_mode','city','P_star_single','P_star_tensor'}
    if not need<=set(d.columns): raise HardStop(f"ACTIVE_OUTPUT_SCHEMA: missing {sorted(need-set(d.columns))}")
    d=d[d.material.astype(str).str.contains('IRA',case=False,regex=False)].copy()
    rows=[]
    for rep,col in [('single','P_star_single'),('tensor','P_star_tensor')]:
        z=d[['branch','temporal_mode','city',col]].rename(columns={'city':'climate',col:'Y_active'}).copy()
        z['representation']=rep; rows.append(z)
    out=pd.concat(rows,ignore_index=True)
    out['regen_kernel']='primary'; out['bridge_endpoint']='P_star_optimized_productivity'
    return out,p


def bridge_active_scores(reps,weather,kernels,regen_kernel_label,cfg):
    # Retained only for compatibility with old cells. v1.2 uses load_active_optimized_scores directly.
    d,p=load_active_optimized_scores(cfg)
    out={}
    for rep in reps:
        z=d[d.representation==rep].copy()
        # This compatibility view is only valid when a caller has already selected one branch/mode.
        if z[['branch','temporal_mode']].drop_duplicates().shape[0] != 1:
            raise HardStop("ACTIVE_VARIANT_UNRESOLVED: select branch/temporal_mode; use load_active_optimized_scores().")
        out[rep]=z[['climate','Y_active']]
    return out,os.path.basename(p)


def bridge_continuous_kinetics(cfg):
    try:
        return ira_continuous_propagator()
    except HardStop:
        return None,None



# ----------------------------------------------------------------------------- NASA POWER solar/wind augmentation
def _power_hourly_json(latitude, longitude, start, end, cfg):
    """Fetch NASA POWER hourly ALLSKY_SFC_SW_DWN and WS10M in UTC.
    T/RH are intentionally not requested here: canonical P0 T/RH remain frozen from MAIN_climate_hourly_2015_2024.csv.
    """
    import time, requests
    base=cfg.get('NASA_POWER_API_BASE','https://power.larc.nasa.gov/api/temporal/hourly/point')
    params=dict(
        parameters=','.join(cfg.get('NASA_POWER_PARAMETERS',['ALLSKY_SFC_SW_DWN','WS10M'])),
        community=cfg.get('NASA_POWER_COMMUNITY','RE'),
        longitude=float(longitude), latitude=float(latitude),
        start=str(start), end=str(end), format='JSON',
        **{'time-standard':'UTC'},
    )
    timeout=float(cfg.get('NASA_POWER_TIMEOUT_S',180))
    retries=int(cfg.get('NASA_POWER_MAX_RETRIES',4))
    last=None
    for k in range(retries):
        try:
            r=requests.get(base,params=params,timeout=timeout)
            if r.status_code==200:
                return r.json(), r.url
            last=f'HTTP {r.status_code}: {r.text[:500]}'
            if r.status_code in (400,401,403,404,422):
                break
        except Exception as e:
            last=f'{type(e).__name__}: {e}'
        time.sleep(min(2**k,8))
    raise HardStop(f'NASA_POWER_REQUEST_FAILED: {last}')


def _power_json_to_frame(payload, city, latitude, longitude):
    try:
        par=payload['properties']['parameter']
    except Exception as e:
        raise HardStop(f'NASA_POWER_SCHEMA: missing properties.parameter for {city}: {e}')
    need=['ALLSKY_SFC_SW_DWN','WS10M']
    miss=[x for x in need if x not in par]
    if miss:
        raise HardStop(f'NASA_POWER_SCHEMA: {city} missing {miss}; available={list(par)}')
    keys=sorted(set(par['ALLSKY_SFC_SW_DWN']) | set(par['WS10M']))
    rows=[]
    for k in keys:
        if not re.fullmatch(r'\d{10}',str(k)):
            continue
        ts=pd.to_datetime(str(k),format='%Y%m%d%H',utc=True,errors='coerce')
        if pd.isna(ts):
            continue
        g=pd.to_numeric(pd.Series([par['ALLSKY_SFC_SW_DWN'].get(k)]),errors='coerce').iloc[0]
        v=pd.to_numeric(pd.Series([par['WS10M'].get(k)]),errors='coerce').iloc[0]
        # POWER missing-value sentinel is commonly -999; any negative G or wind is nonphysical here and treated missing.
        if not np.isfinite(g) or g < 0: g=np.nan
        if not np.isfinite(v) or v < 0: v=np.nan
        rows.append(dict(climate=str(city),timestamp=ts,G_Wm2=float(g) if np.isfinite(g) else np.nan,
                         v_ms=float(v) if np.isfinite(v) else np.nan,lat=float(latitude),lon=float(longitude)))
    z=pd.DataFrame(rows)
    if z.empty:
        raise HardStop(f'NASA_POWER_EMPTY: no hourly rows parsed for {city}')
    if z.duplicated(['climate','timestamp']).any():
        raise HardStop(f'NASA_POWER_DUPLICATE_KEYS: {city}')
    return z


def download_nasa_power_solar_wind(cfg, canonical_weather):
    """Augment canonical 36-city P0 weather with NASA POWER GHI and 10-m wind only.

    Coordinates and UTC timestamps come from the frozen canonical weather file. The returned frame is
    eligible for an exact-key join only; no temporal interpolation, nearest-time matching, gap filling,
    coordinate substitution, or T/RH replacement is performed.
    """
    w=canonical_weather.copy()
    req={'climate','timestamp','lat','lon'}
    if not req<=set(w.columns):
        raise HardStop(f'NASA_POWER_COORDINATES_MISSING: canonical weather requires {sorted(req)}, got {list(w.columns)}')
    if w[['lat','lon']].isna().any().any():
        raise HardStop('NASA_POWER_COORDINATES_MISSING: canonical weather contains missing lat/lon')
    w['timestamp']=pd.to_datetime(w.timestamp,utc=True)
    cache=cfg['NASA_POWER_CACHE_DIR']; os.makedirs(cache,exist_ok=True)
    merged_path=cfg['NASA_POWER_MERGED_PATH']; os.makedirs(os.path.dirname(merged_path),exist_ok=True)
    force=bool(cfg.get('NASA_POWER_FORCE_REDOWNLOAD',False))
    canonical_keys=w[['climate','timestamp']].drop_duplicates()
    if canonical_keys.duplicated(['climate','timestamp']).any():
        raise HardStop('NASA_POWER_CANONICAL_DUPLICATE_KEYS')
    if os.path.exists(merged_path) and not force:
        old=pd.read_csv(merged_path)
        need={'climate','timestamp','G_Wm2','v_ms'}
        if need<=set(old.columns):
            old['timestamp']=pd.to_datetime(old.timestamp,utc=True,errors='coerce')
            if (not old.duplicated(['climate','timestamp']).any() and old[['G_Wm2','v_ms']].notna().all().all()):
                chk=canonical_keys.merge(old[['climate','timestamp']],on=['climate','timestamp'],how='left',indicator=True,validate='one_to_one')
                if (chk._merge=='both').all():
                    print('Using cached NASA POWER solar/wind file:', merged_path)
                    return old, merged_path
        print('Ignoring structurally incomplete NASA POWER merged cache and rebuilding:', merged_path)
    t0=w.timestamp.min(); t1=w.timestamp.max()
    start=t0.strftime('%Y%m%d'); end=t1.strftime('%Y%m%d')
    rows=[]
    for city,g in w.groupby('climate',sort=False):
        ll=g[['lat','lon']].drop_duplicates()
        if len(ll)!=1:
            raise HardStop(f'NASA_POWER_COORDINATES_AMBIGUOUS: {city} has {len(ll)} coordinate pairs')
        lat=float(ll.iloc[0].lat); lon=float(ll.iloc[0].lon)
        safe=re.sub(r'[^A-Za-z0-9_.-]+','_',str(city))
        target=os.path.join(cache,f'{safe}_{start}_{end}_POWER_G_WS10M.csv')
        if os.path.exists(target) and not force:
            z=pd.read_csv(target); z['timestamp']=pd.to_datetime(z.timestamp,utc=True,errors='coerce')
        else:
            print('NASA POWER request',city,lat,lon,start,end)
            try:
                payload,url=_power_hourly_json(lat,lon,start,end,cfg)
                z=_power_json_to_frame(payload,city,lat,lon)
            except HardStop as e:
                # Some POWER deployments can reject very long hourly requests. Fall back to year chunks without changing semantics.
                if 'NASA_POWER_REQUEST_FAILED' not in str(e): raise
                parts=[]
                for yr in range(t0.year,t1.year+1):
                    ys=max(t0,pd.Timestamp(f'{yr}-01-01',tz='UTC')).strftime('%Y%m%d')
                    ye=min(t1,pd.Timestamp(f'{yr}-12-31 23:00:00',tz='UTC')).strftime('%Y%m%d')
                    payload,url=_power_hourly_json(lat,lon,ys,ye,cfg)
                    parts.append(_power_json_to_frame(payload,city,lat,lon))
                z=pd.concat(parts,ignore_index=True).drop_duplicates(['climate','timestamp'])
            z.to_csv(target,index=False)
        # Exact canonical timestamp support; extra POWER rows are harmless and removed here.
        ck=g[['climate','timestamp']].drop_duplicates()
        zz=ck.merge(z[['climate','timestamp','G_Wm2','v_ms']],on=['climate','timestamp'],how='left',indicator=True,validate='one_to_one')
        missing_key=int((zz._merge!='both').sum())
        missing_val=int(zz[['G_Wm2','v_ms']].isna().any(axis=1).sum())
        if missing_key or missing_val:
            raise HardStop(f'NASA_POWER_CITY_TIME_COVERAGE_MISMATCH: {city}: missing_keys={missing_key}, missing_values={missing_val}, canonical_hours={len(ck)}')
        zz=zz.drop(columns='_merge')
        if not (0<=zz.G_Wm2.min() and zz.G_Wm2.max()<1600 and 0<=zz.v_ms.min() and zz.v_ms.max()<80):
            raise HardStop(f'NASA_POWER_UNIT_RANGE: {city} G=[{zz.G_Wm2.min()},{zz.G_Wm2.max()}] W/m2, v=[{zz.v_ms.min()},{zz.v_ms.max()}] m/s')
        zz['lat']=lat; zz['lon']=lon
        zz['source_dataset']='NASA POWER Hourly API'; zz['solar_parameter']='ALLSKY_SFC_SW_DWN'; zz['wind_parameter']='WS10M'; zz['time_standard']='UTC'
        rows.append(zz)
    out=pd.concat(rows,ignore_index=True)
    if out.duplicated(['climate','timestamp']).any():
        raise HardStop('NASA_POWER_DUPLICATE_KEYS: duplicate (climate,timestamp) after concatenation')
    if len(out)!=len(canonical_keys):
        raise HardStop(f'NASA_POWER_GLOBAL_KEY_COUNT_MISMATCH: {len(out)} vs canonical {len(canonical_keys)}')
    out.to_csv(merged_path,index=False)
    return out, merged_path


In [ ]:
# Section 0 — frozen configuration. Edit ONLY the binding/path fields (marked BIND) before the first run.
# Scientific settings (marked FROZEN) must not be changed after results are viewed (§43).
import json, os, datetime
CONFIG = dict(
    VERSION="v1.6.5-climate-cache-key-fix",
    FROZEN_AT=datetime.datetime.now().isoformat(timespec="seconds"),
    # ---------------- BIND: Repository discovery / resolution ----------------
    REPO_ROOT="/content/drive/MyDrive",  # Drive Repository root verified 2026-11-13
    RESUME_RUN_DIR=None,                 # set to an existing run dir to continue it after a Colab restart
    SEARCH_ROOTS=["/content/drive/MyDrive", "/content/drive/Shareddrives", "/content/drive/Othercomputers"],
    AUDIT_FILENAME="process_dataset_9_materials_exhaustive_audit.xlsx",
    AUDIT_DRIVE_ID="1_hDXMRxNa1Wuv9ayWsTWwnrPsJ6n0klL",
    EQ_MASTER_FILENAME="MAIN_equilibrium_dataset_master.xlsx",
    EQ_MASTER_PATH="/content/drive/MyDrive/climate_screening_v1_3_1_uncertainty_audit/climate_screening_backbone/recent_experimental_results/manuscript_required_outputs_only/results/SI_machine_readable_supporting_data/MAIN_equilibrium_dataset_master.xlsx",
    EQ_MASTER_DRIVE_ID=None,  # path fixed from canonical P0 v4 lineage
    P0_CODE_PATH="/content/drive/MyDrive/codes/CODE_P0_tensor_reproduction_v4_regression_repair.ipynb",
    P0_OUTPUTS_DIR="/content/drive/MyDrive/climate_screening_v1_3_1_uncertainty_audit/climate_screening_backbone/recent_experimental_results/manuscript_required_outputs_only/results/P0_submission_gate_outputs_2026-10-14/P0_tensor_outputs",
    WEATHER_PATH="/content/drive/MyDrive/climate_screening_v1_3_1_uncertainty_audit/climate_screening_backbone/recent_experimental_results/manuscript_required_outputs_only/results/SI_machine_readable_supporting_data/MAIN_climate_hourly_2015_2024.csv",
    SOLAR_WIND_PATH=None,                # optional prebuilt G/v source for the SAME 36 climates and exact timestamps
    NASA_POWER_AUTO_DOWNLOAD=True,       # primary augmentation when canonical file lacks G/v
    NASA_POWER_API_BASE="https://power.larc.nasa.gov/api/temporal/hourly/point",
    NASA_POWER_PARAMETERS=["ALLSKY_SFC_SW_DWN", "WS10M"],
    NASA_POWER_COMMUNITY="RE",
    NASA_POWER_CACHE_DIR="/content/drive/MyDrive/climate_screening_v1_3_1_uncertainty_audit/climate_screening_backbone/recent_experimental_results/manuscript_required_outputs_only/results/NASA_POWER_G_v_2015_2024",
    NASA_POWER_MERGED_PATH="/content/drive/MyDrive/climate_screening_v1_3_1_uncertainty_audit/climate_screening_backbone/recent_experimental_results/manuscript_required_outputs_only/results/NASA_POWER_G_v_2015_2024/NASA_POWER_36city_G_v_2015_2024.csv",
    NASA_POWER_FORCE_REDOWNLOAD=False,
    NASA_POWER_TIMEOUT_S=180,
    NASA_POWER_MAX_RETRIES=4,
    WEATHER_TIMESTAMPS_ARE_LOCAL=False,  # canonical timestamp_utc is UTC; civil time is obtained with WEATHER_CITY_TIMEZONES
    WEATHER_UTC_OFFSET_H=None,           # fixed offsets are NOT used for this dataset because DST matters
    WEATHER_CITY_TIMEZONES=dict(
        adelaide="Australia/Adelaide", amsterdam="Europe/Amsterdam", athens="Europe/Athens", auckland="Pacific/Auckland",
        beijing="Asia/Shanghai", belem="America/Belem", dalian="Asia/Shanghai", doha="Asia/Qatar", dubai="Asia/Dubai",
        dublin="Europe/Dublin", harbin="Asia/Shanghai", helsinki="Europe/Helsinki", jakarta="Asia/Jakarta", jeddah="Asia/Riyadh",
        kuala_lumpur="Asia/Kuala_Lumpur", las_vegas="America/Los_Angeles", lisbon="Europe/Lisbon", london="Europe/London",
        madrid="Europe/Madrid", manaus="America/Manaus", medan="Asia/Jakarta", minneapolis="America/Chicago",
        montreal="America/Toronto", moscow="Europe/Moscow", ottawa="America/Toronto", paris="Europe/Paris",
        phoenix="America/Phoenix", pyongyang="Asia/Pyongyang", riyadh="Asia/Riyadh", rome="Europe/Rome",
        sacramento="America/Los_Angeles", seoul="Asia/Seoul", shenyang="Asia/Shanghai", singapore="Asia/Singapore",
        vancouver="America/Vancouver", warsaw="Europe/Warsaw"),
    WEATHER_PROVENANCE="DIRECT_REPORTED_MEASURED", # canonical frozen hourly weather used by P0
    IRA_KERNEL_PROVENANCE="DIGITIZED_FROM_FIGURE", # bridge kernels are literature-digitized process data
    BRIDGE_DIR_PATH="/content/drive/MyDrive/climate_screening_v1_3_1_uncertainty_audit/climate_screening_backbone/process_bridge/results/three_material_cycle_productivity_bridge",
    BRIDGE_FINAL_DIR_PATH="/content/drive/MyDrive/climate_screening_v1_3_1_uncertainty_audit/climate_screening_backbone/process_bridge/results/three_material_cycle_productivity_bridge_final",
    IRA_CONTINUOUS_CODE_PATH="/content/drive/MyDrive/codes/IRA_continuous_weather_process_bridge_v7.ipynb",
    TPD_PASS_CURVES_PATH="/content/drive/MyDrive/thermal_process_highres_QA_pass_curves_2026-11-13.csv",
    TPD_QA_MANIFEST_PATH="/content/drive/MyDrive/thermal_process_highres_QA_manifest_2026-11-13.csv",
    P0_BINDINGS=dict(                    # v1.2 adapter wraps the actual P0 v4 functions/data contract
        build_representations=["eval_single", "eval_tensor"],
        regression_targets=["trajectory_scores", "regression_compare"],
        weather_loader=[]),
    BRIDGE_BINDINGS=dict(
        active_cycle_scores=["optimized_productivity_climate_scores.csv"],
        continuous_kinetics=["propagate_city"],
        score_column=None),
    FULL_REPRESENTATION_NAME="tensor",
    MATERIALS=dict(IRA="IRA-900-C", ImCOF="ImCOF-TAEA"),
    WEATHER_COLUMNS=dict(climate=None, timestamp=None, T_C=None, RH_pct=None, G_Wm2=None, v_ms=None),
    IRA_KERNEL_COLUMNS=dict(time=None, fraction=None), IRA_KERNEL_TIME_UNIT=None,   # "s" | "min" | "h"
    IRA_MEASURED_RH_LEVELS=None,         # list, only if P0 representation metadata lacks 'rh_levels'
    TPD_COLUMNS=dict(curve_id=None, T_C=None, time_min=None, signal=None, T_ads_C=None, RH_ads_pct=None,
                     heating_rate=None, adsorbed_amount=None),
    TPD_SIGNAL_KIND="concentration",     # verified high-res ImCOF S20 ordinate is CO2 concentration (ppm)
    PADAC_PDF_PATH=None, IMCOF_PDF_PATHS=None,
    PADAC_FIELD_FORCING_PATH=None,       # exact March 17-19 hourly T_amb, G, v (+ optional measured T_s)
    PADAC_FIELD_FORCING_PROVENANCE=None, PADAC_FIELD_TARGET_PROVENANCE=None,
    PADAC_FIELD_SAME_FIGURE=None,        # True if forcing and targets are digitized from the same figure
    N_JOBS=2,
    RUN_PADAC_THERMAL_SENSITIVITY=False,  # primary S21 runs only the first thermal variant; set True to run S21b sensitivity
    # ---------------- FROZEN: tolerances, endpoints, controller families ----------------
    N_CLIMATES_EXPECTED=36, CLIMATE_ORDER=None,
    P0_REGRESSION_RTOL=1e-6, P0_REGRESSION_ATOL=1e-9, BRIDGE_REGRESSION_RTOL=1e-4,
    EXTRAPOLATION_POLICY="P0_RULE",      # preserve P0 no-extrapolation rule: unsupported states remain NaN
    IRA_ENV_PRIMARY_ENDPOINT="continuous_weather_process_endpoint",
    IRA_KINETICS_PRIMARY="repository_continuous_v7",
    IRA_DWELL_FROM_KERNEL_FRACTIONS=[],  # v7 explicitly prohibits RH-threshold state conversion; retained empty
    ADSORPTION_WINDOW_LOCAL_H=[18, 8],               # PADAC night adsorption (fan on) / day regeneration
    ADSORPTION_WINDOW_SENSITIVITY=[[20, 8]],
    IMCOF_ADS_CONDITION="end_of_window",             # sensitivity: "window_mean"
    TPD_PRIMARY_HEATING_RATE=0.5, TPD_HEATING_RATE_TOL=0.05,
    TPD_BASELINE="linear_endpoints", TPD_NORMALIZATION="adsorbed_amount_if_available",
    TPD_KERNEL_SELECTION="nearest_measured_condition", TPD_RH_SCALE_PCT=10.0, TPD_T_SCALE_C=10.0,
    IMCOF_ACTIVE=dict(T_reg_C=80.0, duration_h=2.0, N2_sccm=60.0),
    IMCOF_SUPPORT=dict(T_C=(-20.0, 25.0), RH_pct=(0.0, 70.0)),
    PADAC_NZ=20,
    IMCOF_ENV_THERMAL_MODE="padac_13X_controller", # primary: exogenous T_s from field-tested PADAC+13X thermal realisation
    # Alternative "material_specific" requires ImCOF C_s/rho_bulk/eps_bed/d_p/k_s or explicit sensitivity theta.
    PADAC_CONTROLLER_CACHE_DIR=None,      # None -> RUN_DIR/thermal_runs_PADAC13X; set Drive path to persist across runs
    # Solver optimization changes orchestration only: no hourly forcing/output is aggregated or dropped.
    PADAC_SOLVER_MODE="optimized_if_regression_passes",
    PADAC_OPT_CHUNK_HOURS=744,            # 31 days; state is carried exactly between chunks
    PADAC_OPT_MAX_STEP_S=None,            # None preserves adaptive BDF stepping; equivalence gate decides acceptance
    PADAC_OPT_VALIDATION_DAYS=7,          # regression window per selected climate
    PADAC_OPT_VALIDATION_N_CLIMATES=3,    # low/median/high mean-solar climates
    PADAC_OPT_MAX_ABS_T_C=0.05,           # computational-equivalence gate, not a physical uncertainty
    PADAC_OPT_RMSE_T_C=0.01,
    # ---------------- PADAC (Filahi et al., ECM 338 (2025) 119925) ----------------
    PADAC=dict(  # value, unit, provenance, side
        L=(0.35, "m", "DIRECT_REPORTED_DESIGN", "reactor"), D_int=(0.100, "m", "DIRECT_REPORTED_DESIGN", "reactor"),
        D_ext=(0.1015, "m", "DIRECT_REPORTED_DESIGN", "reactor"), e_v=(0.008, "m", "DIRECT_REPORTED_DESIGN", "reactor"),
        e_w=(0.002, "m", "DIRECT_REPORTED_DESIGN", "reactor"), tilt_deg=(30.0, "deg", "DIRECT_REPORTED_DESIGN", "reactor"),
        m_s_13X=(1.5, "kg", "DIRECT_REPORTED_DESIGN", "sorbent"), Q_m3_h=(12.0, "m3/h", "DIRECT_REPORTED_DESIGN", "reactor"),
        alpha_v=(0.06, "-", "LITERATURE_PARAMETER", "reactor"), eps_v=(0.9, "-", "LITERATURE_PARAMETER", "reactor"),
        tau_g=(0.95, "-", "LITERATURE_PARAMETER", "reactor"), alpha_w=(0.9, "-", "LITERATURE_PARAMETER", "reactor"),
        eps_w=(0.9, "-", "LITERATURE_PARAMETER", "reactor"),
        sky_coeff=(0.0522, "K^-0.5", "LITERATURE_PARAMETER", "reactor"),    # as printed (Eq. 7); Swinbank original 0.0552 -> sensitivity
        h_wind_table3=(23.8, "W/m2/K", "FITTED_BY_SOURCE_AUTHORS", "reactor"),
        h_w_fitted=(1.83, "W/m2/K", "FITTED_BY_SOURCE_AUTHORS", "reactor"),
        lambda_gz_fitted=(0.11, "W/m/K", "FITTED_BY_SOURCE_AUTHORS", "reactor"),
        h_gs_13X=(148.61, "W/m2/K", "FITTED_BY_SOURCE_AUTHORS", "sorbent"),
        C_s_13X=(880.0, "J/kg/K", "DIRECT_REPORTED_MEASURED", "sorbent"),
        alpha_s_13X=(1.60e-7, "m2/s", "DIRECT_REPORTED_MEASURED", "sorbent"),
        minus_dH_13X=(32682.0, "J/mol", "FITTED_BY_SOURCE_AUTHORS", "sorbent"),
        rho_v=(2230.0, "kg/m3", "ASSUMED_SENSITIVITY_PARAMETER", "reactor"),  # borosilicate; replace if paper reports
        C_v=(750.0, "J/kg/K", "ASSUMED_SENSITIVITY_PARAMETER", "reactor"),
        rho_w=(7900.0, "kg/m3", "ASSUMED_SENSITIVITY_PARAMETER", "reactor"),  # steel; replace if paper reports
        C_w=(500.0, "J/kg/K", "ASSUMED_SENSITIVITY_PARAMETER", "reactor"),
        eps_bed_13X=(0.40, "-", "ASSUMED_SENSITIVITY_PARAMETER", "sorbent"),  # replace by Table-3 value after PDF check
        d_p_13X=(0.003, "m", "ASSUMED_SENSITIVITY_PARAMETER", "sorbent"),     # replace by Table-3 value after PDF check
        rho_p_13X=(1100.0, "kg/m3", "ASSUMED_SENSITIVITY_PARAMETER", "sorbent"),  # only enters k_s = alpha_s*rho_p*C_s
    ),
    # Expressions whose printed form must be verified against the PADAC PDF (variables: D_int, D_ext, e_v, e_w, L, np).
    PADAC_FORM=dict(
        a_v="1/e_v",
        wall_solar_factor="D_ext/(np.pi*(D_ext**2-D_int**2)/4)",      # projected absorber width / wall cross-section
        wall_rad_factor="np.pi*D_ext/(np.pi*(D_ext**2-D_int**2)/4)",  # outer perimeter / wall cross-section
        wall_gas_factor="np.pi*D_int/(np.pi*(D_ext**2-D_int**2)/4)",  # inner perimeter / wall cross-section
        gas_wall_factor="4/D_int",
        wind_correlation="watmuff", coefficient_mode="table3_fitted",
        eq6_glass_wall_sign="physical", gas_solid_sign_factor=1.0),
    PADAC_PDF_VERIFIED=dict(eq6=False, eq7=False, eq8=False, eq9=False, eq10=False, eq11=False, eq12=False,
                            eq13=False, eq14=False, eq15=False, bc=False, eq16=False, eq17_20=False),
    TOTH_13X=dict(minus_dH=32682.0, q_m0=7.88, K0=115.853, n0=0.35, gamma=-2.34, beta=-0.72,
                  T0=None, k_L=None, isotherm_form=None, q_m_form=None, K_form=None, n_form=None),
    # Variables in Toth forms: T, P_bar, T0, q_m0, K0, n0, gamma, beta, dH, R, np, q_m, K, n. Fill ONLY from the PDF.
    THERMAL_VARIANTS=[   # model-form sensitivity, all frozen a priori; the first is primary
        dict(id="primary", wind_correlation="watmuff", sky_coeff=None, coefficient_mode="table3_fitted", formulation="B"),
        dict(id="wind_kumar", wind_correlation="kumar", sky_coeff=None, coefficient_mode="table3_fitted", formulation="B"),
        dict(id="wind_mcadams_le5", wind_correlation="mcadams_low", sky_coeff=None, coefficient_mode="table3_fitted", formulation="B"),
        dict(id="sky_0.0552", wind_correlation="watmuff", sky_coeff=0.0552, coefficient_mode="table3_fitted", formulation="B"),
        dict(id="coeff_correlation", wind_correlation="watmuff", sky_coeff=None, coefficient_mode="correlation", formulation="B"),
        dict(id="formulation_A", wind_correlation="watmuff", sky_coeff=None, coefficient_mode="table3_fitted", formulation="A"),
    ],
    WIND_VALIDITY_MS=dict(watmuff=(0, 7), kumar=(0, 4), mcadams_low=(0, 5), table3_constant=(0, 1e9)),
    # ---------------- ImCOF sorbent-side (never inherited from 13X) ----------------
    IMCOF_THERMAL_PROPS=dict(C_s=None, rho_bulk=None, rho_p=None, eps_bed=None, k_s=None, alpha_s=None, d_p=None, dH_ads=None),
    # each entry: None (NOT_REPORTED) or dict(value=..., unit=..., cls="DIRECT_IMCOF"|"DERIVABLE_IMCOF", source="SI p.X / Table Y")
    IMCOF_THERMAL_THETA=None,
    # list of realizations; each a dict of ALL unresolved props with explicit values, e.g.
    # dict(id="theta01", C_s=..., rho_bulk=..., eps_bed=..., d_p=..., k_s=..., h_gs=None,
    #      provenance="ASSUMED_SENSITIVITY_PARAMETER", rationale="range from <cited source>")
    IMCOF_TPD_ED=dict(dry_25C=77.9, RH70_25C=94.8),   # kJ/mol, TPD descriptors ONLY (never used as ΔH)
    IMCOF_ANCHORS=dict(T25=dict(RH0=(0.46, 0.02), RH30=(0.53, 0.07), RH50=(0.76, 0.03), RH70=(1.09, 0.06)),
                       RH70=dict(Tm20=0.11, Tm5=1.03, T5=1.12, T15=1.25, T25=1.09)),
)
json.dump(CONFIG, open("/content/fidelity_config.json", "w"), indent=2, default=list)
print("CONFIG written. Frozen at", CONFIG["FROZEN_AT"])


In [ ]:
import sys, os, importlib
sys.path.insert(0, "/content")
import numpy as np, pandas as pd
import fidelity_core as fc, fidelity_adapters as ad
importlib.reload(fc); importlib.reload(ad)
CFG = fc.load_config()
S = "S1"
# 1. Drive mount
from google.colab import drive
drive.mount('/content/drive')
import sys
sys.path.insert(0, "/content")
if CFG.get("RESUME_RUN_DIR"):
    json_path = os.path.join(CFG["RESUME_RUN_DIR"], "session.json")
    import json
    fc.update_session(**json.load(open(json_path)))
    print("Resumed run", CFG["RESUME_RUN_DIR"])


In [ ]:
import sys, os, importlib
sys.path.insert(0, "/content")
import numpy as np, pandas as pd
import fidelity_core as fc, fidelity_adapters as ad
importlib.reload(fc); importlib.reload(ad)
CFG = fc.load_config()
S = "S2"
# 2. Repository discovery — by canonical audit workbook + Drive ID, never "first search hit".
import datetime, re
cands = []
if CFG["REPO_ROOT"]:
    root = CFG["REPO_ROOT"]
    if not os.path.isfile(os.path.join(root, CFG["AUDIT_FILENAME"])):
        raise fc.HardStop(f"REPO_ROOT_INVALID: {CFG['AUDIT_FILENAME']} not at configured REPO_ROOT root.")
    reason = "configured"
else:
    hits = fc.walk_find(CFG["SEARCH_ROOTS"], "^" + re.escape(CFG["AUDIT_FILENAME"]) + "$")
    for h in hits:
        cands.append(dict(path=h, drive_id=fc.drive_id(h), size=os.path.getsize(h)))
    print(pd.DataFrame(cands).to_string() if cands else "no audit workbook found")
    match = [c for c in cands if c["drive_id"] == CFG["AUDIT_DRIVE_ID"]]
    if len(match) == 1:
        root, reason = os.path.dirname(match[0]["path"]), "drive_id_match"
    elif len(cands) == 1 and cands[0]["drive_id"] is None:
        root, reason = os.path.dirname(cands[0]["path"]), "single_candidate_drive_id_unverifiable"
    else:
        raise fc.HardStop(f"AMBIGUOUS_REPOSITORY_ROOT: {len(cands)} audit workbooks, {len(match)} with the canonical Drive ID. "
                          "Set CONFIG['REPO_ROOT'].")
run = CFG.get("RESUME_RUN_DIR") or os.path.join(
    root, "process_fidelity_runs", datetime.datetime.now().strftime("%Y%m%d_%H%M%S") + "_" + CFG["VERSION"])
fc.update_session(REPO_ROOT=root, RUN_DIR=run, REPO_REASON=reason, CONFIG_FROZEN_AT=CFG["FROZEN_AT"])
import shutil, json
os.makedirs(os.path.join(run, "figures"), exist_ok=True)
if not os.path.exists(os.path.join(run, "frozen_config.json")):   # never overwritten on resume
    shutil.copy("/content/fidelity_config.json", os.path.join(run, "frozen_config.json"))
fc.put("discovery", dict(candidates=cands, root=root, reason=reason))
if reason != "drive_id_match" and reason != "configured":
    fc.warn("DRIVE_ID_UNVERIFIABLE", "Drive ID xattr not exposed; single audit workbook accepted.", S)
print("REPO_ROOT =", root, "| RUN_DIR =", run)


In [ ]:
import sys, os, importlib
sys.path.insert(0, "/content")
import numpy as np, pandas as pd
import fidelity_core as fc, fidelity_adapters as ad
importlib.reload(fc); importlib.reload(ad)
CFG = fc.load_config()
S = "S3"
# 3. Canonical source resolution (lineage, not filename order).
import re, glob
sess = fc.load_session(); ROOT = sess["REPO_ROOT"]
res, cand_rows = {}, []

def note(role, paths, chosen, reason):
    for p in paths:
        cand_rows.append(dict(role=role, path=p, drive_id=fc.drive_id(p) if os.path.exists(p) else None,
                              chosen=(p == chosen), reason=reason))

def unique_by_content(paths):
    """Identical-content copies are one source; different content is a real ambiguity."""
    if len(paths) <= 1:
        return paths
    hashes = {p: fc.sha256(p) if os.path.isfile(p) else p for p in paths}
    return paths if len(set(hashes.values())) > 1 else [sorted(paths, key=lambda p: (not p.startswith(ROOT), len(p)))[0]]

def resolve_exact(role, name, roots, dirs=False, required=True):
    hits = [h for h in fc.walk_find(roots, "^" + re.escape(name) + r"(\.\w+)?$", dirs=dirs) if "_SOURCE" not in os.path.basename(h)]
    u = unique_by_content(hits) if not dirs else hits
    chosen = u[0] if len(u) == 1 else None
    note(role, hits, chosen, "unique" if chosen else ("missing" if not hits else "AMBIGUOUS"))
    if chosen is None and required:
        fc.warn(f"UNRESOLVED_{role}", f"{name}: {len(hits)} candidates -> not resolved", S)
    return chosen

# 3.1 audit baseline (repo root only; *_SOURCE copies never count)
res["AUDIT"] = os.path.join(ROOT, CFG["AUDIT_FILENAME"])
note("AUDIT", [res["AUDIT"]], res["AUDIT"], sess["REPO_REASON"])

# 3.2 equilibrium master: lineage from the audit workbook and existing audit/provenance records
eq_hits = fc.walk_find(CFG["SEARCH_ROOTS"], "^" + re.escape(CFG["EQ_MASTER_FILENAME"]) + "$")
refs = []
try:
    for sh, df in pd.read_excel(res["AUDIT"], sheet_name=None, header=None).items():
        for v in df.astype(str).values.ravel():
            if "MAIN_equilibrium_dataset_master" in v or re.search(r"\b[0-9A-Za-z_-]{25,44}\b", v) and "equilibrium" in v.lower():
                refs.append(("audit:" + sh, v))
except Exception as e:
    fc.warn("AUDIT_READ", f"audit workbook not readable for lineage: {e}", S)
for p in fc.walk_find([ROOT], r"(audit|provenance|lineage|manifest).*\.(csv|md|txt|json)$"):
    try:
        txt = open(p, errors="ignore").read()
    except Exception:
        continue
    for line in txt.splitlines():
        if "MAIN_equilibrium_dataset_master" in line:
            refs.append((p, line.strip()[:500]))
ids = set(re.findall(r"[0-9A-Za-z_-]{25,44}", " ".join(r[1] for r in refs)))
frags = set(m.group(0) for r in refs for m in re.finditer(r"[\w ./-]*MAIN_equilibrium_dataset_master\.xlsx", r[1]))
def lineage_ok(p):
    if CFG["EQ_MASTER_PATH"]:
        return os.path.abspath(p) == os.path.abspath(CFG["EQ_MASTER_PATH"])
    if CFG["EQ_MASTER_DRIVE_ID"]:
        return fc.drive_id(p) == CFG["EQ_MASTER_DRIVE_ID"]
    did = fc.drive_id(p)
    return (did in ids) or any(len(f.strip("/ ")) > len(CFG["EQ_MASTER_FILENAME"]) and p.endswith(f.strip()) for f in frags)
eq_ok = [p for p in eq_hits if lineage_ok(p)]
eq_ok = unique_by_content(eq_ok)
res["EQ_MASTER"] = eq_ok[0] if len(eq_ok) == 1 else None
note("EQ_MASTER", eq_hits, res["EQ_MASTER"], "lineage" if res["EQ_MASTER"] else "AMBIGUOUS_OR_NO_LINEAGE")
fc.put("eq_lineage_refs", refs)
if res["EQ_MASTER"] is None:
    for b in ("EQUILIBRIUM_P0_IRA", "EQUILIBRIUM_P0_ImCOF"):
        fc.not_executed(b, "AMBIGUOUS_CANONICAL_EQUILIBRIUM_SOURCE",
                        f"{len(eq_hits)} copies, {len(eq_ok)} with audit lineage; refs={len(refs)}", S)

# 3.3 P0 code / outputs / weather
if CFG["P0_CODE_PATH"]:
    res["P0_CODE"] = CFG["P0_CODE_PATH"]
else:
    res["P0_CODE"] = resolve_exact("P0_CODE", "CODE_P0_tensor_reproduction.ipynb", [ROOT])
res["P0_OUTPUTS"] = CFG["P0_OUTPUTS_DIR"]
if not res["P0_OUTPUTS"] and res["P0_CODE"]:
    near = [d for d in fc.walk_find([os.path.dirname(res["P0_CODE"])], r"P0.*(output|result)", dirs=True)]
    res["P0_OUTPUTS"] = near[0] if len(near) == 1 else None
    note("P0_OUTPUTS", near, res["P0_OUTPUTS"], "unique_near_P0" if res["P0_OUTPUTS"] else "AMBIGUOUS_OR_MISSING")
res["WEATHER"] = CFG["WEATHER_PATH"]
if not res["WEATHER"] and res["P0_CODE"]:
    src = open(res["P0_CODE"], errors="ignore").read()
    named = set(re.findall(r"[\w.-]*(?:weather|climate)[\w.-]*\.(?:csv|parquet|xlsx)", src, re.I))
    hits = [h for n in named for h in fc.walk_find([ROOT], "^" + re.escape(n) + "$")]
    hits = unique_by_content(sorted(set(hits)))
    res["WEATHER"] = hits[0] if len(hits) == 1 else None
    note("WEATHER", hits, res["WEATHER"], "referenced_by_P0_code" if res["WEATHER"] else "AMBIGUOUS_OR_MISSING")

# 3.4 process bridge
res["BRIDGE_DIR"] = CFG.get("BRIDGE_DIR_PATH") if CFG.get("BRIDGE_DIR_PATH") and os.path.isdir(CFG["BRIDGE_DIR_PATH"]) else resolve_exact("BRIDGE_DIR", "three_material_cycle_productivity_bridge", [ROOT], dirs=True)
res["BRIDGE_FINAL_DIR"] = CFG.get("BRIDGE_FINAL_DIR_PATH") if CFG.get("BRIDGE_FINAL_DIR_PATH") and os.path.isdir(CFG["BRIDGE_FINAL_DIR_PATH"]) else resolve_exact("BRIDGE_FINAL_DIR", "three_material_cycle_productivity_bridge_final", [ROOT], dirs=True)
res["BRIDGE_CODE"] = sorted(glob.glob(os.path.join(res["BRIDGE_DIR"], "**", "*.py"), recursive=True) + glob.glob(os.path.join(res["BRIDGE_DIR"], "**", "*.ipynb"), recursive=True)) if res["BRIDGE_DIR"] else []
if CFG.get("IRA_CONTINUOUS_CODE_PATH") and os.path.isfile(CFG["IRA_CONTINUOUS_CODE_PATH"]):
    res["IRA_CONTINUOUS_CODE"] = CFG["IRA_CONTINUOUS_CODE_PATH"]
    if CFG["IRA_CONTINUOUS_CODE_PATH"] not in res["BRIDGE_CODE"]: res["BRIDGE_CODE"].append(CFG["IRA_CONTINUOUS_CODE_PATH"])
for k, n in dict(IRA_ADS_S9="IRA900C_s9_adsorption_kernel.csv", IRA_REG_S9="IRA900C_s9_regeneration_kernel.csv",
                 IRA_REG_S12="IRA900C_s12_regeneration_kernel.csv").items():
    res[k] = resolve_exact(k, n, [res["BRIDGE_DIR"]] if res["BRIDGE_DIR"] else [ROOT])
for n in ["process_endpoint_climate_scores.csv", "fixed_policy_Wcyc_ordering.csv", "optimized_productivity_climate_scores.csv",
          "optimized_productivity_ordering.csv", "pairwise_endpoint_reversals.csv", "process_model_provenance.csv"]:
    res["FINAL:" + n] = resolve_exact("FINAL:" + n, n, [res["BRIDGE_FINAL_DIR"]] if res["BRIDGE_FINAL_DIR"] else [ROOT])

# 3.5 ImCOF TPD (QA-pass high-res curves are the only primary numerical source)
res["TPD_DATABASE"] = resolve_exact("TPD_DATABASE", "thermal_process_regeneration_database_2026-11-13_UPDATED", [ROOT]) or \
                      resolve_exact("TPD_DATABASE_DIR", "thermal_process_regeneration_database_2026-11-13_UPDATED", [ROOT], dirs=True)
res["TPD_PASS_CURVES"] = CFG.get("TPD_PASS_CURVES_PATH") if CFG.get("TPD_PASS_CURVES_PATH") and os.path.isfile(CFG["TPD_PASS_CURVES_PATH"]) else resolve_exact("TPD_PASS_CURVES", "thermal_process_highres_QA_pass_curves_2026-11-13.csv", [ROOT])
res["TPD_QA_MANIFEST"] = CFG.get("TPD_QA_MANIFEST_PATH") if CFG.get("TPD_QA_MANIFEST_PATH") and os.path.isfile(CFG["TPD_QA_MANIFEST_PATH"]) else resolve_exact("TPD_QA_MANIFEST", "thermal_process_highres_QA_manifest_2026-11-13.csv", [ROOT])
for k, n in dict(TPD_DIGITIZED_POINTS="thermal_process_curve_points_digitized_2026-11-13",
                 TPD_DIGITIZATION_MANIFEST="thermal_process_digitization_manifest_2026-11-13",
                 TPD_REFINED_CURVES="thermal_process_curves_refined_2026-11-13").items():
    res[k] = resolve_exact(k, n, [ROOT], required=False)     # audit only, never primary numerics

# 3.6 papers (text audit only)
res["PADAC_PDF"] = CFG["PADAC_PDF_PATH"] or (lambda h: h[0] if len(h) == 1 else None)(
    fc.walk_find([ROOT], r"(filahi|padac).*\.pdf$"))
res["IMCOF_PDFS"] = CFG["IMCOF_PDF_PATHS"] or fc.walk_find([ROOT], r"imcof.*\.(pdf|txt|md)$")
fc.put("resolved", res)
cand = pd.DataFrame(cand_rows); fc.put("resolution_candidates", cand)
fc.save_csv(cand, "source_resolution_candidates.csv")
print(pd.Series({k: (v if not isinstance(v, list) else f"{len(v)} files") for k, v in res.items()}).to_string())


In [ ]:
import sys, os, importlib
sys.path.insert(0, "/content")
import numpy as np, pandas as pd
import fidelity_core as fc, fidelity_adapters as ad
importlib.reload(fc); importlib.reload(ad)
CFG = fc.load_config()
S = "S4"
# 4. Input integrity manifest (path, Drive ID, size, mtime, SHA256, provenance, role)
res = fc.get("resolved")
prov = dict(AUDIT=("DIRECT_REPORTED_MEASURED", "canonical 9-material process/equilibrium audit baseline"),
            EQ_MASTER=("DIRECT_REPORTED_MEASURED", "canonical equilibrium dataset (lineage-resolved)"),
            P0_CODE=("CODE_NON_NUMERIC", "canonical P0 representation implementation"),
            WEATHER=(CFG["WEATHER_PROVENANCE"] or "NOT_AVAILABLE", "36-climate hourly weather (P0)"),
            IRA_ADS_S9=(CFG["IRA_KERNEL_PROVENANCE"] or "NOT_AVAILABLE", "IRA primary adsorption kernel"),
            IRA_REG_S9=(CFG["IRA_KERNEL_PROVENANCE"] or "NOT_AVAILABLE", "IRA primary regeneration kernel"),
            IRA_REG_S12=(CFG["IRA_KERNEL_PROVENANCE"] or "NOT_AVAILABLE", "IRA sensitivity regeneration kernel"),
            TPD_PASS_CURVES=("DIGITIZED_FROM_FIGURE", "ImCOF TPD QA-passed high-resolution curves (primary)"),
            TPD_QA_MANIFEST=("AUDIT_ONLY", "TPD high-res QA manifest"),
            TPD_DATABASE=("DIGITIZED_FROM_FIGURE", "thermal regeneration database (updated)"),
            TPD_DIGITIZED_POINTS=("AUDIT_ONLY", "provenance audit only"), TPD_DIGITIZATION_MANIFEST=("AUDIT_ONLY", "provenance audit only"),
            TPD_REFINED_CURVES=("AUDIT_ONLY", "superseded/refined curves — audit only"),
            PADAC_PDF=("AUDIT_ONLY", "PADAC paper (transcription audit)"))
for k, v in res.items():
    if k.startswith("FINAL:"):
        prov[k] = ("AUDIT_ONLY", "existing active-process output (regression target)")
for k in ("WEATHER", "IRA_ADS_S9"):
    if prov[k][0] == "NOT_AVAILABLE":
        fc.warn("PROVENANCE_UNSET", f"{k}: provenance class not supplied in CONFIG; recorded NOT_AVAILABLE.", S)
open(os.path.join(fc._state_dir(), "inputs.jsonl"), "w").close()   # rebuild idempotently
for k, (pv, role) in prov.items():
    p = res.get(k)
    if isinstance(p, str) and os.path.exists(p):
        fc.register_input(p, pv, role, S)
for p in res.get("BRIDGE_CODE", []):
    fc.register_input(p, "CODE_NON_NUMERIC", "existing process-bridge implementation", S)
for p in res.get("IMCOF_PDFS", []):
    fc.register_input(p, "AUDIT_ONLY", "ImCOF paper/SI (thermal-property audit)", S)
man = pd.DataFrame(fc._jsonl_read("inputs.jsonl"))
fc.save_csv(man, "input_manifest.csv")
dups = man[man.duplicated("filename", keep=False)] if len(man) else man
if len(dups):
    fc.warn("DUPLICATE_FILENAMES", f"{len(dups)} manifest rows share filenames (see input_manifest.csv)", S)
man[["filename", "provenance", "role", "size_bytes", "sha256"]]


In [ ]:
import sys, os, importlib
sys.path.insert(0, "/content")
import numpy as np, pandas as pd
import fidelity_core as fc, fidelity_adapters as ad
importlib.reload(fc); importlib.reload(ad)
CFG = fc.load_config()
S = "S5"
# 5. Equilibrium representation reproduction — the canonical P0 implementation, no new family.
try:
    p0 = ad.load_p0()
    lst = fc.list_defs(p0); fc.save_csv(lst, "p0_code_definitions.csv")
    print("skipped P0 cells:", p0["__skipped_cells__"][:10]); display(lst)
except fc.HardStop as e:
    for b in ("EQUILIBRIUM_P0_IRA", "EQUILIBRIUM_P0_ImCOF"):
        fc.not_executed(b, str(e).split(":")[0], str(e), S)
    p0 = None
meta = {}
for key, mat in CFG["MATERIALS"].items():
    branch = f"EQUILIBRIUM_P0_{key}"
    if p0 is None or fc.status_of(branch).startswith("NOT_EXECUTED"):
        continue
    def _go():
        reps = ad.build_representations(mat, CFG)
        return {k: dict(level=v.get("level"), domain=v.get("domain"), rh_levels=v.get("rh_levels"),
                        t_levels=v.get("t_levels")) for k, v in reps.items()}
    m = fc.guarded(branch, S, _go)
    if m is not None:
        meta[key] = m
        fc.set_status(branch, "EXECUTED_REPRESENTATIONS_BUILT", f"{len(m)} representations: {list(m)}", S)
fc.put("rep_meta", meta)


In [ ]:
import sys, os, importlib
sys.path.insert(0, "/content")
import numpy as np, pandas as pd
import fidelity_core as fc, fidelity_adapters as ad
importlib.reload(fc); importlib.reload(ad)
CFG = fc.load_config()
S = "S6"
# 6. P0 regression gate and existing-H baseline freeze (+ ImCOF reported-anchor QA, informative only)
rows = []
for key, mat in CFG["MATERIALS"].items():
    branch = f"EQUILIBRIUM_P0_{key}"
    if not fc.status_of(branch).startswith("EXECUTED"):
        print(branch, fc.status_of(branch)); continue
    def _go():
        reps = ad.build_representations(mat, CFG)
        t = ad.p0_regression_targets(mat, reps, CFG)
        need = {"rep", "key", "reproduced", "reference"}
        if not need <= set(t.columns):
            raise fc.HardStop(f"P0_REGRESSION_FORMAT: need columns {need}, got {list(t.columns)}")
        t = t.copy(); t["material"] = mat
        t["abs_err"] = (t.reproduced - t.reference).abs()
        t["pass"] = np.isclose(t.reproduced, t.reference, rtol=CFG["P0_REGRESSION_RTOL"], atol=CFG["P0_REGRESSION_ATOL"])
        t["check"] = "P0_regression"
        if key == "ImCOF":   # reported anchors (QA/reference only)
            q = reps[CFG["FULL_REPRESENTATION_NAME"]]["q"]
            A = CFG["IMCOF_ANCHORS"]
            for rh, (v, sd) in A["T25"].items():
                r = float(rh[2:]); qq = float(np.asarray(q(np.array([25.0]), np.array([r])))[0])
                t.loc[len(t)] = dict(rep=CFG["FULL_REPRESENTATION_NAME"], key=f"anchor_25C_RH{r:g}", reproduced=qq, reference=v, material=mat,
                                     abs_err=abs(qq - v), check="reported_anchor_QA", **{"pass": abs(qq - v) <= 2 * sd})
            for tk, v in A["RH70"].items():
                T = float(tk[1:].replace("m", "-")); qq = float(np.asarray(q(np.array([T]), np.array([70.0])))[0])
                t.loc[len(t)] = dict(rep=CFG["FULL_REPRESENTATION_NAME"], key=f"anchor_RH70_T{T:g}", reproduced=qq, reference=v, material=mat,
                                     abs_err=abs(qq - v), check="reported_anchor_QA", **{"pass": np.nan})
        return t
    t = fc.guarded(branch, S, _go)
    if t is None:
        continue
    rows.append(t)
    g = t[t.check == "P0_regression"]
    if len(g) and g["pass"].all():
        fc.set_status(branch, "PASSED_P0_REGRESSION", f"{len(g)} targets within tolerance", S)
    else:
        fc.not_executed(branch, "P0_REPRESENTATION_REGRESSION_FAILURE", f"{(~g['pass']).sum()} of {len(g)} targets failed", S)
    bad = t[(t.check == "reported_anchor_QA") & (t["pass"] == False)]
    if len(bad):
        fc.warn("IMCOF_ANCHOR_DEVIATION", f"{len(bad)} 25 °C anchors outside ±2σ: {list(bad.key)}", S)
reg = pd.concat(rows) if rows else pd.DataFrame(columns=["material", "rep", "key", "reproduced", "reference", "pass", "check"])
fc.save_csv(reg, "equilibrium_representation_regression.csv")
# Passed P0 regression reference H values are the frozen equilibrium-side baseline used later by the H-Y bridge.
fc.put("H_baseline_regression", reg[reg.get("check")=="P0_regression"].copy())


# PART A — IRA-900-C process pair


In [ ]:
import sys, os, importlib
sys.path.insert(0, "/content")
import numpy as np, pandas as pd
import fidelity_core as fc, fidelity_adapters as ad
importlib.reload(fc); importlib.reload(ad)
CFG = fc.load_config(); S = "S7"; branch = "ACTIVE_BRIDGE_REGRESSION"
# 7. Existing active-process bridge audit.
# The canonical final bridge is a notebook pipeline, not a callable score function. Therefore v1.2 does NOT pretend to
# re-run it through a nonexistent API. It verifies the saved climate-level P* schema/key structure and records
# STRUCTURAL_REPRODUCTION_ONLY. Numerical bridge reproduction remains a separate artifact requirement.
def _go():
    if not fc.is_ok("EQUILIBRIUM_P0_IRA"):
        raise fc.HardStop("UPSTREAM_P0_IRA: " + fc.status_of("EQUILIBRIUM_P0_IRA"))
    d,p = ad.load_active_optimized_scores(CFG)
    if set(d.representation) != {"single","tensor"}:
        raise fc.HardStop(f"ACTIVE_OUTPUT_SCHEMA: representations={sorted(d.representation.unique())}")
    k=d[["branch","temporal_mode","climate","representation"]]
    if k.duplicated().any(): raise fc.HardStop("ACTIVE_OUTPUT_KEY_DUPLICATE: duplicate branch/mode/climate/representation")
    counts=(d.groupby(["branch","temporal_mode","representation"]).climate.nunique().reset_index(name="n_climates"))
    fc.save_csv(counts,"active_bridge_structural_audit.csv")
    if (counts.n_climates < 2).any(): raise fc.HardStop("ACTIVE_OUTPUT_INCOMPLETE: fewer than 2 climates in a variant")
    fc.register_input(p,"AUDIT_ONLY","canonical active-process optimized-productivity output",S)
    return len(d), os.path.basename(p)
r=fc.guarded(branch,S,_go)
if r:
    fc.set_status(branch,"STRUCTURAL_REPRODUCTION_ONLY",f"{r[0]} long rows from {r[1]}; saved canonical bridge output used without claiming numerical re-execution",S)


In [ ]:
import sys, os, importlib
sys.path.insert(0, "/content")
import numpy as np, pandas as pd
import fidelity_core as fc, fidelity_adapters as ad
importlib.reload(fc); importlib.reload(ad)
CFG = fc.load_config()
S = "S8"
# 8. IRA kinetic-data QA (s9 primary; s12 regeneration sensitivity)
import re
res = fc.get("resolved"); unit = dict(s=1/3600, min=1/60, h=1.0)
rows, K = [], {}
for lab, key in dict(ads="IRA_ADS_S9", s9="IRA_REG_S9", s12="IRA_REG_S12").items():
    p = res.get(key)
    if not p:
        rows.append(dict(kernel=lab, status="MISSING")); continue
    df = pd.read_csv(p)
    tc = CFG["IRA_KERNEL_COLUMNS"]["time"] or fc.detect_col(df, [r"^t(ime)?(_|$|\s)", r"time"], f"{lab} time column")
    fcol = CFG["IRA_KERNEL_COLUMNS"]["fraction"] or fc.detect_col(df, [r"frac", r"^F", r"norm", r"approach", r"conv"], f"{lab} fraction column")
    u = CFG["IRA_KERNEL_TIME_UNIT"] or next((k for k in ("min", "h", "s") if re.search(rf"[\[_(]{k}[\])]?$", tc)), None)
    if u is None:
        raise fc.HardStop(f"UNIT_INCOMPATIBILITY: time unit of {lab} kernel column '{tc}' not determinable; set IRA_KERNEL_TIME_UNIT.")
    t = df[tc].to_numpy(float) * unit[u]; F = df[fcol].to_numpy(float)
    flipped = bool(F[0] > F[-1])
    if flipped:   # remaining-fraction form -> approach fraction (definitional transform, recorded)
        F = 1 - F
        fc.warn("KERNEL_ORIENTATION", f"{lab}: decreasing kernel interpreted as remaining fraction; using 1-F.", S)
    t2, F2, ncorr = fc.monotone_fraction(t, F)
    K[lab] = (t2 - t2[0], F2)
    rows.append(dict(kernel=lab, path=p, time_col=tc, fraction_col=fcol, time_unit=u, n=len(t), t_end_h=t2[-1] - t2[0],
                     F_start=F2[0], F_end=F2[-1], n_monotone_corrections=ncorr, flipped=flipped,
                     t50_h=float(np.interp(0.5, F2, t2 - t2[0])) if F2[-1] >= 0.5 else np.nan,
                     t90_h=float(np.interp(0.9, F2, t2 - t2[0])) if F2[-1] >= 0.9 else np.nan, status="LOADED"))
    if F2[-1] < 0.9:
        fc.warn("KERNEL_INCOMPLETE", f"{lab}: kernel ends at F={F2[-1]:.3f} (<0.9); beyond-end held at last value.", S)
qa = pd.DataFrame(rows); fc.save_csv(qa, "IRA_kinetic_kernel_QA.csv"); display(qa)
fc.put("ira_kernels", K)
if not {"ads", "s9"} <= set(K):
    for b in ("IRA_ENV", "IRA_ACTIVE"):
        fc.not_executed(b, "MISSING_IRA_PRIMARY_KINETIC_KERNEL", "s9 adsorption/regeneration kernel missing", S)
else:
    fc.set_status("IRA_KERNELS", "EXECUTED_QA", "", S)


In [ ]:
import sys, os, importlib
sys.path.insert(0, "/content")
import numpy as np, pandas as pd
import fidelity_core as fc, fidelity_adapters as ad
importlib.reload(fc); importlib.reload(ad)
CFG = fc.load_config(); S = "S9"; branch = "IRA_ENV"
# 9. IRA continuous-weather environmental response using the ACTUAL v7 state propagator.
# No RH threshold/state controller. Unsupported P0 states are handled by v7 gap_mode='hold'.
def _go():
    if not fc.is_ok("EQUILIBRIUM_P0_IRA"):
        raise fc.HardStop(f"UPSTREAM_EQUILIBRIUM_P0_IRA: {fc.status_of('EQUILIBRIUM_P0_IRA')}")
    prop,pname=ad.ira_continuous_propagator()
    w,order,_,_=ad.load_climate_weather(CFG)
    if len(order)!=CFG["N_CLIMATES_EXPECTED"]:
        raise fc.HardStop(f"CLIMATE_ENSEMBLE_MISMATCH: {len(order)} climates found")
    reps=ad.build_representations(CFG["MATERIALS"]["IRA"],CFG)
    out=[]
    for br in ("lower","upper"):
        for c in order:
            wc=w[w.climate==c].sort_values("timestamp").reset_index(drop=True)
            dt=np.diff(wc.timestamp.values).astype("timedelta64[s]").astype(float)/3600
            if len(dt) and not np.allclose(dt,1.0):
                raise fc.HardStop(f"CLIMATE_TIME_MERGE_MISMATCH: {c} not uniformly hourly")
            T,RH=wc.T_C.to_numpy(float),wc.RH_pct.to_numpy(float)
            for rep,R in reps.items():
                qs=np.asarray(R["q_branch"](br,T,RH),float)
                z=pd.DataFrame({"qstar":qs})
                q=np.asarray(prop(z,"qstar",tau_h=1.0,gap_mode="hold"),float)
                dq=np.diff(q); Y=float(np.nansum(np.maximum(-dq,0.0)))
                out.append(dict(material=CFG["MATERIALS"]["IRA"],controller="environmental_moisture",
                                variant=f"continuous_v7_{br}_subsample_1h",branch=br,temporal_mode="subsample_1h",
                                representation=rep,level=R.get("level"),climate=c,
                                supported_state_fraction=float(np.isfinite(qs).mean()),Y=Y,
                                endpoint="cumulative_finite_rate_loading_release_mmol_g",propagator=pname))
    df=pd.DataFrame(out); fc.save_csv(df,"IRA_environmental_loading_climate_scores.csv"); fc.put("ira_env_scores",df)
    fc.record_assumption("IRA_environmental_controller","continuous hourly RH/T; v7 propagate_city(tau_h=1,gap_mode=hold); no RH switching thresholds",
                         "CALCULATED_THIS_NOTEBOOK","Exact controller logic from canonical IRA continuous-weather v7; branch support follows P0.",S,branch)
    return len(df)
n=fc.guarded(branch,S,_go)
if n: fc.set_status(branch,"EXECUTED",f"{n} branch-climate-representation rows using canonical v7 propagate_city",S)


In [ ]:
import sys, os, importlib
sys.path.insert(0, "/content")
import numpy as np, pandas as pd
import fidelity_core as fc, fidelity_adapters as ad
importlib.reload(fc); importlib.reload(ad)
CFG = fc.load_config(); S = "S10"; branch = "IRA_ACTIVE"
# 10. IRA externally imposed controller: use the EXISTING bridge's climate-level optimized-productivity output.
# This avoids fabricating a callable API that the bridge notebook does not expose.
def _go():
    for up in ("EQUILIBRIUM_P0_IRA","ACTIVE_BRIDGE_REGRESSION"):
        if not fc.is_ok(up): raise fc.HardStop(f"UPSTREAM_{up}: {fc.status_of(up)}")
    df,p=ad.load_active_optimized_scores(CFG)
    df["material"]=CFG["MATERIALS"]["IRA"]; df["controller"]="active_moisture"
    df["level"]=df.representation.map({"tensor":0,"single":1})
    df["variant"]=df.branch.astype(str)+"|"+df.temporal_mode.astype(str)
    fc.save_csv(df,"IRA_active_cycle_climate_scores.csv"); fc.put("ira_active_scores",df)
    return len(df),p
r=fc.guarded(branch,S,_go)
if r: fc.set_status(branch,"EXECUTED",f"{r[0]} rows from canonical optimized_productivity_climate_scores.csv",S)


# PART B — ImCOF-TAEA sorbent-side process inputs


In [ ]:
import sys, os, importlib
sys.path.insert(0, "/content")
import numpy as np, pandas as pd
import fidelity_core as fc, fidelity_adapters as ad
importlib.reload(fc); importlib.reload(ad)
CFG = fc.load_config()
S = "S11"
# 11. ImCOF thermal-property audit. Automatic text hits are CANDIDATES for the user to read; a property is
# DIRECT/DERIVABLE only when entered in CONFIG['IMCOF_THERMAL_PROPS'] with its source. No generic COF values.
import re
res = fc.get("resolved")
kw = dict(C_s=r"heat capacity|C_?p\b|specific heat", rho_bulk=r"bulk density|packing density|tap density",
          rho_p=r"particle density|skeletal density|crystal density", eps_bed=r"porosity|void fraction",
          k_s=r"thermal conductivity", alpha_s=r"thermal diffusivity", d_p=r"particle size|pellet|diameter|mesh|sieve",
          dH_ads=r"isosteric|heat of adsorption|adsorption enthalpy|ΔH|Qst")
texts = []
for p in res.get("IMCOF_PDFS", []):
    try:
        if p.endswith(".pdf"):
            try:
                import pypdf
            except ImportError:
                import subprocess; subprocess.run([sys.executable, "-m", "pip", "-q", "install", "pypdf"]); import pypdf
            texts.append((p, " ".join((pg.extract_text() or "") for pg in pypdf.PdfReader(p).pages)))
        else:
            texts.append((p, open(p, errors="ignore").read()))
    except Exception as e:
        fc.warn("IMCOF_TEXT", f"{p}: {e}", S)
for key in ("AUDIT", "TPD_DATABASE"):
    p = res.get(key)
    if p and os.path.isfile(p) and p.endswith(".xlsx"):
        for sh, df in pd.read_excel(p, sheet_name=None, header=None).items():
            txt = " | ".join(df.astype(str).values.ravel())
            if re.search("imcof", txt, re.I):
                texts.append((f"{p}::{sh}", txt))
props = CFG["IMCOF_THERMAL_PROPS"]; rows = []
need_B = {"C_s", "rho_bulk", "eps_bed", "d_p", "k_s"}
for k, rx in kw.items():
    snippets = [f"{os.path.basename(src)}: …{t[max(0, m.start() - 80):m.end() + 120]}…".replace("\n", " ")
                for src, t in texts for m in list(re.finditer(rx, t, re.I))[:3]]
    e = props.get(k)
    cls = e["cls"] if e else "NOT_REPORTED"
    rows.append(dict(property=k, classification=cls, value=e["value"] if e else None, unit=e.get("unit") if e else None,
                     source=e.get("source") if e else None, required_for="Formulation A+B" if k in need_B else ("Formulation A only" if k == "dH_ads" else "derivation"),
                     resolution="RESOLVED" if e else ("UNRESOLVED_SENSITIVITY_PARAMETER" if k in need_B | {"dH_ads"} else "NOT_REQUIRED"),
                     candidate_snippets_for_manual_review=" || ".join(snippets)[:3000]))
for lab, val in CFG["IMCOF_TPD_ED"].items():
    rows.append(dict(property=f"E_d_TPD_{lab}", classification="DIRECT_IMCOF", value=val, unit="kJ/mol", source="TPD heating-rate analysis",
                     required_for="none (descriptor)", resolution="NOT_INTERCHANGEABLE_WITH_EQUILIBRIUM_dH"))
aud = pd.DataFrame(rows); fc.save_csv(aud, "ImCOF_thermal_property_audit.csv")
# --- guards: 13X inheritance and E_d substitution ---
P = CFG["PADAC"]; V_bed = np.pi * P["D_int"][0] ** 2 / 4 * P["L"][0]
x13 = dict(C_s=P["C_s_13X"][0], rho_bulk=P["m_s_13X"][0] / V_bed, eps_bed=P["eps_bed_13X"][0], d_p=P["d_p_13X"][0],
           k_s=P["alpha_s_13X"][0] * P["rho_p_13X"][0] * P["C_s_13X"][0], h_gs=P["h_gs_13X"][0], dH_ads=P["minus_dH_13X"][0])
def check(src_name, d, explicit_ok):
    for k, v in d.items():
        if k in x13 and v is not None and np.isclose(float(v), x13[k], rtol=1e-3) and not explicit_ok:
            raise fc.HardStop(f"THIRTEENX_PROPERTY_SILENTLY_INHERITED: {src_name}.{k}={v} equals the 13X value")
    dh = d.get("dH_ads")
    if dh is not None and any(np.isclose(abs(float(dh)), e * s, rtol=1e-3) for e in CFG["IMCOF_TPD_ED"].values() for s in (1, 1e3)):
        raise fc.HardStop("TPD_ED_SUBSTITUTED_FOR_EQUILIBRIUM_DH: dH_ads equals a TPD desorption energy")
def _guard():
    check("IMCOF_THERMAL_PROPS", {k: (e["value"] if e else None) for k, e in props.items()}, False)
    thetas = CFG["IMCOF_THERMAL_THETA"] or []
    for th in thetas:
        check(th.get("id", "theta"), th, th.get("explicit_13X_sensitivity") and th.get("provenance") == "ASSUMED_SENSITIVITY_PARAMETER")
    unresolved = sorted(k for k in need_B if not props.get(k))
    if unresolved and not thetas and CFG.get("IMCOF_ENV_THERMAL_MODE") == "padac_13X_controller":
        fc.put("imcof_theta", [])
        fc.not_executed("IMCOF_THERMAL_PROPS", "NOT_REQUIRED_FOR_PRIMARY_PADAC13X_CONTROLLER",
                        f"unresolved ImCOF-specific properties {unresolved}; primary environmental branch uses exogenous PADAC+13X T_s(t), not an ImCOF bed-temperature prediction", S)
        return 0
    if unresolved and not thetas:
        raise fc.HardStop(f"UNRESOLVED_IMCOF_THERMAL_PROPERTY: {unresolved} not reported and no IMCOF_THERMAL_THETA declared")
    for th in thetas:
        miss = [k for k in unresolved if th.get(k) is None]
        if miss:
            raise fc.HardStop(f"UNRESOLVED_IMCOF_THERMAL_PROPERTY: theta {th.get('id')} lacks {miss}")
        fc.record_assumption(f"ImCOF_theta:{th.get('id')}", {k: th[k] for k in unresolved}, th.get("provenance", "ASSUMED_SENSITIVITY_PARAMETER"),
                             th.get("rationale", ""), S, "PADAC_IMCOF")
    real = [dict(id=th.get("id"), **{k: (props[k]["value"] if props.get(k) else th.get(k)) for k in need_B | {"h_gs"}}) for th in thetas] \
           or [dict(id="direct", **{k: (props[k]["value"] if props.get(k) else None) for k in need_B | {"h_gs"}})]
    fc.put("imcof_theta", real)
    return len(real)
n = fc.guarded("IMCOF_THERMAL_PROPS", S, _guard)
if n:
    fc.set_status("IMCOF_THERMAL_PROPS", "EXECUTED", f"{n} thermal-property realisation(s)", S)
fc.set_status("IMCOF_FORMULATION_A_DH", "EXECUTED" if props.get("dH_ads") else "NOT_EXECUTED_IMCOF_EQUILIBRIUM_DH_UNAVAILABLE",
              "" if props.get("dH_ads") else "E_d not used as ΔH; Formulation A is model-form uncertainty", S)
aud[["property", "classification", "value", "resolution"]]


In [ ]:
import sys, os, importlib
sys.path.insert(0, "/content")
import numpy as np, pandas as pd
import fidelity_core as fc, fidelity_adapters as ad
importlib.reload(fc); importlib.reload(ad)
CFG=fc.load_config(); S="S12"; branch="IMCOF_TPD"
# 12. ImCOF TPD loading and QA — verified high-resolution file schema.
# Actual key is (figure,panel,curve), not a pre-existing curve_id. The high-res file also contains Mg-Al S7;
# this branch explicitly filters material == ImCOF-TAEA before building kernels.
import re
def _go():
    res=fc.get("resolved")
    if not (res.get("TPD_PASS_CURVES") and res.get("TPD_QA_MANIFEST")):
        raise fc.HardStop("TPD_INPUT_MISSING: high-res QA pass curves or manifest not resolved")
    cur=pd.read_csv(res["TPD_PASS_CURVES"]); man=pd.read_csv(res["TPD_QA_MANIFEST"])
    need_cur={'material','figure','panel','curve','x_value','x_unit','y_value','y_unit','qa_status'}
    need_man={'figure','panel','curve','qa_status'}
    if not need_cur<=set(cur.columns): raise fc.HardStop(f"TPD_SCHEMA: curves missing {sorted(need_cur-set(cur.columns))}")
    if not need_man<=set(man.columns): raise fc.HardStop(f"TPD_SCHEMA: manifest missing {sorted(need_man-set(man.columns))}")
    cur=cur[cur.material.astype(str).eq(CFG["MATERIALS"]["ImCOF"])].copy()
    if cur.empty: raise fc.HardStop("TPD_IMCOF_MISSING: no ImCOF-TAEA rows in high-res pass file")
    cur=cur[cur.qa_status.astype(str).str.contains('PASS',case=False,na=False)].copy()
    if not cur.x_unit.astype(str).str.lower().str.contains('degc').all(): raise fc.HardStop("UNIT_INCOMPATIBILITY: ImCOF high-res x must be degC")
    # Current verified ImCOF high-res primary set is S20: curve label '<Tads>C,<RH>%RH'.
    cur['curve_id']=cur.figure.astype(str)+'|'+cur.panel.astype(str)+'|'+cur.curve.astype(str)
    parsed=cur['curve'].astype(str).str.extract(r'^\s*(-?\d+(?:\.\d+)?)C\s*,\s*(\d+(?:\.\d+)?)%RH\s*$')
    if parsed.isna().any().any():
        bad=sorted(cur.loc[parsed.isna().any(axis=1),'curve'].astype(str).unique())
        raise fc.HardStop(f"AMBIGUOUS_TPD_CURVE_IDENTITY: unparseable ImCOF curve labels {bad}")
    cur['T_ads_C']=pd.to_numeric(parsed[0],errors='raise'); cur['RH_ads_pct']=pd.to_numeric(parsed[1],errors='raise')
    cur['T_C']=pd.to_numeric(cur.x_value,errors='raise'); cur['signal']=pd.to_numeric(cur.y_value,errors='raise')
    cur['heating_rate']=float(CFG['TPD_PRIMARY_HEATING_RATE'])
    # Source protocol: 45 min N2 purge, then 0.5 C/min ramp to 80 C for S19-S21. Construct ramp time from T.
    cur['time_min']=cur.groupby('curve_id')['T_C'].transform(lambda x:(x-x.min())/float(CFG['TPD_PRIMARY_HEATING_RATE']))
    cur['adsorbed_amount']=np.nan
    cur['signal_unit']=cur['y_unit'].astype(str)
    d=cur[['curve_id','figure','panel','curve','T_C','time_min','signal','signal_unit','T_ads_C','RH_ads_pct','heating_rate','adsorbed_amount']].copy()
    # Manifest uses the same composite key. 'supersedes' describes OLD low-resolution data replaced by this current curve;
    # it must NOT be interpreted as saying the current high-res curve is superseded.
    man=man.copy(); man['curve_id']=man.figure.astype(str)+'|'+man.panel.astype(str)+'|'+man.curve.astype(str)
    mm=man.set_index('curve_id',drop=False)
    ids=set(d.curve_id.astype(str)); miss=ids-set(mm.index.astype(str))
    if miss: raise fc.HardStop(f"AMBIGUOUS_TPD_CURVE_IDENTITY: current curves absent from manifest {sorted(miss)}")
    bad=[i for i in ids if not re.search('pass',str(mm.loc[i,'qa_status']),re.I)]
    if bad: raise fc.HardStop(f"TPD_QA_FAILURE: non-pass manifest rows {bad}")
    qa=(d.groupby('curve_id',as_index=False).agg(T_ads_C=('T_ads_C','first'),RH_ads_pct=('RH_ads_pct','first'),
          heating_rate=('heating_rate','first'),n_points=('T_C','size'),T_min_C=('T_C','min'),T_max_C=('T_C','max'),
          signal_unit=('signal_unit','first')))
    qa['primary_heating_rate']=np.isclose(qa.heating_rate,CFG['TPD_PRIMARY_HEATING_RATE'],atol=CFG['TPD_HEATING_RATE_TOL'])
    qa['provenance']='DIGITIZED_FROM_FIGURE'
    qa['protocol_metadata_provenance']='DIRECT: SI Sec 4.4 / Figs S19-S21; 45 min N2 purge; 0.5 C/min ramp'
    qa['supersedes_note']=qa.curve_id.map(mm['supersedes']) if 'supersedes' in mm.columns else ''
    fc.put('tpd_long',d); fc.put('tpd_qa',qa)
    fc.save_csv(qa,'ImCOF_TPD_input_schema_QA.csv')
    return len(qa)
n=fc.guarded(branch,S,_go)
if n:
    fc.set_status(branch,'EXECUTED_QA',f'{n} QA-passed ImCOF S20 curves; composite keys + protocol metadata reconstructed from verified source',S)
    display(fc.get('tpd_qa'))


In [ ]:
import sys, os, importlib
sys.path.insert(0, "/content")
import numpy as np, pandas as pd
import fidelity_core as fc, fidelity_adapters as ad
importlib.reload(fc); importlib.reload(ad)
CFG = fc.load_config()
S = "S13"
# 13. Construct F_R(T | T_ads, RH_ads) per measured condition (no universal collapse unless shown adequate).
# Processing chain (documented): sort by time (or T) -> keep ramp portion (T non-decreasing up to its maximum) ->
# baseline: linear between first and last ramp points (rate/concentration signals only) -> T->t via heating rate
# (t = (T-T0)/beta) when no time column -> trapezoidal integration -> endpoint = end of available data ->
# normalisation: adsorbed amount if the signal is a cumulative amount AND adsorbed amount is given, else the
# integral to endpoint (then F_R(endpoint)=1 by construction: CO2 not released by the endpoint is not represented) ->
# monotone clip to [0,1] -> linear interpolation in T.
import matplotlib.pyplot as plt
branch = "IMCOF_KERNEL"
def _go():
    if not fc.is_ok("IMCOF_TPD"):
        raise fc.HardStop("UPSTREAM_IMCOF_TPD: " + fc.status_of("IMCOF_TPD"))
    d = fc.get("tpd_long"); qa = fc.get("tpd_qa").set_index("curve_id"); kind = CFG["TPD_SIGNAL_KIND"]
    if kind not in ("cumulative_fraction", "cumulative_amount", "rate", "concentration"):
        raise fc.HardStop("UNIT_INCOMPATIBILITY: TPD_SIGNAL_KIND must be concentration for the verified ImCOF S20 high-res curves")
    kern, rows, long = {}, [], []
    for cid, g in d.groupby("curve_id"):
        g = g.sort_values("time_min" if "time_min" in g and g.time_min.notna().all() else "T_C")
        T = g.T_C.to_numpy(float); y = g.signal.to_numpy(float)
        imax = int(np.argmax(T)); hold = g.iloc[imax:]
        Tr, yr = T[:imax + 1], y[:imax + 1]
        if kind in ("rate", "concentration"):
            base = np.interp(Tr, [Tr[0], Tr[-1]], [yr[0], yr[-1]]) if CFG["TPD_BASELINE"] == "linear_endpoints" else 0 * yr
            tt = (g.time_min.to_numpy(float)[:imax + 1] if "time_min" in g and g.time_min.notna().all()
                  else (Tr - Tr[0]) / qa.loc[cid, "heating_rate"])
            cum = np.concatenate([[0], np.cumsum(0.5 * (np.clip(yr - base, 0, None)[1:] + np.clip(yr - base, 0, None)[:-1]) * np.diff(tt))])
            # hold portion (if present): integrate too, so F at end of hold is available for the active branch
            if len(hold) > 1 and "time_min" in g:
                yh = np.clip(hold.signal.to_numpy(float) - base[-1], 0, None); th = hold.time_min.to_numpy(float)
                cum_end = cum[-1] + float(np.sum(0.5 * (yh[1:] + yh[:-1]) * np.diff(th)))
                hold_min = th[-1] - th[0]
            else:
                cum_end, hold_min = cum[-1], 0.0
            norm = cum_end
        else:
            cum = yr; cum_end = y[-1]; hold_min = (hold.time_min.iloc[-1] - hold.time_min.iloc[0]) if "time_min" in g and len(hold) > 1 else 0.0
            if kind == "cumulative_amount":
                ads_amt = g.adsorbed_amount.iloc[0] if "adsorbed_amount" in g else np.nan
                norm = ads_amt if (CFG["TPD_NORMALIZATION"] == "adsorbed_amount_if_available" and np.isfinite(ads_amt)) else cum_end
            else:
                norm = 1.0
        Tm, Fm, ncorr = fc.monotone_fraction(Tr, cum / norm)
        F_end = float(np.clip(cum_end / norm, 0, 1))
        kern[cid] = dict(T=Tm, F=Fm, F_end_after_hold=F_end, hold_min=hold_min, **qa.loc[cid, ["T_ads_C", "RH_ads_pct", "heating_rate"]].to_dict())
        at = lambda x: float(np.interp(x, Tm, Fm, left=0, right=Fm[-1]))
        T_at = lambda f: float(np.interp(f, Fm, Tm)) if Fm[-1] >= f else np.nan
        cold = qa.loc[cid, "T_ads_C"] <= 15 and qa.loc[cid, "RH_ads_pct"] > 0
        rows.append(dict(curve_id=cid, **qa.loc[cid].to_dict(), n_monotone_corrections=ncorr, F_R_45=at(45), F_R_55=at(55), F_R_65=at(65),
                         F_R_80_ramp=at(80), F_R_end_after_hold=F_end, hold_min=hold_min, T10=T_at(.1), T50=T_at(.5), T90=T_at(.9),
                         anchor="cold_humid: majority released 45-55 °C" if cold else ("25C: >90% within 45-65 °C" if qa.loc[cid, "T_ads_C"] == 25 else "none"),
                         anchor_pass=(45 <= T_at(.5) <= 55) if cold else ((at(65) >= 0.9) if qa.loc[cid, "T_ads_C"] == 25 else np.nan),
                         normalisation="adsorbed_amount" if (kind == "cumulative_amount" and norm != cum_end) else "integral_to_endpoint"))
        long.append(pd.DataFrame(dict(curve_id=cid, T_C=Tm, F_R=Fm, T_ads_C=qa.loc[cid, "T_ads_C"], RH_ads_pct=qa.loc[cid, "RH_ads_pct"],
                                      heating_rate=qa.loc[cid, "heating_rate"], provenance="CALCULATED_THIS_NOTEBOOK(from DIGITIZED_FROM_FIGURE)")))
    Q = pd.DataFrame(rows)
    if (Q.normalisation == "integral_to_endpoint").any():
        fc.warn("TPD_NORMALISATION", "F_R is an endpoint-normalised release-shape fraction because S20 provides concentration-vs-T, not absolute residual loading. F_R(80 C)=1 by definition; do not interpret this alone as measured q_lean=0.", S)
    if (Q.anchor_pass == False).any():
        fc.warn("TPD_ANCHOR_FAIL", f"QA anchors not met: {Q[Q.anchor_pass == False].curve_id.tolist()}", S)
    # universal-collapse adequacy (reported, never imposed)
    prim = Q[Q.primary_heating_rate].curve_id.tolist(); Tg = np.arange(0, 81, 1.0)
    Fs = np.array([np.interp(Tg, kern[c]["T"], kern[c]["F"], left=0, right=kern[c]["F"][-1]) for c in prim]) if prim else np.zeros((0, Tg.size))
    Q["max_abs_dev_from_pooled"] = Q.curve_id.map({c: float(np.max(np.abs(Fs[i] - Fs.mean(0)))) for i, c in enumerate(prim)}) if prim else np.nan
    fc.save_csv(Q, "ImCOF_TPD_QA.csv"); fc.save_csv(pd.concat(long), "ImCOF_TPD_kernel.csv")
    fc.put("tpd_kernel", kern); fc.put("tpd_primary_meta", Q[Q.primary_heating_rate][["curve_id", "T_ads_C", "RH_ads_pct"]].reset_index(drop=True))
    if prim:
        kern_pooled = dict(T=Tg, F=Fs.mean(0), F_end_after_hold=float(np.mean([kern[c]["F_end_after_hold"] for c in prim])))
        fc.put("tpd_kernel_pooled", kern_pooled)
    fig, ax = plt.subplots(figsize=(6, 4))
    for c in prim:
        k = kern[c]; ax.plot(k["T"], k["F"], label=f"T_ads={k['T_ads_C']:g} °C, RH={k['RH_ads_pct']:g}%")
    ax.set_xlabel("T (°C)"); ax.set_ylabel("F_R"); ax.set_title(f"ImCOF-TAEA F_R (digitised TPD, β={CFG['TPD_PRIMARY_HEATING_RATE']} °C/min)")
    ax.legend(fontsize=7); fig.savefig(os.path.join(fc.run_dir(), "figures", "fig2_ImCOF_FR.png"), dpi=150, bbox_inches="tight")
    return len(prim)
n = fc.guarded(branch, S, _go)
if n:
    fc.set_status(branch, "EXECUTED", f"{n} primary-β condition kernels", S)
elif n == 0:
    fc.not_executed(branch, "NO_PRIMARY_HEATING_RATE_CURVES", f"no curve at β={CFG['TPD_PRIMARY_HEATING_RATE']}", S)


# PART C — PADAC architecture-level thermal controller


In [ ]:
import sys, os, importlib
sys.path.insert(0, "/content")
import numpy as np, pandas as pd
import fidelity_core as fc, fidelity_adapters as ad
importlib.reload(fc); importlib.reload(ad)
CFG = fc.load_config()
S = "S14"
# 14. PADAC equation transcription audit. Nothing here is reconstructed from memory: forms that must be
# checked against the PDF are CONFIG strings, and their verification flags are reported verbatim.
res = fc.get("resolved"); V = CFG["PADAC_PDF_VERIFIED"]; Fm = CFG["PADAC_FORM"]
eqs = [
 ("eq6", "glass energy balance", "rho_v C_v dTv/dt = a_v alpha_v G + a_v h_ray,w-v (Tw-Tv)[physical sign] - h_wind a_v (Tv-Tamb) - h_rad,v-sky a_v (Tv-Tsky); a_v=" + Fm["a_v"],
  "Spec prints +a_v h_ray (Tv-Tw); implemented sign selectable (eq6_glass_wall_sign) — verify against PDF"),
 ("eq7", "Swinbank sky temperature", "Tsky = c * Tamb^1.5, Tamb and Tsky in K; c=0.0522 as printed; 0.0552 (Swinbank 1963) as sensitivity",
  "Correlation requires absolute temperature (K); Celsius never used"),
 ("eq8", "glass-sky radiation", "h = eps_v sigma (Tv^2+Tsky^2)(Tv+Tsky)", ""),
 ("eq9", "glass-wall radiation", "h = sigma (Tv^2+Tw^2)(Tv+Tw)/(1/eps_v+1/eps_w-1)", ""),
 ("eq10", "wall energy balance", f"rho_w C_w dTw/dt = [{Fm['wall_solar_factor']}] alpha_w tau_g G + [{Fm['wall_rad_factor']}] h_ray (Tv-Tw) + [{Fm['wall_gas_factor']}] h_w (Tg-Tw)",
  "geometric factors must equal the printed Eq. 10 factors"),
 ("eq11", "gas-wall Nu", "Nu = h_w D_int/k_g = 12.5 + 0.048 Re ; Table 3 h_w=1.83 (fitted)", "coefficient_mode selects fitted constant or correlation"),
 ("eq12", "gas energy balance", f"eps rho_g c_g dTg/dt = -d(rho_g c_g u Tg)/dz + d/dz(lambda_gz dTg/dz) - 6h_gs/d_p (Tg-Ts) - [{Fm['gas_wall_factor']}] h_w (Tg-Tw)",
  "FV upwind advection; inlet T = Tamb (flow on); Danckwerts-type zero dispersive flux at both faces"),
 ("eq13", "axial dispersion", "lambda_gz/k_g = 7 + 0.5 Pr Re ; Table 3 lambda=0.11 (fitted)", "Re basis (particle) to verify"),
 ("eq14", "gas-solid Nu", "2 R_p h_gs/k_g = 2 + 1.1 Pr^0.33 Re^0.6 ; Table 3 h_gs=148.61 (fitted)", "not transferred to ImCOF"),
 ("eq15", "solid energy balance", "(1-eps) rho_s C_s dTs/dt = d/dz(k_s dTs/dz) + 6h_gs/d_p (Tg-Ts) + (1-eps) rho_s (-dH) dq/dt", "adiabatic solid ends"),
 ("bc", "initial/boundary conditions", "t=0: Tv=Tw=Tg=Ts=Tamb(0); gas inlet = Tamb when fan on; no periodic initialisation", ""),
 ("eq16", "Toth isotherm (13X only)", str(CFG["TOTH_13X"]["isotherm_form"]), "reproduction gate only"),
 ("eq17_20", "Toth T-dependence (13X only)", f"q_m={CFG['TOTH_13X']['q_m_form']}; K={CFG['TOTH_13X']['K_form']}; n={CFG['TOTH_13X']['n_form']}",
  "must be transcribed from the PDF; None = not transcribed"),
]
pdf_txt = ""
if res.get("PADAC_PDF"):
    try:
        import pypdf
    except ImportError:
        import subprocess; subprocess.run([sys.executable, "-m", "pip", "-q", "install", "pypdf"]); import pypdf
    pdf_txt = " ".join((pg.extract_text() or "") for pg in pypdf.PdfReader(res["PADAC_PDF"]).pages)
probes = {"eq7": ["0.0522", "0.0552", "1.5"], "eq11": ["12.5", "0.048", "1.83"], "eq13": ["0.11"], "eq14": ["148.61", "1.1"],
          "eq15": ["880", "1.6"], "eq16": ["Toth", "Tóth"], "eq17_20": ["115.853", "7.88", "0.35", "2.34", "0.72"]}
man = pd.DataFrame([dict(equation=e, description=d, implemented_form=f, note=n, pdf_verified_by_user=bool(V.get(e, False)),
                         pdf_number_probe=("; ".join(f"{p}:{'found' if p in pdf_txt else 'absent'}" for p in probes.get(e, [])) if pdf_txt else "PDF_NOT_AVAILABLE"))
                    for e, d, f, n in eqs])
man["status"] = np.where(man.pdf_verified_by_user, "VERIFIED_AGAINST_PDF", "UNVERIFIED_TRANSCRIPTION")
fc.save_csv(man, "PADAC_equation_manifest.csv")
if pdf_txt and "0.0552" in pdf_txt and "0.0522" not in pdf_txt:
    fc.warn("SKY_COEFF", "PDF text contains 0.0552, not 0.0522: set PADAC.sky_coeff accordingly (spec prints 0.0522).", S)
thermal_ok = man[man.equation.isin(["eq6", "eq7", "eq8", "eq9", "eq10", "eq11", "eq12", "eq13", "eq14", "eq15", "bc"])].pdf_verified_by_user.all()
fc.set_status("PADAC_EQUATIONS", "EXECUTED_VERIFIED" if thermal_ok else "EXECUTED_UNVERIFIED_TRANSCRIPTION",
              "" if thermal_ok else "reproduction capped at STRUCTURAL_REPRODUCTION_ONLY", S)
toth = CFG["TOTH_13X"]
if not all(toth[k] for k in ("isotherm_form", "q_m_form", "K_form", "n_form", "T0", "k_L")):
    fc.not_executed("PADAC_13X_SORPTION_COUPLED", "TOTH_EQ17_20_NOT_TRANSCRIBED", "Toth forms/T0/k_L not entered from PDF", S)
man[["equation", "status", "pdf_number_probe"]]


In [ ]:
import sys, os, importlib
sys.path.insert(0, "/content")
import numpy as np, pandas as pd
import fidelity_core as fc, fidelity_adapters as ad
importlib.reload(fc); importlib.reload(ad)
CFG = fc.load_config()
S = "S15"
# 15. PADAC parameter / provenance audit (reactor-side vs sorbent-side)
P = CFG["PADAC"]
rows = [dict(parameter=k, value=v[0], unit=v[1], provenance=v[2], side=v[3], material_scope="PADAC reactor" if v[3] == "reactor" else "Zeolite 13X only")
        for k, v in P.items()]
V_bed = np.pi * P["D_int"][0] ** 2 / 4 * P["L"][0]
rows.append(dict(parameter="rho_bulk_13X=(1-eps)rho_s", value=P["m_s_13X"][0] / V_bed, unit="kg/m3", provenance="CALCULATED_THIS_NOTEBOOK",
                 side="sorbent", material_scope="Zeolite 13X only", ))
rows.append(dict(parameter="k_s_13X=alpha_s*rho_p*C_s", value=P["alpha_s_13X"][0] * P["rho_p_13X"][0] * P["C_s_13X"][0], unit="W/m/K",
                 provenance="CALCULATED_THIS_NOTEBOOK", side="sorbent", material_scope="Zeolite 13X only (rho_p assumed)"))
for k, v in CFG["TOTH_13X"].items():
    rows.append(dict(parameter="Toth_" + k, value=v, unit="", side="sorbent", material_scope="Zeolite 13X only",
                     provenance="FITTED_BY_SOURCE_AUTHORS" if v is not None and k in ("minus_dH", "q_m0", "K0", "n0", "gamma", "beta") else "NOT_AVAILABLE"))
rows.append(dict(parameter="reported_MRE_fig6", value="10%, 11%, 9%, 20%", unit="", provenance="DIRECT_REPORTED_MEASURED", side="model",
                 material_scope="authors calibrated selected heat-transfer coefficients: field-tested realisation, not ground truth"))
df = pd.DataFrame(rows); fc.save_csv(df, "PADAC_parameter_provenance.csv")
for r in rows:
    if r["provenance"] == "ASSUMED_SENSITIVITY_PARAMETER":
        fc.record_assumption("PADAC:" + r["parameter"], r["value"], r["provenance"], "not given in spec; replace with PDF value", S, "PADAC")
df


In [ ]:
import sys, os, importlib
sys.path.insert(0, "/content")
import numpy as np, pandas as pd
import fidelity_core as fc, fidelity_adapters as ad
importlib.reload(fc); importlib.reload(ad)
CFG = fc.load_config()
S = "S16"
# 16. PADAC+13X numerical implementation (method of lines, BDF) + numerical verification.
# Idealised constant inputs below are NUMERICAL VERIFICATION ONLY — never used as climate forcing.
padac_params = fc.padac_params
P = CFG["PADAC"]; V_bed = np.pi * P["D_int"][0] ** 2 / 4 * P["L"][0]
eps13 = P["eps_bed_13X"][0]
sorb13 = dict(eps_bed=eps13, d_p=P["d_p_13X"][0], h_gs=P["h_gs_13X"][0], C_s=P["C_s_13X"][0],
              rho_s=P["m_s_13X"][0] / V_bed / (1 - eps13), k_s=P["alpha_s_13X"][0] * P["rho_p_13X"][0] * P["C_s_13X"][0],
              minus_dH_J_mol=P["minus_dH_13X"][0])
fc.put("padac_sorbent_13X", sorb13)
rows = []
def const_forcing(hours, Ta_C, G, v, flow):
    t = np.arange(hours + 1) * 3600.0
    return fc.Forcing(t, np.full(t.size, Ta_C + 273.15), np.full(t.size, G), np.full(t.size, v), np.full(t.size, flow))
# (a) night, no sun: sky radiation must pull the bed below ambient
r = fc.PadacModel(padac_params(CFG, sorb13), nz=CFG["PADAC_NZ"]).run(const_forcing(48, 15, 0, 2, False))
rows.append(dict(check="night_sign_Ts_below_Tamb", value=r.T_s_mean_C.iloc[-1] - 15, pass_=bool(r.T_s_mean_C.iloc[-1] < 15)))
# (b) constant sun: monotone response to wind (higher h_wind -> lower Ts)
hi = [fc.PadacModel(padac_params(CFG, sorb13), nz=CFG["PADAC_NZ"]).run(const_forcing(12, 20, 800, vv, False)).T_s_mean_C.iloc[-1] for vv in (1, 4)]
rows.append(dict(check="wind_monotonicity", value=hi[0] - hi[1], pass_=bool(hi[0] > hi[1])))
# (c) mesh convergence & (d) tolerance halving
ref = None
for nz in (10, 20, 40):
    x = fc.PadacModel(padac_params(CFG, sorb13), nz=nz).run(const_forcing(12, 20, 800, 2, False)).T_s_max_C.iloc[-1]
    rows.append(dict(check=f"mesh_nz{nz}_Tsmax_C", value=x, pass_=np.nan)); ref = x if nz == 40 else ref
d20 = abs(rows[-2]["value"] - ref)
rows.append(dict(check="mesh_|nz20-nz40|_K", value=d20, pass_=bool(d20 < 0.5)))
x2 = fc.PadacModel(padac_params(CFG, sorb13), nz=CFG["PADAC_NZ"]).run(const_forcing(12, 20, 800, 2, False), rtol=1e-7, atol=1e-6).T_s_max_C.iloc[-1]
rows.append(dict(check="tolerance_|tight-default|_K", value=abs(x2 - rows[-3]["value"]), pass_=bool(abs(x2 - rows[-3]["value"]) < 0.1)))
nv = pd.DataFrame(rows).rename(columns={"pass_": "pass"}); nv["kind"] = "NUMERICAL_VERIFICATION_ONLY"
fc.put("padac_numerical_verification", nv); display(nv)
fc.set_status("PADAC_IMPLEMENTATION", "EXECUTED" if nv["pass"].dropna().all() else "EXECUTED_WITH_NUMERICAL_CHECK_FAILURES",
              "thermal-only (dq/dt=0) unless Toth coupling transcribed", S)


In [ ]:
import sys, os, importlib
sys.path.insert(0, "/content")
import numpy as np, pandas as pd
import fidelity_core as fc, fidelity_adapters as ad
importlib.reload(fc); importlib.reload(ad)
CFG = fc.load_config()
S = "S17"
# 17. PADAC reproduction / validation gate. No force-fitting. Forcing and targets digitised from the same
# figure are refused as independent validation.
targets = pd.DataFrame(dict(day=["Mar17", "Mar18", "Mar19"], Ts_max_C=[90, 80, 93], night_premidnight_C=[21, 22, 17],
                            cooling_rate_C_per_h=[8, 4.5, 9], T_20h_C=[34, 28, 23], thermal_efficiency_pct=[51, 58, 47]))
nv = fc.get("padac_numerical_verification"); rows = []
fp = CFG["PADAC_FIELD_FORCING_PATH"]
if fp and CFG["PADAC_FIELD_SAME_FIGURE"]:
    fc.warn("PADAC_FIELD_CIRCULAR", "forcing and targets digitised from the same figure -> not independent validation", S); fp = None
if fp and os.path.exists(fp):
    fc.register_input(fp, CFG["PADAC_FIELD_FORCING_PROVENANCE"] or "NOT_AVAILABLE", "PADAC field forcing (March 17-19)", S)
    f = pd.read_csv(fp); f["timestamp"] = pd.to_datetime(f[fc.detect_col(f, [r"time", r"date"], "field time")])
    Ta = f[fc.detect_col(f, [r"^t(_?amb|emp)", r"ambient"], "field T_amb")].to_numpy(float) + 273.15
    G = f[fc.detect_col(f, [r"^g(_|$)", r"irrad", r"solar", r"ghi"], "field G")].to_numpy(float)
    v = f[fc.detect_col(f, [r"wind", r"^v(_|$)"], "field v")].to_numpy(float)
    t = (f.timestamp - f.timestamp.iloc[0]).dt.total_seconds().to_numpy()
    flow = (f.timestamp.dt.hour >= CFG["ADSORPTION_WINDOW_LOCAL_H"][0]) | (f.timestamp.dt.hour < CFG["ADSORPTION_WINDOW_LOCAL_H"][1])
    sim = fc.PadacModel(fc.padac_params(CFG, fc.get("padac_sorbent_13X")), nz=CFG["PADAC_NZ"]).run(fc.Forcing(t, Ta, G, v, flow.values))
    sim["timestamp"] = f.timestamp.iloc[0] + pd.to_timedelta(sim.t_s, unit="s"); sim["day"] = sim.timestamp.dt.strftime("%b%d")
    for (day, g), tg in zip(sim.groupby("day"), targets.itertuples()):
        pk = g.T_s_max_C.idxmax()
        after = g.loc[pk:]; t20 = after[after.timestamp.dt.hour == 20].T_s_mean_C
        cool = (g.loc[pk, "T_s_mean_C"] - t20.iloc[0]) / max((after.timestamp[after.timestamp.dt.hour == 20].iloc[0] - g.loc[pk, "timestamp"]).total_seconds() / 3600, 1e-9) if len(t20) else np.nan
        rows.append(dict(day=day, sim_Ts_max_C=g.T_s_max_C.max(), sim_T_20h_C=t20.iloc[0] if len(t20) else np.nan,
                         sim_mean_cooling_peak_to_20h_C_per_h=cool, target_Ts_max_C=tg.Ts_max_C, target_T_20h_C=tg.T_20h_C,
                         target_cooling_C_per_h=tg.cooling_rate_C_per_h))
    meas = fc.detect_col(f, [r"t_?s", r"bed", r"measured"], "measured T_s", required=False)
    if meas:
        m = np.interp(sim.t_s, t, f[meas].to_numpy(float))
        mre = float(np.mean(np.abs(sim.T_s_mean_C - m) / np.abs(m)))
        rows.append(dict(day="all", sim_vs_measured_MRE=mre, reported_MRE_range="9-20%"))
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(8, 3.5)); ax.plot(sim.timestamp, sim.T_s_mean_C, label="simulated bed-mean T_s")
        ax.plot(f.timestamp, f[meas], "k.", ms=3, label=f"field T_s ({CFG['PADAC_FIELD_TARGET_PROVENANCE']})")
        ax.set_ylabel("°C"); ax.legend(); ax.set_title("PADAC+13X source reproduction"); fig.autofmt_xdate()
        fig.savefig(os.path.join(fc.run_dir(), "figures", "fig1_PADAC_reproduction.png"), dpi=150, bbox_inches="tight")
    verified = fc.status_of("PADAC_EQUATIONS") == "EXECUTED_VERIFIED"
    exact = CFG["PADAC_FIELD_FORCING_PROVENANCE"] in ("DIRECT_REPORTED_MEASURED",)
    status = "EXECUTED_FIELD_COMPARISON" if (verified and exact) else "FIELD_TRACE_NOT_EXACTLY_REPRODUCIBLE_FROM_AVAILABLE_INPUT"
else:
    status = "STRUCTURAL_REPRODUCTION_ONLY"
    fc.warn("PADAC_FIELD_FORCING_UNAVAILABLE", "exact March 17-19 T_amb/G/v not available -> structural reproduction only", S)
val = pd.concat([targets.assign(kind="reported_target"), pd.DataFrame(rows).assign(kind="simulated_vs_target"), nv], ignore_index=True)
val["gate_status"] = status
val["note"] = "PADAC thermal model is a field-tested realisation (authors' MRE ~9-20%), not ground truth; thermal efficiency not recomputed (definition not transcribed)."
fc.save_csv(val, "PADAC_reproduction_validation.csv")
fc.set_status("PADAC_REPRODUCTION", status, "", S)


### Weather-forcing policy
Canonical P0 temperature/RH remain unchanged. If solar radiation and wind are absent from `MAIN_climate_hourly_2015_2024.csv`, Section 18 retrieves only `ALLSKY_SFC_SW_DWN` and `WS10M` from the NASA POWER Hourly API at the canonical 36 city coordinates and 2015–2024 UTC timestamps. The downloaded G/v values are joined only on exact `(city, timestamp_utc)` keys; canonical T/RH are never replaced, interpolated, or re-downloaded for the process calculation. The primary ImCOF environmental branch then uses the PADAC+13X thermal response as an exogenous controller trajectory; it does not infer an ImCOF bed temperature or ImCOF thermal properties.


In [ ]:

import sys, os, importlib
sys.path.insert(0, "/content")
import numpy as np, pandas as pd
import fidelity_core as fc, fidelity_adapters as ad
importlib.reload(fc); importlib.reload(ad)
CFG = fc.load_config(); S = "S18"
# 18. Canonical NASA POWER T/RH + NASA POWER G/v augmentation.
# Frozen P0 T/RH from MAIN_climate_hourly_2015_2024.csv are retained exactly.
# If G/v are absent, ONLY ALLSKY_SFC_SW_DWN and WS10M are retrieved from NASA POWER,
# using the same canonical coordinates and exact UTC timestamps. No T/RH substitution or gap filling.
branch = "PADAC_GLOBAL_FORCING"; elig=[]
def _go():
    w, order, wsrc, col = ad.load_climate_weather(CFG)
    have={k:(k in w.columns and w[k].notna().all()) for k in ("T_C","RH_pct","G_Wm2","v_ms")}
    sw_path=None; sw_prov=None
    if not (have["G_Wm2"] and have["v_ms"]):
        if CFG.get("SOLAR_WIND_PATH"):
            sw_path=CFG["SOLAR_WIND_PATH"]; sw_prov="EXPLICIT_EXTERNAL_INPUT"
            if not os.path.exists(sw_path): raise fc.HardStop(f"SOLAR_WIND_PATH_NOT_FOUND: {sw_path}")
            sw=pd.read_parquet(sw_path) if sw_path.endswith('.parquet') else pd.read_csv(sw_path)
            cc=fc.detect_col(sw,[r"^city$",r"^climate",r"^site$"],"solar/wind climate")
            tc=fc.detect_col(sw,[r"^timestamp(_utc)?$",r"time",r"date"],"solar/wind time")
            gc=fc.detect_col(sw,[r"^G_Wm2$",r"ghi",r"^g(_|$)",r"irrad",r"solar"],"G")
            vc=fc.detect_col(sw,[r"^v_ms$",r"wind",r"^ws",r"^v(_|$)"],"v")
            sw=sw.rename(columns={cc:"climate",tc:"timestamp",gc:"G_Wm2",vc:"v_ms"})[["climate","timestamp","G_Wm2","v_ms"]]
            sw["timestamp"]=pd.to_datetime(sw.timestamp,utc=True)
        elif CFG.get("NASA_POWER_AUTO_DOWNLOAD"):
            sw,sw_path=ad.download_nasa_power_solar_wind(CFG,w); sw_prov="NASA_POWER_API"
        else:
            raise fc.HardStop("PADAC_GLOBAL_FORCING_INCOMPLETE: G/v absent; provide SOLAR_WIND_PATH or enable NASA_POWER_AUTO_DOWNLOAD")
        fc.register_input(sw_path, sw_prov if sw_prov in fc.PROVENANCE_CLASSES else "NASA_POWER_API", "solar/wind forcing for canonical 36-city NASA POWER weather", S)
        if 'timestamp' not in sw: raise fc.HardStop('SOLAR_WIND_SCHEMA: timestamp missing')
        sw['timestamp']=pd.to_datetime(sw.timestamp,utc=True)
        if not (0<=sw.G_Wm2.min() and sw.G_Wm2.max()<1600 and 0<=sw.v_ms.min() and sw.v_ms.max()<80):
            raise fc.HardStop("UNIT_INCOMPATIBILITY: G not W/m2 or v not m/s")
        base=w.drop(columns=[c for c in ('G_Wm2','v_ms') if c in w.columns])
        keys=sw[['climate','timestamp','G_Wm2','v_ms']].drop_duplicates(['climate','timestamp'])
        m=base.merge(keys,on=['climate','timestamp'],how='left',indicator=True,validate='one_to_one')
        miss=int((m._merge!='both').sum())
        missval=int(m[['G_Wm2','v_ms']].isna().any(axis=1).sum())
        if miss or missval:
            raise fc.HardStop(f"CLIMATE_TIME_MERGE_MISMATCH: missing_keys={miss}, missing_Gv_values={missval} canonical (city,timestamp) rows")
        w=m.drop(columns='_merge'); have.update(G_Wm2=True,v_ms=True)
    for c in order:
        wc=w[w.climate==c]
        elig.append(dict(climate=c,n_hours=len(wc),start=wc.timestamp.min(),end=wc.timestamp.max(),
            **{f"has_{k}":(k in wc and wc[k].notna().all()) for k in ('T_C','RH_pct','G_Wm2','v_ms')},
            **{f"n_missing_{k}":int(wc[k].isna().sum()) if k in wc else len(wc) for k in ('T_C','RH_pct','G_Wm2','v_ms')}))
    E=pd.DataFrame(elig); E['eligible']=E[[f'has_{k}' for k in ('T_C','RH_pct','G_Wm2','v_ms')]].all(1)
    E['weather_T_RH_source']=wsrc; E['solar_wind_source']=sw_path or wsrc
    E['solar_wind_provenance']='NASA_POWER_API' if sw_prov=='NASA_POWER_API' else ('DIRECT_IN_FILE' if sw_path is None else sw_prov)
    fc.save_csv(E,'PADAC_weather_forcing_eligibility.csv')
    if len(order)!=CFG['N_CLIMATES_EXPECTED'] or not E.eligible.all() or E[['start','end']].nunique().max()>1:
        raise fc.HardStop(f"PADAC_GLOBAL_FORCING_INCOMPLETE: {int((~E.eligible).sum())} of {len(E)} climates incomplete")
    w.to_parquet(os.path.join(fc._state_dir(),'forcing.parquet'),index=False)
    return True
if fc.guarded(branch,S,_go):
    fc.set_status(branch,'EXECUTED_COMPLETE','canonical NASA POWER T/RH + exact-key NASA POWER G/v augmentation',S)


In [ ]:
import sys, os, importlib, time
sys.path.insert(0, "/content")
import numpy as np, pandas as pd
import fidelity_core as fc, fidelity_adapters as ad
importlib.reload(fc); importlib.reload(ad)
CFG=fc.load_config(); S="S19"; branch="PADAC_SOLVER_OPTIMIZATION"
# 19. Information-preserving solver optimization gate.
# No weather sample is subsampled. The optimized engine must reproduce the legacy segmented BDF
# on identical hourly inputs before it is allowed to run the 36-climate calculation.
def _go():
    if not fc.is_ok('PADAC_GLOBAL_FORCING'):
        raise fc.HardStop(f"UPSTREAM_PADAC_GLOBAL_FORCING: {fc.status_of('PADAC_GLOBAL_FORCING')}")
    mode=CFG.get('PADAC_SOLVER_MODE','legacy_segmented')
    if mode=='legacy_segmented':
        fc.put('padac_solver_engine','legacy_segmented')
        fc.save_csv(pd.DataFrame([dict(engine='legacy_segmented',reason='configured legacy',passed=True)]),'PADAC_solver_optimization_validation.csv')
        return 'legacy_segmented'
    w=pd.read_parquet(os.path.join(fc._state_dir(),'forcing.parquet'))
    stats=w.groupby('climate',sort=False).G_Wm2.mean().sort_values()
    n=max(1,min(int(CFG.get('PADAC_OPT_VALIDATION_N_CLIMATES',3)),len(stats)))
    pick=np.unique(np.linspace(0,len(stats)-1,n).round().astype(int))
    climates=list(stats.index[pick])
    rr=fc.get('padac_sorbent_13X'); var=CFG['THERMAL_VARIANTS'][0]; P=fc.padac_params(CFG,rr,var)
    h0,h1=CFG['ADSORPTION_WINDOW_LOCAL_H']; rows=[]
    for c in climates:
        wc=w[w.climate==c].sort_values('timestamp').reset_index(drop=True)
        nh=min(len(wc),int(CFG.get('PADAC_OPT_VALIDATION_DAYS',7))*24+1)
        wc=wc.iloc[:nh].copy()
        if len(wc)<25: raise fc.HardStop(f'PADAC_OPT_VALIDATION_TOO_SHORT: {c}')
        local=fc.local_time(wc,CFG); ads,_=fc.cycle_labels(pd.Series(pd.to_datetime(local.values)),h0,h1)
        t=(wc.timestamp-wc.timestamp.iloc[0]).dt.total_seconds().to_numpy()
        F=fc.Forcing(t,wc.T_C.to_numpy(float)+273.15,wc.G_Wm2.to_numpy(float),wc.v_ms.to_numpy(float),ads)
        m=fc.PadacModel(P,nz=CFG['PADAC_NZ'])
        cmp,_,_=m.compare_engines(F,chunk_hours=CFG.get('PADAC_OPT_CHUNK_HOURS',744),max_step_s=CFG.get('PADAC_OPT_MAX_STEP_S'))
        cmp.insert(0,'climate',c); rows.append(cmp)
    V=pd.concat(rows,ignore_index=True)
    temp=V[V.variable.str.startswith(('T_v','T_w','T_g','T_s'))]
    max_abs=float(temp.max_abs.max()); rmse=float(temp.rmse.max())
    passed=(max_abs<=float(CFG['PADAC_OPT_MAX_ABS_T_C']) and rmse<=float(CFG['PADAC_OPT_RMSE_T_C']))
    V['gate_max_abs_C']=CFG['PADAC_OPT_MAX_ABS_T_C']; V['gate_rmse_C']=CFG['PADAC_OPT_RMSE_T_C']; V['passed']=passed
    fc.save_csv(V,'PADAC_solver_optimization_validation.csv')
    engine='optimized' if passed else 'legacy_segmented'
    fc.put('padac_solver_engine',engine)
    fc.save_csv(pd.DataFrame([dict(selected_engine=engine,validation_passed=passed,max_abs_temperature_C=max_abs,max_rmse_temperature_C=rmse,
        n_climates=len(climates),validation_days=CFG.get('PADAC_OPT_VALIDATION_DAYS',7),chunk_hours=CFG.get('PADAC_OPT_CHUNK_HOURS',744),
        information_policy='ALL_ORIGINAL_HOURLY_FORCING_AND_OUTPUT_TIMESTAMPS_RETAINED')]),'PADAC_solver_optimization_manifest.csv')
    if not passed:
        fc.warn('PADAC_OPT_REGRESSION_FAILED',f'optimized max_abs={max_abs:.6g} C, rmse={rmse:.6g} C; falling back to legacy segmented solver',S)
        fc.set_status(branch,'EXECUTED_FALLBACK_LEGACY','optimized engine failed strict regression; no scientific result uses it',S)
    else:
        sp=float(np.nanmedian(V.speedup)) if 'speedup' in V else np.nan
        fc.set_status(branch,'PASSED',f'optimized solver selected; validation max_abs={max_abs:.6g} C, rmse={rmse:.6g} C, median short-window speedup={sp:.2f}x',S)
    return engine
fc.guarded(branch,S,_go)
print('selected engine:',fc.get('padac_solver_engine','legacy_segmented'))


# PART D — ImCOF-TAEA process pair


In [ ]:
import sys, os, importlib
sys.path.insert(0, "/content")
import numpy as np, pandas as pd
import fidelity_core as fc, fidelity_adapters as ad
importlib.reload(fc); importlib.reload(ad)
CFG = fc.load_config()
S = "S20"
# 20. ImCOF active 80 C branch — supported-domain process endpoint.
# Unsupported equilibrium states are NOT extrapolated and do not invalidate an entire climate.
# single/tensor are evaluated on the same cycles; the COMMON finite-support mask is frozen first.
# Primary Y_active = mean(q_rich * F_R) over those common-supported cycles.
# Supported-cycle count/fraction are reported separately; zero-supported climates remain non-evaluable (NaN).
branch = "IMCOF_ACTIVE"

def _go():
    for up in ("EQUILIBRIUM_P0_ImCOF", "IMCOF_KERNEL"):
        if not fc.is_ok(up):
            raise fc.HardStop(f"UPSTREAM_{up}: {fc.status_of(up)}")

    kern = fc.get("tpd_kernel")
    meta = fc.get("tpd_primary_meta")
    pooled = fc.get("tpd_kernel_pooled", None)
    w, order, *_ = ad.load_climate_weather(CFG)
    if len(order) != CFG["N_CLIMATES_EXPECTED"]:
        raise fc.HardStop(f"CLIMATE_ENSEMBLE_MISMATCH: {len(order)} climates")
    w["local"] = fc.local_time(w, CFG).values
    reps = ad.build_representations(CFG["MATERIALS"]["ImCOF"], CFG)
    rep_names = list(reps)
    Treg = CFG["IMCOF_ACTIVE"]["T_reg_C"]
    rows, cyc_rows, coverage_rows = [], [], []

    windows = [("primary", CFG["ADSORPTION_WINDOW_LOCAL_H"])] + [
        (f"window_{a}-{b}", [a, b]) for a, b in CFG["ADSORPTION_WINDOW_SENSITIVITY"]
    ]
    modes = (
        CFG["IMCOF_ADS_CONDITION"],
        "window_mean" if CFG["IMCOF_ADS_CONDITION"] == "end_of_window" else "end_of_window"
    )

    for wlab, (h0, h1) in windows:
        for mode in modes:
            for c in order:
                wc = w[w.climate == c].sort_values("timestamp")
                cy = fc.imcof_cycles(
                    wc.local.values,
                    wc.T_C.to_numpy(float),
                    wc.RH_pct.to_numpy(float),
                    h0, h1, mode
                )
                if cy.empty:
                    for rep, R in reps.items():
                        rows.append(dict(
                            material=CFG["MATERIALS"]["ImCOF"], controller="active_thermal_80C",
                            representation=rep, level=R.get("level"), window=wlab, ads_condition=mode,
                            climate=c, n_cycles=0, n_supported_cycles=0, supported_cycle_fraction=0.0,
                            Y_active=np.nan, Y_active_supported_sum=np.nan,
                            Y_active_pooled_kernel=np.nan, Y_active_pooled_kernel_supported_sum=np.nan,
                            mean_q_rich=np.nan, mean_q_lean=np.nan,
                            frac_cycles_outside_support=np.nan, mean_kernel_distance=np.nan,
                            endpoint_definition="SUPPORTED_CYCLE_MEAN_COMMON_REPRESENTATION_SUPPORT"
                        ))
                    coverage_rows.append(dict(window=wlab, ads_condition=mode, climate=c, n_cycles=0,
                                              n_supported_cycles=0, supported_cycle_fraction=0.0))
                    continue

                sel = [fc.tpd_select(meta, a, b, CFG) for a, b in zip(cy.T_ads, cy.RH_ads)]
                cy["curve_id"] = [s[0] for s in sel]
                cy["kernel_distance"] = [s[1] for s in sel]
                Fk = np.array([
                    kern[k]["F_end_after_hold"]
                    if kern[k]["hold_min"] >= 60 * CFG["IMCOF_ACTIVE"]["duration_h"] - 1e-9
                    else fc.F_R(kern, k, Treg)
                    for k in cy.curve_id
                ], float)
                Fp = float(pooled["F_end_after_hold"]) if pooled else np.nan

                sup = CFG["IMCOF_SUPPORT"]
                out_sup = (
                    (cy.T_ads < sup["T_C"][0]) | (cy.T_ads > sup["T_C"][1]) |
                    (cy.RH_ads < sup["RH_pct"][0]) | (cy.RH_ads > sup["RH_pct"][1])
                )
                if out_sup.any() and CFG["EXTRAPOLATION_POLICY"] != "P0_RULE":
                    raise fc.HardStop(
                        f"UNSUPPORTED_EQUILIBRIUM_EXTRAPOLATION: {c}: {out_sup.mean():.1%} cycles outside ImCOF support"
                    )

                q_by_rep = {
                    rep: np.asarray(R["q"](cy.T_ads.to_numpy(float), cy.RH_ads.to_numpy(float)), float)
                    for rep, R in reps.items()
                }
                common = np.ones(len(cy), dtype=bool)
                for rep in rep_names:
                    common &= np.isfinite(q_by_rep[rep])
                # Process kernel must also be finite on an eligible cycle.
                common &= np.isfinite(Fk)

                n_sup = int(common.sum())
                frac_sup = float(common.mean()) if len(common) else 0.0
                coverage_rows.append(dict(
                    window=wlab, ads_condition=mode, climate=c, n_cycles=len(cy),
                    n_supported_cycles=n_sup, supported_cycle_fraction=frac_sup,
                    T_ads_min=float(cy.T_ads.min()), T_ads_max=float(cy.T_ads.max()),
                    RH_ads_min=float(cy.RH_ads.min()), RH_ads_max=float(cy.RH_ads.max())
                ))

                for rep, R in reps.items():
                    qr = q_by_rep[rep]
                    if n_sup:
                        vals = qr[common] * Fk[common]
                        y_mean = float(np.mean(vals))
                        y_sum = float(np.sum(vals))
                        if np.isfinite(Fp):
                            pv = qr[common] * Fp
                            yp_mean = float(np.mean(pv))
                            yp_sum = float(np.sum(pv))
                        else:
                            yp_mean = yp_sum = np.nan
                        mean_q = float(np.mean(qr[common]))
                    else:
                        y_mean = y_sum = yp_mean = yp_sum = mean_q = np.nan

                    rows.append(dict(
                        material=CFG["MATERIALS"]["ImCOF"], controller="active_thermal_80C",
                        representation=rep, level=R.get("level"), window=wlab, ads_condition=mode,
                        climate=c, n_cycles=len(cy), n_supported_cycles=n_sup,
                        supported_cycle_fraction=frac_sup,
                        Y_active=y_mean, Y_active_supported_sum=y_sum,
                        Y_active_pooled_kernel=yp_mean,
                        Y_active_pooled_kernel_supported_sum=yp_sum,
                        mean_q_rich=mean_q, mean_q_lean=np.nan,
                        frac_cycles_outside_support=float(out_sup.mean()),
                        mean_kernel_distance=float(cy.kernel_distance.mean()),
                        endpoint_definition="SUPPORTED_CYCLE_MEAN_COMMON_REPRESENTATION_SUPPORT"
                    ))

                if wlab == "primary" and mode == CFG["IMCOF_ADS_CONDITION"]:
                    cc = cy.drop(columns=["ads_idx", "reg_idx"]).copy()
                    cc["common_supported_cycle"] = common
                    cc["climate"] = c
                    cyc_rows.append(cc)

    df = pd.DataFrame(rows)
    cov = pd.DataFrame(coverage_rows)
    fc.save_csv(df, "ImCOF_active_thermal_climate_scores.csv")
    fc.save_csv(cov, "ImCOF_active_supported_cycle_coverage.csv")
    fc.put("imcof_active_scores", df)
    fc.put("imcof_active_coverage", cov)
    fc.put("imcof_cycles_primary", pd.concat(cyc_rows, ignore_index=True) if cyc_rows else pd.DataFrame())

    if len(df) and df.n_cycles.nunique() > 1:
        fc.warn("CYCLE_COUNT_VARIES", "complete-cycle counts differ across climates (period edges / gaps)", S)
    zero = cov[cov.n_supported_cycles == 0]
    if len(zero):
        fc.warn("IMCOF_ZERO_SUPPORTED_CLIMATES",
                f"{len(zero)} climate/window/ads-condition cells have zero common-supported cycles; they remain non-evaluable.", S)
    return True

if fc.guarded(branch, S, _go):
    fc.set_status(branch, "EXECUTED_SUPPORTED_DOMAIN",
                  "Y_active is supported-cycle mean on common single/tensor support; coverage reported separately", S)


In [ ]:
import sys, os, importlib, time
sys.path.insert(0, "/content")
import numpy as np, pandas as pd, hashlib, json
import fidelity_core as fc, fidelity_adapters as ad
importlib.reload(fc); importlib.reload(ad)
CFG=fc.load_config(); S="S21"; branch="PADAC_IMCOF_ENV"
# 21. PRIMARY ImCOF environmental thermal endpoint.
# Primary-first execution: only THERMAL_VARIANTS[0] is solved here.
# Alternative wind/sky/coefficient variants are moved to optional S21b.
# This changes orchestration only: the primary controller equations, forcing, ODE grid,
# solver, supported-cycle endpoint, and downstream fidelity definitions are unchanged.
from multiprocessing import Pool

def _go():
    mode=CFG.get('IMCOF_ENV_THERMAL_MODE','padac_13X_controller')
    required=['PADAC_GLOBAL_FORCING','EQUILIBRIUM_P0_ImCOF','IMCOF_KERNEL','PADAC_IMPLEMENTATION']
    if mode=='material_specific': required.append('IMCOF_THERMAL_PROPS')
    for up in required:
        if not fc.is_ok(up): raise fc.HardStop(f"UPSTREAM_{up}: {fc.status_of(up)}")

    w=pd.read_parquet(os.path.join(fc._state_dir(),'forcing.parquet'))
    w['local']=fc.local_time(w,CFG).values
    order=CFG.get('CLIMATE_ORDER') or list(pd.unique(w.climate))
    kern=fc.get('tpd_kernel'); meta=fc.get('tpd_primary_meta')
    reps=ad.build_representations(CFG['MATERIALS']['ImCOF'],CFG)
    rep_names=list(reps)
    h0,h1=CFG['ADSORPTION_WINDOW_LOCAL_H']
    cache=CFG.get('PADAC_CONTROLLER_CACHE_DIR') or os.path.join(
        fc.run_dir(),'thermal_runs_PADAC13X' if mode=='padac_13X_controller' else 'thermal_runs_ImCOF')
    engine=fc.get('padac_solver_engine','legacy_segmented')
    os.makedirs(cache,exist_ok=True)
    jobs=[]; cyc={}; tags=[]; queued_paths=set(); static={}

    if mode=='padac_13X_controller':
        realisations=[dict(id='PADAC_13X_FIELD_REALISATION',sorb=fc.get('padac_sorbent_13X'))]
        fc.record_assumption('ImCOF_environmental_Ts_source','PADAC+13X field-tested thermal realisation','CALCULATED_THIS_NOTEBOOK',
            'T_s(t) is an exogenous architecture/controller forcing; not an ImCOF bed-temperature prediction',S,branch)
    elif mode=='material_specific':
        thetas=fc.get('imcof_theta'); realisations=[]
        for th in thetas:
            eps=th['eps_bed']; rho_s=th['rho_bulk']/(1-eps); h_gs=th.get('h_gs')
            if h_gs is None:
                rho,cp,mu,k=fc.air_props(300.0)
                u=CFG['PADAC']['Q_m3_h'][0]/3600/(np.pi*CFG['PADAC']['D_int'][0]**2/4)
                Re=rho*u*th['d_p']/mu
                h_gs=(2+1.1*(cp*mu/k)**0.33*Re**0.6)*k/th['d_p']
            sorb=dict(eps_bed=eps,d_p=th['d_p'],h_gs=h_gs,C_s=th['C_s'],rho_s=rho_s,k_s=th['k_s'],minus_dH_J_mol=0.0)
            realisations.append(dict(id=th['id'],sorb=sorb))
    else:
        raise fc.HardStop(f'UNKNOWN_IMCOF_ENV_THERMAL_MODE: {mode}')

    primary_variant=CFG['THERMAL_VARIANTS'][0]
    variants=[primary_variant]
    if primary_variant.get('id')!='primary':
        fc.warn('PRIMARY_THERMAL_VARIANT_ID',f"First thermal variant id is {primary_variant.get('id')!r}, not 'primary'. It is still treated as primary by frozen order.",S)

    zero_static=[]
    for c in order:
        wc=w[w.climate==c].sort_values('timestamp').reset_index(drop=True)
        cy=fc.imcof_cycles(wc.local.values,wc.T_C.to_numpy(float),wc.RH_pct.to_numpy(float),h0,h1,CFG['IMCOF_ADS_CONDITION'])
        if cy.empty: raise fc.HardStop(f'NO_COMPLETE_CYCLES: {c}')
        cyc[c]=(wc,cy)
        sel=[fc.tpd_select(meta,a,b,CFG) for a,b in zip(cy.T_ads,cy.RH_ads)]
        qrich={rep:np.asarray(R['q'](cy.T_ads.to_numpy(float),cy.RH_ads.to_numpy(float)),float) for rep,R in reps.items()}
        common=np.ones(len(cy),dtype=bool)
        for rep in rep_names: common &= np.isfinite(qrich[rep])
        static[c]=dict(sel=sel,qrich=qrich,common=common,n_supported_cycles=int(common.sum()),
                       supported_cycle_fraction=float(common.mean()) if len(common) else 0.0)
        if not common.any(): zero_static.append(c)
        ads_mask,_=fc.cycle_labels(pd.Series(pd.to_datetime(wc.local.values)),h0,h1)
        t=(wc.timestamp-wc.timestamp.iloc[0]).dt.total_seconds().to_numpy()
        for var in variants:
            for rr in realisations:
                P=fc.padac_params(CFG,rr['sorb'],var)
                sig_payload=dict(P=P,nz=CFG['PADAC_NZ'],engine=engine,
                                 chunk_hours=CFG.get('PADAC_OPT_CHUNK_HOURS',744),max_step_s=CFG.get('PADAC_OPT_MAX_STEP_S'))
                sig=hashlib.sha256(json.dumps(sig_payload,sort_keys=True,default=str).encode()).hexdigest()[:16]
                safe=fc.re.sub(r'[^A-Za-z0-9_.-]+','_',str(c))
                out=os.path.join(cache,f"{mode}__{rr['id']}__{safe}__{sig}.parquet")
                args=(P,CFG['PADAC_NZ'],t,wc.T_C.to_numpy(float)+273.15,wc.G_Wm2.to_numpy(float),wc.v_ms.to_numpy(float),
                      ads_mask,out,None,engine,CFG.get('PADAC_OPT_CHUNK_HOURS',744),CFG.get('PADAC_OPT_MAX_STEP_S'))
                if out not in queued_paths:
                    jobs.append(args); queued_paths.add(out)
                tags.append(dict(mode=mode,variant=var['id'],theta=rr['id'],climate=c,path=out,
                                 solver_engine=engine,thermal_signature=sig))

    D=pd.DataFrame(tags)
    # Primary controller must retain one distinct thermal path per climate when there is one realization.
    if mode=='padac_13X_controller' and len(realisations)==1:
        n_path=int(D['path'].nunique())
        if n_path!=len(order):
            raise fc.HardStop(f'PADAC_PRIMARY_CACHE_KEY_COLLISION: {n_path} unique thermal paths for {len(order)} climates')
    dedup=(D.groupby(['path','thermal_signature','solver_engine'],as_index=False)
             .agg(n_scientific_variant_labels=('variant','size'),climate=('climate','first'),theta=('theta','first'),
                  variant_labels=('variant',lambda x:'|'.join(map(str,x)))))
    dedup['execution_tier']='PRIMARY'
    dedup['n_unique_thermal_jobs']=len(jobs); dedup['n_total_scientific_labels']=len(tags)
    dedup['information_loss']='NONE_PRIMARY_ONLY_BY_DESIGN'
    fc.save_csv(dedup,'PADAC_thermal_job_dedup_manifest.csv')
    print(f"PRIMARY: {len(jobs)} unique PADAC thermal-controller runs for {len(tags)} primary labels; mode={mode}; solver={engine}; cache={cache}")
    if zero_static:
        print(f"Note: {len(zero_static)} climates have zero common equilibrium-supported adsorption cycles; thermal solve is retained for a complete primary controller forcing record, but Y remains non-evaluable there.")

    t0=time.monotonic()
    with Pool(CFG['N_JOBS']) as pool:
        for i,_ in enumerate(pool.imap_unordered(fc._thermal_job,jobs),start=1):
            elapsed=(time.monotonic()-t0)/60.0
            print(f"  {i}/{len(jobs)} completed  elapsed={elapsed:.1f} min", flush=True)

    rows=[]; trows=[]; coverage_rows=[]; by_path={}
    for tag in tags: by_path.setdefault(tag['path'],[]).append(tag)
    for path,aliases in by_path.items():
        r=pd.read_parquet(path); c=aliases[0]['climate']; wc,cy=cyc[c]
        tt=(wc.timestamp-wc.timestamp.iloc[0]).dt.total_seconds().to_numpy()
        if len(r)!=len(tt) or not np.allclose(r.t_s.to_numpy(float),tt,rtol=0,atol=1e-6):
            raise fc.HardStop(f'PADAC_THERMAL_CACHE_GRID_MISMATCH: {c}/{path}')
        Ts=r.T_s_mean_C.to_numpy(float); Treg=np.array([Ts[ix].max() for ix in cy.reg_idx])
        sel=static[c]['sel']; Fk=np.array([fc.F_R(kern,q[0],x) for q,x in zip(sel,Treg)],float)
        above=np.array([x>kern[q[0]]['T'][-1] for q,x in zip(sel,Treg)])
        common=static[c]['common'] & np.isfinite(Fk)
        n_sup=int(common.sum()); frac_sup=float(common.mean()) if len(common) else 0.0
        for tag in aliases:
            wind_name=next(v['wind_correlation'] for v in CFG['THERMAL_VARIANTS'] if v['id']==tag['variant'])
            wind_lo,wind_hi=CFG['WIND_VALIDITY_MS'][wind_name]
            trows.append(dict(**tag,execution_tier='PRIMARY',Ts_max_C=float(Ts.max()),Ts_mean_C=float(Ts.mean()),
                mean_daily_regen_Tmax_C=float(Treg.mean()),frac_cycles_Treg_ge45=float((Treg>=45).mean()),
                frac_cycles_Treg_ge65=float((Treg>=65).mean()),frac_cycles_above_kernel_range=float(above.mean()),
                frac_hours_wind_outside_validity=float(((wc.v_ms<wind_lo)|(wc.v_ms>wind_hi)).mean()),
                n_supported_cycles=n_sup,supported_cycle_fraction=frac_sup,
                interpretation=('EXOGENOUS_PADAC13X_CONTROLLER_NOT_IMCOF_TEMPERATURE' if mode=='padac_13X_controller' else 'MATERIAL_SPECIFIC_MODEL_COMPOSITION')))
            coverage_rows.append(dict(climate=c,thermal_variant=tag['variant'],theta=tag['theta'],execution_tier='PRIMARY',n_cycles=len(cy),
                                      n_supported_cycles=n_sup,supported_cycle_fraction=frac_sup))
            for rep,R in reps.items():
                qr=static[c]['qrich'][rep]
                if n_sup:
                    vals=qr[common]*Fk[common]; y_mean=float(np.mean(vals)); y_sum=float(np.sum(vals)); mean_q=float(np.mean(qr[common])); mean_f=float(np.mean(Fk[common]))
                else: y_mean=y_sum=mean_q=mean_f=np.nan
                rows.append(dict(material=CFG['MATERIALS']['ImCOF'],controller='environmental_thermal_PADAC',thermal_mode=mode,
                    representation=rep,level=R.get('level'),thermal_variant=tag['variant'],theta=tag['theta'],climate=tag['climate'],execution_tier='PRIMARY',n_cycles=len(cy),
                    n_supported_cycles=n_sup,supported_cycle_fraction=frac_sup,Y_env=y_mean,Y_env_supported_sum=y_sum,
                    mean_q_rich=mean_q,mean_F_R=mean_f,endpoint_definition='SUPPORTED_CYCLE_MEAN_COMMON_REPRESENTATION_SUPPORT',
                    interpretation=('PADAC+13X T_s forcing applied to ImCOF response; not ImCOF bed T prediction' if mode=='padac_13X_controller' else 'PADAC+ImCOF material-specific model composition')))

    df=pd.DataFrame(rows); cov=pd.DataFrame(coverage_rows).drop_duplicates(['climate','thermal_variant','theta'])
    tsum=pd.DataFrame(trows)
    fc.save_csv(df,'ImCOF_environmental_thermal_climate_scores.csv'); fc.save_csv(cov,'ImCOF_environmental_supported_cycle_coverage.csv')
    fc.put('imcof_env_scores',df); fc.put('imcof_env_coverage',cov); fc.put('imcof_thermal_summary',tsum)
    zero=cov[cov.n_supported_cycles==0]
    if len(zero): fc.warn('IMCOF_ENV_ZERO_SUPPORTED_CLIMATES',f'{len(zero)} primary climate cells have zero common-supported cycles; they remain non-evaluable.',S)
    return len(jobs),mode

ans=fc.guarded(branch,S,_go)
if ans:
    n,mode=ans
    status='EXECUTED_CONTROLLER_FORCING' if mode=='padac_13X_controller' else 'EXECUTED_MATERIAL_SPECIFIC_MODEL'
    fc.set_status(branch,status,f"{n} PRIMARY thermal runs; {mode}; sensitivity deferred to optional S21b",S)


In [ ]:
import sys, os, importlib, time
sys.path.insert(0, "/content")
import numpy as np, pandas as pd, hashlib, json
import fidelity_core as fc, fidelity_adapters as ad
importlib.reload(fc); importlib.reload(ad)
CFG=fc.load_config(); S="S21b"; branch="PADAC_IMCOF_ENV_SENSITIVITY"
# 21b. OPTIONAL thermal-controller model-form sensitivity.
# Default OFF. Set RUN_PADAC_THERMAL_SENSITIVITY=True in Section 0 before running this cell.
# Primary S21 must be completed first. This appends alternative thermal variants to the same output tables.
from multiprocessing import Pool

if not CFG.get('RUN_PADAC_THERMAL_SENSITIVITY',False):
    fc.not_executed(branch,'DISABLED_BY_CONFIG','Set RUN_PADAC_THERMAL_SENSITIVITY=True to run the non-primary wind/sky/coefficient variants.',S)
    print('S21b sensitivity skipped (RUN_PADAC_THERMAL_SENSITIVITY=False). Primary S21 results are sufficient for the paired-controller endpoint.')
elif not fc.is_ok('PADAC_IMCOF_ENV'):
    fc.not_executed(branch,'UPSTREAM_PRIMARY_NOT_EXECUTED',fc.status_of('PADAC_IMCOF_ENV'),S)
    print('S21b sensitivity skipped: primary S21 has not executed successfully.')
else:
    def _go_sens():
        mode=CFG.get('IMCOF_ENV_THERMAL_MODE','padac_13X_controller')
        w=pd.read_parquet(os.path.join(fc._state_dir(),'forcing.parquet')); w['local']=fc.local_time(w,CFG).values
        order=CFG.get('CLIMATE_ORDER') or list(pd.unique(w.climate)); kern=fc.get('tpd_kernel'); meta=fc.get('tpd_primary_meta')
        reps=ad.build_representations(CFG['MATERIALS']['ImCOF'],CFG); rep_names=list(reps); h0,h1=CFG['ADSORPTION_WINDOW_LOCAL_H']
        cache=CFG.get('PADAC_CONTROLLER_CACHE_DIR') or os.path.join(fc.run_dir(),'thermal_runs_PADAC13X' if mode=='padac_13X_controller' else 'thermal_runs_ImCOF')
        engine=fc.get('padac_solver_engine','legacy_segmented'); os.makedirs(cache,exist_ok=True)
        if mode=='padac_13X_controller':
            realisations=[dict(id='PADAC_13X_FIELD_REALISATION',sorb=fc.get('padac_sorbent_13X'))]
        else:
            realisations=[]
            for th in fc.get('imcof_theta'):
                eps=th['eps_bed']; rho_s=th['rho_bulk']/(1-eps); h_gs=th.get('h_gs')
                if h_gs is None:
                    rho,cp,mu,k=fc.air_props(300.0); u=CFG['PADAC']['Q_m3_h'][0]/3600/(np.pi*CFG['PADAC']['D_int'][0]**2/4)
                    Re=rho*u*th['d_p']/mu; h_gs=(2+1.1*(cp*mu/k)**0.33*Re**0.6)*k/th['d_p']
                realisations.append(dict(id=th['id'],sorb=dict(eps_bed=eps,d_p=th['d_p'],h_gs=h_gs,C_s=th['C_s'],rho_s=rho_s,k_s=th['k_s'],minus_dH_J_mol=0.0)))
        variants=CFG['THERMAL_VARIANTS'][1:]
        jobs=[]; tags=[]; queued=set(); cyc={}; static={}
        for c in order:
            wc=w[w.climate==c].sort_values('timestamp').reset_index(drop=True)
            cy=fc.imcof_cycles(wc.local.values,wc.T_C.to_numpy(float),wc.RH_pct.to_numpy(float),h0,h1,CFG['IMCOF_ADS_CONDITION']); cyc[c]=(wc,cy)
            sel=[fc.tpd_select(meta,a,b,CFG) for a,b in zip(cy.T_ads,cy.RH_ads)]
            qrich={rep:np.asarray(R['q'](cy.T_ads.to_numpy(float),cy.RH_ads.to_numpy(float)),float) for rep,R in reps.items()}
            common=np.ones(len(cy),dtype=bool)
            for rep in rep_names: common &= np.isfinite(qrich[rep])
            static[c]=dict(sel=sel,qrich=qrich,common=common)
            ads_mask,_=fc.cycle_labels(pd.Series(pd.to_datetime(wc.local.values)),h0,h1); t=(wc.timestamp-wc.timestamp.iloc[0]).dt.total_seconds().to_numpy()
            for var in variants:
                for rr in realisations:
                    P=fc.padac_params(CFG,rr['sorb'],var)
                    sig_payload=dict(P=P,nz=CFG['PADAC_NZ'],engine=engine,chunk_hours=CFG.get('PADAC_OPT_CHUNK_HOURS',744),max_step_s=CFG.get('PADAC_OPT_MAX_STEP_S'))
                    sig=hashlib.sha256(json.dumps(sig_payload,sort_keys=True,default=str).encode()).hexdigest()[:16]
                    safe=fc.re.sub(r'[^A-Za-z0-9_.-]+','_',str(c)); out=os.path.join(cache,f"{mode}__{rr['id']}__{safe}__{sig}.parquet")
                    args=(P,CFG['PADAC_NZ'],t,wc.T_C.to_numpy(float)+273.15,wc.G_Wm2.to_numpy(float),wc.v_ms.to_numpy(float),ads_mask,out,None,engine,CFG.get('PADAC_OPT_CHUNK_HOURS',744),CFG.get('PADAC_OPT_MAX_STEP_S'))
                    if out not in queued: jobs.append(args); queued.add(out)
                    tags.append(dict(mode=mode,variant=var['id'],theta=rr['id'],climate=c,path=out,solver_engine=engine,thermal_signature=sig))
        D=pd.DataFrame(tags); dedup=(D.groupby(['path','thermal_signature','solver_engine'],as_index=False).agg(n_scientific_variant_labels=('variant','size'),climate=('climate','first'),theta=('theta','first'),variant_labels=('variant',lambda x:'|'.join(map(str,x)))))
        dedup['execution_tier']='SENSITIVITY'; dedup['n_unique_thermal_jobs']=len(jobs); dedup['n_total_scientific_labels']=len(tags); dedup['information_loss']='NONE_IDENTICAL_ODE_PARAMETER_SIGNATURES_ONLY'
        primary_manifest_path=os.path.join(fc.run_dir(),'PADAC_thermal_job_dedup_manifest.csv')
        oldm=pd.read_csv(primary_manifest_path) if os.path.exists(primary_manifest_path) else pd.DataFrame()
        fc.save_csv(pd.concat([oldm,dedup],ignore_index=True,sort=False),'PADAC_thermal_job_dedup_manifest.csv')
        print(f"SENSITIVITY: {len(jobs)} unique thermal jobs for {len(tags)} labels. Existing cache files are skipped automatically.")
        t0=time.monotonic()
        with Pool(CFG['N_JOBS']) as pool:
            for i,_ in enumerate(pool.imap_unordered(fc._thermal_job,jobs),start=1):
                print(f"  {i}/{len(jobs)} completed  elapsed={(time.monotonic()-t0)/60:.1f} min",flush=True)
        rows=[]; trows=[]; coverage=[]; by_path={}
        for tag in tags: by_path.setdefault(tag['path'],[]).append(tag)
        for path,aliases in by_path.items():
            r=pd.read_parquet(path); c=aliases[0]['climate']; wc,cy=cyc[c]; Ts=r.T_s_mean_C.to_numpy(float); Treg=np.array([Ts[ix].max() for ix in cy.reg_idx])
            Fk=np.array([fc.F_R(kern,q[0],x) for q,x in zip(static[c]['sel'],Treg)],float); above=np.array([x>kern[q[0]]['T'][-1] for q,x in zip(static[c]['sel'],Treg)])
            common=static[c]['common'] & np.isfinite(Fk); n_sup=int(common.sum()); frac=float(common.mean()) if len(common) else 0.0
            for tag in aliases:
                wind_name=next(v['wind_correlation'] for v in CFG['THERMAL_VARIANTS'] if v['id']==tag['variant']); lo,hi=CFG['WIND_VALIDITY_MS'][wind_name]
                trows.append(dict(**tag,execution_tier='SENSITIVITY',Ts_max_C=float(Ts.max()),Ts_mean_C=float(Ts.mean()),mean_daily_regen_Tmax_C=float(Treg.mean()),frac_cycles_Treg_ge45=float((Treg>=45).mean()),frac_cycles_Treg_ge65=float((Treg>=65).mean()),frac_cycles_above_kernel_range=float(above.mean()),frac_hours_wind_outside_validity=float(((wc.v_ms<lo)|(wc.v_ms>hi)).mean()),n_supported_cycles=n_sup,supported_cycle_fraction=frac,interpretation='EXOGENOUS_PADAC13X_CONTROLLER_NOT_IMCOF_TEMPERATURE' if mode=='padac_13X_controller' else 'MATERIAL_SPECIFIC_MODEL_COMPOSITION'))
                coverage.append(dict(climate=c,thermal_variant=tag['variant'],theta=tag['theta'],execution_tier='SENSITIVITY',n_cycles=len(cy),n_supported_cycles=n_sup,supported_cycle_fraction=frac))
                for rep,R in reps.items():
                    qr=static[c]['qrich'][rep]
                    if n_sup:
                        vals=qr[common]*Fk[common]; ym=float(np.mean(vals)); ys=float(np.sum(vals)); mq=float(np.mean(qr[common])); mf=float(np.mean(Fk[common]))
                    else: ym=ys=mq=mf=np.nan
                    rows.append(dict(material=CFG['MATERIALS']['ImCOF'],controller='environmental_thermal_PADAC',thermal_mode=mode,representation=rep,level=R.get('level'),thermal_variant=tag['variant'],theta=tag['theta'],climate=c,execution_tier='SENSITIVITY',n_cycles=len(cy),n_supported_cycles=n_sup,supported_cycle_fraction=frac,Y_env=ym,Y_env_supported_sum=ys,mean_q_rich=mq,mean_F_R=mf,endpoint_definition='SUPPORTED_CYCLE_MEAN_COMMON_REPRESENTATION_SUPPORT',interpretation='PADAC+13X T_s forcing applied to ImCOF response; not ImCOF bed T prediction' if mode=='padac_13X_controller' else 'PADAC+ImCOF material-specific model composition'))
        old=fc.get('imcof_env_scores',pd.DataFrame()); oldc=fc.get('imcof_env_coverage',pd.DataFrame()); oldt=fc.get('imcof_thermal_summary',pd.DataFrame())
        allr=pd.concat([old,pd.DataFrame(rows)],ignore_index=True,sort=False); allc=pd.concat([oldc,pd.DataFrame(coverage)],ignore_index=True,sort=False); allt=pd.concat([oldt,pd.DataFrame(trows)],ignore_index=True,sort=False)
        fc.save_csv(allr,'ImCOF_environmental_thermal_climate_scores.csv'); fc.save_csv(allc,'ImCOF_environmental_supported_cycle_coverage.csv')
        fc.put('imcof_env_scores',allr); fc.put('imcof_env_coverage',allc); fc.put('imcof_thermal_summary',allt)
        return len(jobs)
    n=fc.guarded(branch,S,_go_sens)
    if n is not None: fc.set_status(branch,'EXECUTED',f'{n} optional thermal sensitivity jobs; appended to primary outputs',S)


In [ ]:
import sys, os, importlib
sys.path.insert(0, "/content")
import numpy as np, pandas as pd
import fidelity_core as fc, fidelity_adapters as ad
importlib.reload(fc); importlib.reload(ad)
CFG=fc.load_config(); S="S22"
# 22. Thermal-controller / model sensitivity table. Primary rows are always present after S21; optional alternatives appear only if S21b was enabled.
if fc.is_ok('PADAC_IMCOF_ENV'):
    T=fc.get('imcof_thermal_summary').copy()
    var=pd.DataFrame(CFG['THERMAL_VARIANTS']).rename(columns={'id':'variant'})
    sens=T.merge(var,on='variant',how='left')
    if CFG.get('IMCOF_ENV_THERMAL_MODE')=='material_specific':
        th=pd.DataFrame(fc.get('imcof_theta'))
        if len(th): sens=sens.merge(th.rename(columns={'id':'theta'}),on='theta',how='left')
    else:
        sens['property_basis']='PADAC_13X_FIELD_REALISATION'
        sens['claim_boundary']='T_s is controller forcing only; no ImCOF thermal properties inferred'
else:
    sens=pd.DataFrame(dict(status=[fc.status_of('PADAC_IMCOF_ENV')],note=['Environmental thermal endpoint not computed.']))
fc.save_csv(sens,'ImCOF_thermal_sensitivity.csv'); sens.head(20)


# PART E — Common process-fidelity and H–Y correspondence analysis


In [ ]:
import sys, os, importlib
sys.path.insert(0, "/content")
import numpy as np, pandas as pd
import fidelity_core as fc, fidelity_adapters as ad
importlib.reload(fc); importlib.reload(ad)
CFG = fc.load_config(); S = "S23"
# 23. IRA representation fidelity, matched on canonical branch + temporal mode.
full=CFG["FULL_REPRESENTATION_NAME"]; fid,pairs,errs=[],[],[]
def run(df,ycol,keys,controller):
    for kv,g in df.groupby(keys):
        kv=kv if isinstance(kv,tuple) else (kv,)
        piv=g.pivot_table(index="climate",columns="representation",values=ycol)
        if full not in piv: continue
        for rep in [r for r in piv.columns if r!=full]:
            sm,pr,ce=fc.fidelity(piv[rep],piv[full],f"{controller}/{kv}/{rep}")
            tag=dict(material=CFG["MATERIALS"]["IRA"],controller=controller,representation=rep,
                     variant="|".join(map(str,kv)),**dict(zip(keys,kv)))
            fid.append({**tag,**sm}); pairs.append(pr.assign(**tag)); errs.append(ce.assign(**tag))
if fc.is_ok("IRA_ENV"):
    run(fc.get("ira_env_scores"),"Y",["branch","temporal_mode"],"environmental_moisture")
if fc.is_ok("IRA_ACTIVE"):
    run(fc.get("ira_active_scores"),"Y_active",["branch","temporal_mode"],"active_moisture")
F=pd.DataFrame(fid)
if len(F):
    # subsample_1h is the native hourly weather representation; both P0 branches are retained rather than choosing one silently.
    F["is_primary"]=F.temporal_mode.eq("subsample_1h")
fc.save_csv(F,"IRA_representation_fidelity.csv")
fc.put("fid_IRA",F); fc.put("pairs_IRA",pd.concat(pairs) if pairs else pd.DataFrame()); fc.put("err_IRA",pd.concat(errs) if errs else pd.DataFrame())
if len(F):
    E=F[F.controller=="environmental_moisture"]; A=F[F.controller=="active_moisture"]
    sens=E.merge(A[["representation","branch","temporal_mode","kendall_tau_b","spearman_rho"]].rename(columns={"kendall_tau_b":"tau_active","spearman_rho":"rho_active"}),
                 on=["representation","branch","temporal_mode"],how="left")
    sens["delta_tau"]=sens.kendall_tau_b-sens.tau_active
    fc.save_csv(sens,"IRA_controller_sensitivity.csv")
else:
    fc.save_csv(pd.DataFrame(dict(status=[fc.status_of("IRA_ENV")])),"IRA_controller_sensitivity.csv")
F.head(20) if len(F) else print("No IRA fidelity computed:",fc.status_of("IRA_ENV"),fc.status_of("IRA_ACTIVE"))


In [ ]:
import sys, os, importlib
sys.path.insert(0, "/content")
import numpy as np, pandas as pd
import fidelity_core as fc, fidelity_adapters as ad
importlib.reload(fc); importlib.reload(ad)
CFG = fc.load_config()
S = "S24"
# 24. ImCOF representation-fidelity analysis (within controller; full = reference).
# Climate eligibility follows the manuscript's supported-domain principle: only climates with finite
# full AND reduced process endpoints are used for that representation comparison. Evaluable-climate count is reported.
import matplotlib.pyplot as plt
full = CFG["FULL_REPRESENTATION_NAME"]; fid, pairs, errs = [], [], []

def run(df, ycol, keys, controller):
    for kv, g in df.groupby(keys):
        kv = kv if isinstance(kv, tuple) else (kv,)
        piv = g.pivot_table(index="climate", columns="representation", values=ycol, dropna=False)
        if full not in piv.columns:
            fc.warn("FULL_REFERENCE_MISSING",
                    f"{controller}/{kv}: full representation {full!r} absent after pivot; available={list(piv.columns)}.", S)
            continue
        for rep in [r for r in piv.columns if r != full]:
            z = piv[[rep, full]].dropna()
            if len(z) < 2:
                fc.warn("INSUFFICIENT_EVALUABLE_CLIMATES",
                        f"{controller}/{kv}/{rep}: only {len(z)} climates have both process endpoints; fidelity not computed.", S)
                continue
            sm, pr, ce = fc.fidelity(z[rep], z[full], f"{controller}/{kv}/{rep}")
            tag = dict(material=CFG["MATERIALS"]["ImCOF"], controller=controller, representation=rep, endpoint=ycol,
                       variant="|".join(map(str, kv)), n_evaluable_climates=len(z),
                       evaluable_climates="|".join(sorted(map(str,z.index))), **dict(zip(keys, kv)))
            fid.append({**tag, **sm})
            pairs.append(pr.assign(**tag))
            errs.append(ce.assign(**tag))

if fc.is_ok("IMCOF_ACTIVE"):
    a = fc.get("imcof_active_scores")
    run(a, "Y_active", ["window", "ads_condition"], "active_thermal_80C")
    run(a, "Y_active_pooled_kernel", ["window", "ads_condition"], "active_thermal_80C_pooled_kernel_sensitivity")
if fc.is_ok("PADAC_IMCOF_ENV"):
    run(fc.get("imcof_env_scores"), "Y_env", ["thermal_variant", "theta"], "environmental_thermal_PADAC")

F = pd.DataFrame(fid)
if len(F):
    F["is_primary"] = ((F.controller == "active_thermal_80C") & (F.get("window") == "primary") & (F.get("ads_condition") == CFG["IMCOF_ADS_CONDITION"])) | \
                      ((F.controller == "environmental_thermal_PADAC") & (F.get("thermal_variant") == "primary"))
fc.save_csv(F, "ImCOF_representation_fidelity.csv")
fc.put("fid_ImCOF", F)
fc.put("pairs_ImCOF", pd.concat(pairs) if pairs else pd.DataFrame())
fc.put("err_ImCOF", pd.concat(errs) if errs else pd.DataFrame())

if len(F) and fc.is_ok("PADAC_IMCOF_ENV"):
    theta_df=pd.DataFrame(fc.get("imcof_theta"))
    if len(theta_df):
        env = F[F.controller == "environmental_thermal_PADAC"].merge(theta_df.rename(columns={"id": "theta"}), on="theta")
        pcols = [c for c in ("C_s", "rho_bulk", "k_s", "d_p", "eps_bed") if c in env.columns and env[c].nunique() > 1]
        if pcols:
            fig, axs = plt.subplots(1, len(pcols), figsize=(3.2 * len(pcols), 3), squeeze=False)
            for ax, pc in zip(axs[0], pcols):
                for rep, g in env[env.thermal_variant == "primary"].groupby("representation"):
                    ax.plot(g[pc], g.kendall_tau_b, "o", label=rep)
                ax.set_xlabel(pc); ax.set_ylabel("Kendall tau (env, primary variant)")
            axs[0][0].legend(fontsize=7); fig.tight_layout()
            fig.savefig(os.path.join(fc.run_dir(), "figures", "fig5_tau_vs_thermal_theta.png"), dpi=150)

F.head(20) if len(F) else print("No ImCOF fidelity computed:", fc.status_of("IMCOF_ACTIVE"), fc.status_of("PADAC_IMCOF_ENV"))


In [ ]:
import sys, os, importlib
sys.path.insert(0, "/content")
import numpy as np, pandas as pd
import fidelity_core as fc, fidelity_adapters as ad
importlib.reload(fc); importlib.reload(ad)
CFG = fc.load_config()
S = "S25"
# 25. Cross-controller comparison.
# Delta-tau is computed only on an identical climate set for the two controllers.
# For ImCOF, if controller-specific evaluable sets differ, both fidelities are recomputed on their exact intersection.
# Absolute Y_env and Y_active are never compared.
import matplotlib.pyplot as plt

rows=[]

def _fid_on_common(raw_env, raw_act, y_env, y_act, rep, full, env_filter, act_filter, label):
    E=raw_env.copy(); A=raw_act.copy()
    for k,v in env_filter.items(): E=E[E[k].astype(str)==str(v)]
    for k,v in act_filter.items(): A=A[A[k].astype(str)==str(v)]
    ep=E.pivot_table(index='climate',columns='representation',values=y_env,dropna=False)
    ap=A.pivot_table(index='climate',columns='representation',values=y_act,dropna=False)
    if rep not in ep or full not in ep or rep not in ap or full not in ap:
        return None
    common=ep[[rep,full]].dropna().index.intersection(ap[[rep,full]].dropna().index)
    if len(common)<2: return None
    se,_,_=fc.fidelity(ep.loc[common,rep],ep.loc[common,full],label+'/env')
    sa,_,_=fc.fidelity(ap.loc[common,rep],ap.loc[common,full],label+'/active')
    return se,sa,common

# IRA: recompute both controllers on the exact branch + temporal_mode + climate intersection.
Fira=fc.get('fid_IRA',pd.DataFrame())
if len(Fira) and fc.is_ok('IRA_ENV') and fc.is_ok('IRA_ACTIVE'):
    env_raw=fc.get('ira_env_scores'); act_raw=fc.get('ira_active_scores'); full=CFG['FULL_REPRESENTATION_NAME']
    E=Fira[Fira.controller=='environmental_moisture']; A=Fira[Fira.controller=='active_moisture']
    for _,e in E.iterrows():
        cand=A[(A.representation==e.representation) & (A.variant.astype(str)==str(e.variant))]
        for _,a in cand.iterrows():
            branch_name=str(e.variant).split('|',1)[0]; tm=str(e.variant).split('|',1)[1]
            z=_fid_on_common(env_raw,act_raw,'Y','Y_active',e.representation,full,
                             {'branch':branch_name,'temporal_mode':tm},{'branch':branch_name,'temporal_mode':tm},
                             f"IRA/{e.variant}/{e.representation}")
            if z is None:
                fc.warn('CROSS_CONTROLLER_COMMON_CLIMATE_INSUFFICIENT',
                        f"IRA {e.variant}, rep={e.representation}: <2 common evaluable climates; pair skipped.",S)
                continue
            se,sa,common=z
            rows.append(dict(material=CFG['MATERIALS']['IRA'],representation=e.representation,
                env_variant=e.variant,active_variant=a.variant,env_primary=bool(e.is_primary),active_primary=bool(a.is_primary),
                tau_env=se['kendall_tau_b'],tau_active=sa['kendall_tau_b'],delta_tau=se['kendall_tau_b']-sa['kendall_tau_b'],
                rho_env=se['spearman_rho'],rho_active=sa['spearman_rho'],delta_rho=se['spearman_rho']-sa['spearman_rho'],
                reversal_frac_env=se['reversal_fraction'],reversal_frac_active=sa['reversal_fraction'],
                n_rev_env=se['n_strict_reversals'],n_rev_active=sa['n_strict_reversals'],
                n_common_climates=len(common),common_climates='|'.join(sorted(map(str,common))),
                common_climate_rule='EXACT_ENV_ACTIVE_INTERSECTION_RECOMPUTED'))

# ImCOF: recompute on exact controller-common climate intersection.
Fim=fc.get('fid_ImCOF',pd.DataFrame())
if len(Fim) and fc.is_ok('IMCOF_ACTIVE') and fc.is_ok('PADAC_IMCOF_ENV'):
    env_raw=fc.get('imcof_env_scores'); act_raw=fc.get('imcof_active_scores')
    full=CFG['FULL_REPRESENTATION_NAME']
    E=Fim[Fim.controller=='environmental_thermal_PADAC']
    A=Fim[Fim.controller=='active_thermal_80C']
    for _,e in E.iterrows():
        # Primary active variant(s) and other active variants remain explicit; no averaging across them.
        for _,a in A[A.representation==e.representation].iterrows():
            env_filter={'thermal_variant':e.thermal_variant,'theta':e.theta}
            act_filter={'window':a.window,'ads_condition':a.ads_condition}
            z=_fid_on_common(env_raw,act_raw,'Y_env','Y_active',e.representation,full,env_filter,act_filter,
                             f"ImCOF/{e.variant}/{a.variant}/{e.representation}")
            if z is None:
                fc.warn('CROSS_CONTROLLER_COMMON_CLIMATE_INSUFFICIENT',
                        f"ImCOF {e.variant} vs {a.variant}, rep={e.representation}: <2 common evaluable climates; pair skipped.",S)
                continue
            se,sa,common=z
            rows.append(dict(material=CFG['MATERIALS']['ImCOF'],representation=e.representation,
                env_variant=e.variant,active_variant=a.variant,env_primary=bool(e.is_primary),active_primary=bool(a.is_primary),
                tau_env=se['kendall_tau_b'],tau_active=sa['kendall_tau_b'],delta_tau=se['kendall_tau_b']-sa['kendall_tau_b'],
                rho_env=se['spearman_rho'],rho_active=sa['spearman_rho'],delta_rho=se['spearman_rho']-sa['spearman_rho'],
                reversal_frac_env=se['reversal_fraction'],reversal_frac_active=sa['reversal_fraction'],
                n_rev_env=se['n_strict_reversals'],n_rev_active=sa['n_strict_reversals'],
                n_common_climates=len(common),common_climates='|'.join(sorted(map(str,common))),
                common_climate_rule='EXACT_ENV_ACTIVE_INTERSECTION_RECOMPUTED'))

P=pd.DataFrame(rows)
fc.save_csv(P,'paired_controller_fidelity_comparison.csv'); fc.put('paired',P)

figdir=os.path.join(fc.run_dir(),'figures'); os.makedirs(figdir,exist_ok=True)
# Controller-specific fidelity figures use their own supported/evaluable climate sets; paired Delta-tau above uses exact intersections.
spec=dict(IRA=('fid_IRA','environmental_moisture','active_moisture'),ImCOF=('fid_ImCOF','environmental_thermal_PADAC','active_thermal_80C'))
for mat,(key,env_c,act_c),fname in [('IRA',spec['IRA'],'fig3_IRA_fidelity.png'),('ImCOF',spec['ImCOF'],'fig4_ImCOF_fidelity.png')]:
    F=fc.get(key,pd.DataFrame())
    if not len(F) or not {env_c,act_c}<=set(F.controller): continue
    reps=sorted(F.representation.unique()); x=np.arange(len(reps)); fig,ax=plt.subplots(figsize=(1.2*len(reps)+3,3.5))
    for off,cname,mk in [(-0.15,env_c,'o'),(0.15,act_c,'s')]:
        g=F[F.controller==cname]
        for i,r in enumerate(reps):
            v=g[g.representation==r].kendall_tau_b
            ax.plot(np.full(len(v),i+off),v,mk,alpha=.35,color='C0' if mk=='o' else 'C1')
            vp=g[(g.representation==r)&g.is_primary].kendall_tau_b
            ax.plot(np.full(len(vp),i+off),vp,mk,ms=9,mfc='none',color='k')
    ax.set_xticks(x,reps,rotation=30,ha='right'); ax.set_ylabel('Kendall tau vs full')
    ax.set_title(f"{CFG['MATERIALS'][mat]}: {env_c} vs {act_c}; black = primary",fontsize=8)
    fig.tight_layout(); fig.savefig(os.path.join(figdir,fname),dpi=150)

# Fig 6: raw supported-cycle-mean Y, full vs reduced, primary variants only.
src=[]
if fc.is_ok('IRA_ENV'):
    d=fc.get('ira_env_scores'); src.append(('IRA env (continuous-weather loading release)',d[d.variant=='continuous_v7_no_RH_threshold'],'Y'))
if fc.is_ok('IRA_ACTIVE'):
    d=fc.get('ira_active_scores'); src.append(('IRA active',d[d.regen_kernel=='s9'],'Y_active'))
if fc.is_ok('IMCOF_ACTIVE'):
    d=fc.get('imcof_active_scores'); src.append(('ImCOF active 80 C',d[(d.window=='primary')&(d.ads_condition==CFG['IMCOF_ADS_CONDITION'])],'Y_active'))
if fc.is_ok('PADAC_IMCOF_ENV'):
    d=fc.get('imcof_env_scores'); theta0=d['theta'].dropna().iloc[0] if len(d['theta'].dropna()) else None
    sel=d[d.thermal_variant=='primary']; sel=sel[sel.theta==theta0] if theta0 is not None else sel
    src.append(('ImCOF env PADAC (model composition)',sel,'Y_env'))
if src:
    fig,axs=plt.subplots(1,len(src),figsize=(3.6*len(src),3.4),squeeze=False); plotted=0
    for ax,(lab,d,y) in zip(axs[0],src):
        piv=d.pivot_table(index='climate',columns='representation',values=y,dropna=False); full=CFG['FULL_REPRESENTATION_NAME']
        if full not in piv.columns:
            ax.set_axis_off(); ax.set_title(lab+'\n(full reference unavailable)',fontsize=8); continue
        reduced=[r for r in piv.columns if r!=full]
        for r in reduced:
            z=piv[[full,r]].dropna()
            if z.empty: continue
            ax.plot(z[full],z[r],'.',label=r); plotted+=1
        finite=piv.replace([np.inf,-np.inf],np.nan); lo,hi=finite.min().min(),finite.max().max()
        if np.isfinite(lo) and np.isfinite(hi): ax.plot([lo,hi],[lo,hi],'k-',lw=.5)
        ax.set_title(lab,fontsize=8); ax.set_xlabel('Y (full)'); ax.set_ylabel('Y (reduced)')
    if plotted:
        for ax in axs[0]:
            h,l=ax.get_legend_handles_labels()
            if h: ax.legend(fontsize=6); break
    fig.tight_layout(); fig.savefig(os.path.join(figdir,'fig6_full_vs_reduced_Y.png'),dpi=150)

P[P.env_primary & P.active_primary] if len(P) else print('No paired comparison executable.')


## Existing equilibrium descriptor H versus process outcome Y
Only exact climate/key-aligned comparisons are evaluated. No H/Y magnitude comparison is performed; ImCOF is not forced into this bridge because its existing H and current Y have incompatible branch granularity.


In [ ]:
import sys, os, importlib, itertools
sys.path.insert(0, "/content")
import numpy as np, pandas as pd
from scipy.stats import kendalltau, spearmanr
import fidelity_core as fc, fidelity_adapters as ad
importlib.reload(fc); importlib.reload(ad)
CFG = fc.load_config(); S = "S26"
# 26. Existing H -> process Y climate-ordering bridge.
# H and Y have different physical meanings/units; ONLY rank correspondence and pairwise ordering agreement are evaluated.
# No magnitude ratio, regression slope, or productivity interpretation is admissible.
# Existing P0 H is branch x temporal_mode x city. Therefore a direct bridge is evaluated only when process Y has the
# same keys. IRA satisfies this contract. Current ImCOF Y is whole-cycle and has no P0 branch key, so existing-H vs
# ImCOF-Y is deliberately NOT constructed by summing/averaging branches post hoc.
R = fc.run_dir()
reg_path = os.path.join(R, "equilibrium_representation_regression.csv")

# Prefer the in-run state written by S6.  This preserves the native boolean dtype of
# the regression gate.  CSV is only a restart/fallback path because mixed bool/NaN
# columns (P0 regression + anchor QA rows) can round-trip through CSV as strings.
H0 = fc.get("H_baseline_regression", pd.DataFrame()).copy()
h_source = "state:H_baseline_regression"
if H0.empty:
    if not os.path.exists(reg_path):
        raise fc.HardStop("H_BASELINE_MISSING: neither H_baseline_regression state nor equilibrium_representation_regression.csv found")
    H0 = pd.read_csv(reg_path)
    h_source = "csv:equilibrium_representation_regression.csv"

if "check" not in H0.columns or "pass" not in H0.columns:
    raise fc.HardStop(f"H_BASELINE_SCHEMA: source={h_source}; columns={list(H0.columns)}")

H0 = H0[H0["check"].astype(str).eq("P0_regression")].copy()
# Robust bool coercion for the CSV fallback; no failed regression row is promoted.
p = H0["pass"]
if pd.api.types.is_bool_dtype(p):
    passed = p.fillna(False)
else:
    passed = p.astype(str).str.strip().str.lower().isin({"true", "1", "1.0"})

if not passed.any():
    diag = (H0.assign(_passed=passed)
              .groupby("material", dropna=False)
              .agg(n_rows=("key", "size"), n_pass=("_passed", "sum"), max_abs_err=("abs_err", "max"))
              .reset_index()) if len(H0) else pd.DataFrame()
    print("H baseline source:", h_source)
    if len(diag):
        display(diag)
    raise fc.HardStop("H_BASELINE_EMPTY: P0 regression rows exist but none passed; inspect S6/equilibrium_representation_regression.csv")

H0 = H0.loc[passed].copy()
print(f"H baseline: {len(H0)} passed P0 regression rows from {h_source}")
parts = H0["key"].astype(str).str.split("|", expand=True)
if parts.shape[1] != 3:
    raise fc.HardStop("H_KEY_SCHEMA: expected branch|temporal_mode|city")
H0[["branch","temporal_mode","climate"]] = parts
H0 = H0.rename(columns={"rep":"representation", "reference":"H"})
H0 = H0[["material","representation","branch","temporal_mode","climate","H"]].copy()
if H0.duplicated(["material","representation","branch","temporal_mode","climate"]).any():
    raise fc.HardStop("H_KEY_DUPLICATE: canonical H baseline contains duplicate keys")

def correspondence(h, y, label):
    h=pd.Series(h,dtype=float); y=pd.Series(y,dtype=float)
    if set(h.index)!=set(y.index):
        raise fc.HardStop(f"H_Y_CLIMATE_MISMATCH: {label}")
    y=y.loc[h.index]
    if h.isna().any() or y.isna().any():
        raise fc.HardStop(f"H_Y_NAN: {label}")
    tau,pt=kendalltau(h.values,y.values); rho,pr=spearmanr(h.values,y.values)
    rows=[]; cl=list(h.index)
    for i,j in itertools.combinations(range(len(cl)),2):
        sh=np.sign(h.iloc[i]-h.iloc[j]); sy=np.sign(y.iloc[i]-y.iloc[j])
        if sh!=sy:
            rows.append(dict(climate_i=cl[i],climate_j=cl[j],sign_H=int(sh),sign_Y=int(sy),
                             kind="STRICT_REVERSAL" if sh*sy<0 else "TIE_CHANGE",
                             H_i=h.iloc[i],H_j=h.iloc[j],Y_i=y.iloc[i],Y_j=y.iloc[j]))
    npairs=len(cl)*(len(cl)-1)//2; strict=sum(x["kind"]=="STRICT_REVERSAL" for x in rows)
    return dict(n_climates=len(cl),kendall_tau_HY=tau,kendall_p_HY=pt,spearman_rho_HY=rho,spearman_p_HY=pr,
                n_pairs=npairs,n_strict_HY_reversals=strict,n_HY_tie_changes=len(rows)-strict,
                HY_reversal_fraction=(strict/npairs if npairs else np.nan)), pd.DataFrame(rows)

summary=[]; disagreements=[]; climate_rows=[]

# IRA environmental and active scores share branch/temporal_mode with canonical H.
# H-Y correspondence is evaluated only on climates that are evaluable for BOTH canonical H and process Y.
# Process rows lacking a canonical supported-transition H are excluded explicitly, not filled or extrapolated.
ira_sources=[]
if fc.is_ok("IRA_ENV"):
    ira_sources.append(("environmental_moisture",fc.get("ira_env_scores"),"Y"))
if fc.is_ok("IRA_ACTIVE"):
    ira_sources.append(("active_moisture",fc.get("ira_active_scores"),"Y_active"))
alignment_audit=[]
for controller,df,ycol in ira_sources:
    need={"representation","branch","temporal_mode","climate",ycol}
    if not need<=set(df.columns):
        raise fc.HardStop(f"H_Y_IRA_SCHEMA: {controller} missing {sorted(need-set(df.columns))}")
    Hb=H0[H0.material.astype(str)==str(CFG["MATERIALS"]["IRA"])].copy()
    exact_modes=set(Hb.temporal_mode.astype(str).unique())
    d0=df[df.temporal_mode.astype(str).isin(exact_modes)].copy()
    if d0.empty:
        continue
    keys=["representation","branch","temporal_mode","climate"]
    chk=d0.merge(Hb[keys],on=keys,how="left",indicator=True,validate="many_to_one")
    unmatched=chk[chk["_merge"]=="left_only"].copy()
    if len(unmatched):
        for _,r in unmatched.iterrows():
            alignment_audit.append(dict(controller=controller,representation=r["representation"],branch=r["branch"],
                temporal_mode=r["temporal_mode"],climate=r["climate"],status="PROCESS_ROW_EXCLUDED_NO_CANONICAL_H",
                reason="Climate is not evaluable for canonical H under the measured supported-transition domain."))
        fc.warn("H_Y_PROCESS_ONLY_KEYS_EXCLUDED",
                f"{controller}: excluded {len(unmatched)} process rows without canonical H support; matched {len(d0)-len(unmatched)} of {len(d0)}.",S)
    z=d0.merge(Hb,on=keys,how="inner",validate="many_to_one")
    if z.empty:
        fc.warn("H_Y_NO_COMMON_EVALUABLE_CLIMATES",f"{controller}: no process rows share the canonical H domain.",S)
        continue
    for (rep,branch,tm),g in z.groupby(["representation","branch","temporal_mode"],sort=False):
        finite=np.isfinite(pd.to_numeric(g["H"],errors="coerce")) & np.isfinite(pd.to_numeric(g[ycol],errors="coerce"))
        gg0=g.loc[finite].copy()
        n_h_evaluable=len(g); n_joint_evaluable=len(gg0)
        if n_joint_evaluable<2:
            fc.warn("H_Y_TOO_FEW_COMMON_CLIMATES",f"{controller}/{rep}/{branch}/{tm}: {n_joint_evaluable} jointly evaluable climates; skipped.",S)
            continue
        h=gg0.set_index("climate")["H"]; y=gg0.set_index("climate")[ycol]
        sm,pr=correspondence(h,y,f"{controller}/{rep}/{branch}/{tm}")
        primary=(tm=="subsample_1h")
        tag=dict(material=CFG["MATERIALS"]["IRA"],controller=controller,representation=rep,branch=branch,
                 temporal_mode=tm,variant=f"{branch}|{tm}",is_primary=primary,
                 n_H_evaluable_climates=n_h_evaluable,n_joint_evaluable_climates=n_joint_evaluable)
        summary.append({**tag,**sm})
        if len(pr): disagreements.append(pr.assign(**tag))
        gg=gg0[["climate","H",ycol]].rename(columns={ycol:"Y"}).copy()
        climate_rows.append(gg.assign(**tag))
HA=pd.DataFrame(alignment_audit)
fc.save_csv(HA,"H_Y_alignment_exclusions.csv")

HY=pd.DataFrame(summary)
HD=pd.concat(disagreements,ignore_index=True) if disagreements else pd.DataFrame(columns=["material","controller","representation","branch","temporal_mode","variant","climate_i","climate_j","kind"])
HC=pd.concat(climate_rows,ignore_index=True) if climate_rows else pd.DataFrame(columns=["material","controller","representation","branch","temporal_mode","variant","climate","H","Y"])
fc.save_csv(HY,"H_Y_climate_ordering_bridge.csv")
fc.save_csv(HD,"H_Y_pairwise_disagreements.csv")
fc.save_csv(HC,"H_Y_climate_scores_aligned.csv")
fc.put("HY_bridge",HY); fc.put("HY_pairs",HD); fc.put("HY_climate",HC)
if len(HY):
    fc.set_status("H_Y_BRIDGE_IRA","EXECUTED",f"{len(HY)} exact-key IRA H-Y correspondence variants; no magnitude comparison",S)
else:
    fc.not_executed("H_Y_BRIDGE_IRA","UPSTREAM_INCOMPLETE","IRA process branches unavailable",S)
fc.not_executed("H_Y_BRIDGE_ImCOF","EXISTING_H_PROCESS_KEY_MISMATCH",
    "Existing P0 H is branch-specific while current ImCOF Y is whole-cycle; no branch aggregation is introduced post hoc.",S)
HY


In [ ]:
import sys, os, importlib
sys.path.insert(0, "/content")
import numpy as np, pandas as pd
import fidelity_core as fc, fidelity_adapters as ad
importlib.reload(fc); importlib.reload(ad)
CFG = fc.load_config()
S = "S27"
# 27. Pairwise reversal decomposition (every reversed pair stored individually)
import matplotlib.pyplot as plt
pr = pd.concat([fc.get("pairs_IRA", pd.DataFrame()), fc.get("pairs_ImCOF", pd.DataFrame())], ignore_index=True)
er = pd.concat([fc.get("err_IRA", pd.DataFrame()), fc.get("err_ImCOF", pd.DataFrame())], ignore_index=True)
fc.save_csv(pr if len(pr) else pd.DataFrame(columns=["material", "controller", "representation", "variant", "climate_i", "climate_j", "kind"]),
            "pairwise_climate_reversals.csv")
fc.save_csv(er if len(er) else pd.DataFrame(columns=["material", "controller", "representation", "variant", "climate", "Y_full", "Y_rep"]),
            "climate_level_error_decomposition.csv")
strict = pr[pr.kind == "STRICT_REVERSAL"] if len(pr) else pr
if len(strict):   # Fig 7 only when reversals actually occur: counts per climate pair, primary variants only
    fids = pd.concat([fc.get("fid_IRA", pd.DataFrame()), fc.get("fid_ImCOF", pd.DataFrame())])
    prim = fids[fids.is_primary][["material", "controller", "variant"]].drop_duplicates()
    sp = strict.merge(prim, on=["material", "controller", "variant"])
    if len(sp):
        sp = sp.assign(pair=sp.climate_i.astype(str) + " / " + sp.climate_j.astype(str))
        tab = sp.groupby(["pair", "controller"]).size().unstack(fill_value=0)
        tab = tab.loc[tab.sum(1).sort_values(ascending=False).index[:40]]
        fig, ax = plt.subplots(figsize=(7, 0.22 * len(tab) + 1.5)); tab.plot.barh(ax=ax, stacked=False)
        ax.set_xlabel("number of reduced representations reversing the pair"); ax.invert_yaxis(); fig.tight_layout()
        fig.savefig(os.path.join(fc.run_dir(), "figures", "fig7_pairwise_reversals.png"), dpi=150)
print(f"{len(strict)} strict reversals across all controllers/variants/representations")


In [ ]:
import sys, os, importlib
sys.path.insert(0, "/content")
import numpy as np, pandas as pd
import fidelity_core as fc, fidelity_adapters as ad
importlib.reload(fc); importlib.reload(ad)
CFG = fc.load_config()
S = "S28"
# 28. Save all results (general manifests + placeholders recording why any required output is absent)
R = fc.run_dir()
el = pd.DataFrame(fc._jsonl_read("eligibility.jsonl"))
latest = el.groupby("branch").tail(1) if len(el) else el
fc.save_csv(latest, "analysis_eligibility.csv"); fc.save_csv(el, "analysis_eligibility_history.csv")
fc.save_csv(pd.DataFrame(fc._jsonl_read("assumptions.jsonl")), "controller_assumption_manifest.csv")
fc.save_csv(pd.DataFrame(fc._jsonl_read("warnings.jsonl")), "analysis_warnings.csv")
fc.save_csv(pd.DataFrame(fc._jsonl_read("inputs.jsonl")), "input_manifest.csv")
owner = {"equilibrium_representation_regression.csv": "EQUILIBRIUM_P0_IRA", "IRA_environmental_loading_climate_scores.csv": "IRA_ENV",
         "IRA_active_cycle_climate_scores.csv": "IRA_ACTIVE", "IRA_controller_sensitivity.csv": "IRA_ENV", "IRA_representation_fidelity.csv": "IRA_ENV",
         "PADAC_equation_manifest.csv": "PADAC_EQUATIONS", "PADAC_parameter_provenance.csv": "PADAC_EQUATIONS",
         "PADAC_reproduction_validation.csv": "PADAC_REPRODUCTION", "PADAC_weather_forcing_eligibility.csv": "PADAC_GLOBAL_FORCING",
         "PADAC_solver_optimization_validation.csv": "PADAC_SOLVER_OPTIMIZATION", "PADAC_solver_optimization_manifest.csv": "PADAC_SOLVER_OPTIMIZATION",
         "PADAC_thermal_job_dedup_manifest.csv": "PADAC_IMCOF_ENV",
         "ImCOF_thermal_property_audit.csv": "IMCOF_THERMAL_PROPS", "ImCOF_TPD_kernel.csv": "IMCOF_KERNEL", "ImCOF_TPD_QA.csv": "IMCOF_KERNEL",
         "ImCOF_active_thermal_climate_scores.csv": "IMCOF_ACTIVE", "ImCOF_active_supported_cycle_coverage.csv": "IMCOF_ACTIVE", "ImCOF_environmental_thermal_climate_scores.csv": "PADAC_IMCOF_ENV", "ImCOF_environmental_supported_cycle_coverage.csv": "PADAC_IMCOF_ENV",
         "ImCOF_thermal_sensitivity.csv": "PADAC_IMCOF_ENV", "ImCOF_representation_fidelity.csv": "IMCOF_ACTIVE",
         "paired_controller_fidelity_comparison.csv": "IRA_ENV", "pairwise_climate_reversals.csv": "IRA_ENV",
         "climate_level_error_decomposition.csv": "IRA_ENV",
         "H_Y_climate_ordering_bridge.csv": "H_Y_BRIDGE_IRA", "H_Y_pairwise_disagreements.csv": "H_Y_BRIDGE_IRA",
         "H_Y_climate_scores_aligned.csv": "H_Y_BRIDGE_IRA"}
for f, b in owner.items():
    if not os.path.exists(os.path.join(R, f)):
        st = fc.statuses().get(b, {})
        pd.DataFrame(dict(status=[st.get("status", "NOT_RUN")], reason=[st.get("reason", "")])).to_csv(os.path.join(R, f), index=False)
        print("placeholder:", f, st.get("status", "NOT_RUN"))
latest[["branch", "status", "reason"]]


In [ ]:
import sys, os, importlib
sys.path.insert(0, "/content")
import numpy as np, pandas as pd
import fidelity_core as fc, fidelity_adapters as ad
importlib.reload(fc); importlib.reload(ad)
CFG = fc.load_config()
S = "S29"
# 29. Execution summary (answers spec §45 from saved state only)
R = fc.run_dir(); st = fc.statuses(); L = []
ok = lambda b: fc.is_ok(b)
def tbl(df, cols):
    return "_none_" if df is None or not len(df) else df[cols].to_markdown(index=False, floatfmt=".3f")
L += ["# Analysis execution summary", "", f"Run directory: `{R}`  ", f"Config frozen at: {CFG['FROZEN_AT']}", "",
      "> **Interpretation constraint.** H = equilibrium-side environmental opportunity descriptor; Y = controller/process-dependent "
      "dynamic outcome; H is not productivity. IRA environmentally induced loading release is **not** recovered CO₂. "
      "PADAC is field-tested with Zeolite 13X; ImCOF adsorption and TPD were measured independently; their coupling here is a "
      "**literature-constrained model composition**, not an experimentally validated ImCOF reactor.", ""]
exe = sorted(b for b, r in st.items() if r["status"].startswith(fc.OK_PREFIXES))
nex = sorted(b for b in st if b not in exe)
L += ["## 1. Branches executed", ", ".join(f"`{b}` ({st[b]['status']})" for b in exe) or "_none_", "",
      "## 2. Branches not executed", "\n".join(f"- `{b}`: {st[b]['status']} — {st[b]['reason']}" for b in nex) or "_none_", ""]
man = pd.read_csv(os.path.join(R, "input_manifest.csv")) if os.path.exists(os.path.join(R, "input_manifest.csv")) else pd.DataFrame()
asm = pd.DataFrame(fc._jsonl_read("assumptions.jsonl"))
L += ["## 3. Provenance of inputs", tbl(man.groupby("provenance").size().rename("n_files").reset_index() if len(man) else None, ["provenance", "n_files"]),
      "", "Assumed / calculated controller parameters:", tbl(asm.groupby("provenance").size().rename("n").reset_index() if len(asm) else None, ["provenance", "n"]),
      "PADAC parameter classes: see `PADAC_parameter_provenance.csv`.", ""]
L += [f"## 4. P0 equilibrium reproduced? IRA: {st.get('EQUILIBRIUM_P0_IRA', {}).get('status', 'NOT_RUN')}; ImCOF: {st.get('EQUILIBRIUM_P0_ImCOF', {}).get('status', 'NOT_RUN')}",
      f"## 5. Active-process bridge reproduced? {st.get('ACTIVE_BRIDGE_REGRESSION', {}).get('status', 'NOT_RUN')}",
      f"## 6. PADAC+13X reproduced? {st.get('PADAC_REPRODUCTION', {}).get('status', 'NOT_RUN')} (equation transcription: {st.get('PADAC_EQUATIONS', {}).get('status', 'NOT_RUN')})",
      f"## 7. Exact PADAC field forcing available? {'yes' if CFG['PADAC_FIELD_FORCING_PATH'] and not CFG['PADAC_FIELD_SAME_FIGURE'] else 'no'}"]
aud = pd.read_csv(os.path.join(R, "ImCOF_thermal_property_audit.csv")) if os.path.exists(os.path.join(R, "ImCOF_thermal_property_audit.csv")) else pd.DataFrame()
if "classification" in aud:
    L += ["## 8. ImCOF thermal properties directly available", ", ".join(aud[aud.classification.isin(["DIRECT_IMCOF", "DERIVABLE_IMCOF"])].property) or "_none_",
          "## 9. Unresolved", ", ".join(aud[aud.resolution == "UNRESOLVED_SENSITIVITY_PARAMETER"].property) or "_none_"]
L += [f"## 10. 36-climate G, v forcing available? {st.get('PADAC_GLOBAL_FORCING', {}).get('status', 'NOT_RUN')}", f"PADAC thermal solver optimization: {st.get('PADAC_SOLVER_OPTIMIZATION', {}).get('status', 'NOT_RUN')} (selected engine: {fc.get('padac_solver_engine','legacy_segmented')}). All canonical hourly forcing/output timestamps are retained; failed optimization validation falls back to the legacy segmented solver.", f"Primary ImCOF environmental thermal mode: `{CFG.get('IMCOF_ENV_THERMAL_MODE')}`. In `padac_13X_controller` mode, T_s is an exogenous PADAC+13X controller forcing and is not an ImCOF bed-temperature prediction.", ""]
F = pd.concat([fc.get("fid_IRA", pd.DataFrame()), fc.get("fid_ImCOF", pd.DataFrame())], ignore_index=True)
cols = ["material", "controller", "variant", "representation", "kendall_tau_b", "spearman_rho", "n_strict_reversals", "reversal_fraction"]
prim = F[F.is_primary] if len(F) else F
L += ["## 11–13. Controller-specific Kendall τ, Spearman ρ, reversals (primary variants; all variants in the fidelity CSVs)",
      "IRA environmental branch uses the canonical continuous hourly RH/T sequence with no ambient RH switching thresholds. "
      "ImCOF Y uses a common single/tensor supported-cycle mean; zero-supported climates are non-evaluable and supported-cycle coverage is reported separately. NaN tau = constant Y across evaluable climates (degenerate_constant_Y).", tbl(prim, cols), ""]
pr = pd.read_csv(os.path.join(R, "pairwise_climate_reversals.csv"))
sp = pr[pr.get("kind", pd.Series(dtype=str)) == "STRICT_REVERSAL"] if "kind" in pr else pr.iloc[0:0]
if len(sp) and len(prim):
    sp = sp.merge(prim[["material", "controller", "variant"]].drop_duplicates(), on=["material", "controller", "variant"])
L += ["## 14. Climate pairs causing reversals (primary variants)", tbl(sp, ["material", "controller", "variant", "representation", "climate_i", "climate_j"]) if len(sp) else "No strict reversals in primary variants.", ""]
P = fc.get("paired", pd.DataFrame())
L += ["## 15. Did controller choice change representation fidelity?"]
if len(P):
    pp = P[P.env_primary & P.active_primary]
    L += [tbl(pp, ["material", "representation", "tau_env", "tau_active", "delta_tau", "delta_rho"]),
          "Full distribution over admissible environmental controllers/thermal realisations:",
          tbl(P.groupby(["material", "representation"]).delta_tau.agg(["min", "median", "max", "count"]).reset_index(), ["material", "representation", "min", "median", "max", "count"]),
          "No significance test is applied; with 36 climates the τ differences are descriptive."]
else:
    L += ["Not evaluable: a paired comparison requires both controllers of a material to execute."]
L += ["", "## 16. Did thermal-property uncertainty change that conclusion?"]
if len(P) and (P.material == CFG["MATERIALS"]["ImCOF"]).any():
    q = P[(P.material == CFG["MATERIALS"]["ImCOF"]) & P.active_primary]
    flips = q.groupby("representation").delta_tau.apply(lambda s: (np.sign(s).nunique() > 1))
    L += [f"Sign of Δτ varies across (variant, θ) for representations: {list(flips[flips].index) or 'none'}"]
else:
    L += ["Not evaluable (PADAC+ImCOF environmental branch not executed)."]
L += ["", "## 17. Did IRA and ImCOF support the same conclusion?"]
both = len(P) and P.material.nunique() == 2
L += ["Compare the Δτ tables above by representation level." if both else "Not evaluable: only one (or neither) material has a complete paired comparison."]
HY = fc.get("HY_bridge", pd.DataFrame())
L += ["", "## 18. Existing H to process-Y climate-ordering correspondence"]
if len(HY):
    hp = HY[HY.is_primary] if "is_primary" in HY else HY
    L += [tbl(hp, ["material","controller","representation","branch","temporal_mode","kendall_tau_HY","spearman_rho_HY","n_strict_HY_reversals","HY_reversal_fraction"]),
          "This is rank correspondence only. H and Y have different physical meanings and units; magnitudes are not compared."]
else:
    L += ["Not evaluable for an exact-key branch."]
L += [f"ImCOF existing-H bridge: {st.get('H_Y_BRIDGE_ImCOF', {}).get('status', 'NOT_RUN')} — current process Y is whole-cycle while existing P0 H is branch-specific; no post-hoc branch aggregation is used."]
L += ["", "## 19. Strongest supported claim",
      ("Within the executed branches only: the tabulated τ/ρ/reversal values describe how much climate ordering of each "
       "controller-specific Y is preserved under each P0 representation reduction, for the frozen climate ensemble and controller families. "
       "Statements about controller dependence are limited to materials where both controllers executed.") if len(F) else
      "No fidelity endpoint was computed; the executed evidence supports only the input/provenance/gate findings listed above.",
      "", "## 20. Claims that cannot be made",
      "- Any absolute productivity or env-vs-active performance ranking (absolute Y are never compared).",
      "- That IRA environmental loading release is recovered CO₂ or productivity.",
      "- That PADAC+ImCOF is an experimentally validated reactor, or that its T_s(t) is exact (PADAC MRE ≈ 9–20% with 13X; ImCOF thermal properties "
      + ("unresolved and treated as sensitivity parameters)." if (len(aud) and (aud.get("resolution") == "UNRESOLVED_SENSITIVITY_PARAMETER").any()) else "as audited).")]
if not ok("PADAC_GLOBAL_FORCING"):
    L += ["- Any 36-climate PADAC ranking (PADAC_GLOBAL_FORCING_INCOMPLETE)."]
if nex:
    L += ["- Conclusions about branches listed under §2 as not executed."]
L += ["", "_Novelty, importance and publication potential are not assessed._"]
txt = "\n".join(L)
open(os.path.join(R, "analysis_execution_summary.md"), "w").write(txt)
from IPython.display import Markdown; Markdown(txt)


In [ ]:
import sys, os, importlib
sys.path.insert(0, "/content")
import numpy as np, pandas as pd
import fidelity_core as fc, fidelity_adapters as ad
importlib.reload(fc); importlib.reload(ad)
CFG = fc.load_config()
S = "S30"
# 30. Final claim-boundary audit (mechanical checks on saved outputs)
import re
R = fc.run_dir(); checks = []
def chk(name, ok, detail=""):
    checks.append(dict(check=name, passed=bool(ok), detail=detail))
f = os.path.join(R, "IRA_environmental_loading_climate_scores.csv")
cols = list(pd.read_csv(f, nrows=0).columns) if os.path.exists(f) else []
chk("IRA env columns avoid recovered_CO2/productivity", not any(re.search(r"recover|productiv", c, re.I) for c in cols), str(cols))
md = open(os.path.join(R, "analysis_execution_summary.md")).read()
chk("summary states loading release is not recovered CO2", "not** recovered CO₂" in md)
chk("summary states PADAC+ImCOF is a model composition", "model composition" in md)
chk("summary avoids novelty/publication judgement", not re.search(r"\bnovel(ty)? (is|finding)|publishable|high impact", md, re.I))
aud = os.path.join(R, "ImCOF_thermal_property_audit.csv")
if os.path.exists(aud):
    a = pd.read_csv(aud)
    if "property" in a:
        chk("E_d recorded as non-interchangeable with ΔH", (a[a.property.str.startswith("E_d")].resolution == "NOT_INTERCHANGEABLE_WITH_EQUILIBRIUM_dH").all())

HYp=os.path.join(R,"H_Y_climate_ordering_bridge.csv")
if os.path.exists(HYp):
    hc=list(pd.read_csv(HYp,nrows=0).columns)
    chk("H-Y bridge contains rank correspondence, not magnitude ratios", not any(re.search(r"ratio|slope|R2|r2|magnitude",x,re.I) for x in hc), str(hc))
chk("summary states H-Y is rank correspondence only", "rank correspondence only" in md)

st = fc.statuses()
chk("no 36-climate PADAC ranking without complete forcing",
    fc.is_ok("PADAC_GLOBAL_FORCING") or not os.path.exists(os.path.join(R, "ImCOF_environmental_thermal_climate_scores.csv"))
    or "status" in pd.read_csv(os.path.join(R, "ImCOF_environmental_thermal_climate_scores.csv"), nrows=0).columns)
P = os.path.join(R, "paired_controller_fidelity_comparison.csv")
if os.path.exists(P):
    c = list(pd.read_csv(P, nrows=0).columns)
    chk("paired comparison contains no absolute-Y comparison columns", not any(re.search(r"^Y_|ratio", x) for x in c), str(c))
import json
_strip = lambda d: {k: v for k, v in d.items() if k not in ("FROZEN_AT", "RESUME_RUN_DIR")}
chk("frozen scientific config unchanged since run start",
    _strip(json.load(open(os.path.join(R, "frozen_config.json")))) == _strip(json.load(open("/content/fidelity_config.json"))))

if CFG.get("IMCOF_ENV_THERMAL_MODE") == "padac_13X_controller" and os.path.exists(os.path.join(R, "ImCOF_environmental_thermal_climate_scores.csv")):
    ee=pd.read_csv(os.path.join(R, "ImCOF_environmental_thermal_climate_scores.csv"))
    chk("PADAC13X controller branch labels non-ImCOF temperature interpretation", ee.get("interpretation",pd.Series(dtype=str)).astype(str).str.contains("not ImCOF bed T prediction",case=False,regex=False).all())

C = pd.DataFrame(checks); fc.save_csv(C, "claim_boundary_audit.csv")
print("ALL PASSED" if C.passed.all() else "FAILURES — revise outputs before use"); C
